In [6]:
"""
Metric correlation heatmap — storm & outage metrics (pooled 2015-2025)
======================================================================
Builds 12 county-level metrics from the event-level tables + MCC, then shows
their Spearman rank-correlation as a heatmap (all counties, pooled period).

Metrics:
  Exposure/hazard
    1  n_storms              # storm records in the county
    2  storm_hours           sum of storm durations (end-start), hours
  Outage
    4  n_outage_events       # outage events
    5  linkage_rate          storm-linked outages / all outage events (12h)
    6  p90_peak_outage       P90 of peak_customers_out
    8  customer_hours        sum of auc  (absolute burden)
    9  norm_customer_hours   customer_hours / customers_served
    10 p90_recovery_hr       P90 of recovery_hr

Any metric whose source column is missing is dropped with a printed note
(never fabricated). Spearman is used throughout (rank-based, robust to skew).

Reads:  county_level_storm_records_2015_2025.csv, outage_event_table_2015_2025.csv,
        storm_linked_outage_events_2015_2025.csv, MCC.csv
Writes: Results/Q3_2/metric_correlation_matrix.csv
        Results/Q3_2/county_metrics_for_correlation.csv
        Results/Q3_2/figures/fig_metric_correlation_heatmap.png
Requires: pandas, numpy, matplotlib, scipy
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
OUTAGE_CSV = f"{Q1_3}/outage_event_table_2015_2025.csv"
LINKED_CSV = f"{Q1_3}/storm_linked_outage_events_2015_2025.csv"

STORM_MAP = {
    "Thunderstorm Wind": "Thunderstorm Wind", "Lightning": "Thunderstorm Wind",
    "Hail": "Thunderstorm Wind",
    "Hurricane": "Tropical Cyclone", "Hurricane (Typhoon)": "Tropical Cyclone",
    "Tropical Storm": "Tropical Cyclone", "Tropical Depression": "Tropical Cyclone",
    "Storm Surge/Tide": "Tropical Cyclone",
    "Flood": "Flood", "Flash Flood": "Flood", "Coastal Flood": "Flood",
    "Heavy Rain": "Flood", "Excessive Rainfall": "Flood",
    "Tornado": "Tornado", "Funnel Cloud": "Tornado", "Waterspout": "Tornado",
    "Winter Storm": "Winter Storm", "Winter Weather": "Winter Storm",
    "Blizzard": "Winter Storm", "Ice Storm": "Winter Storm",
    "Heavy Snow": "Winter Storm", "Sleet": "Winter Storm",
    "Lake-Effect Snow": "Winter Storm",
}


def load_mcc(path):
    m = pd.read_csv(path); m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    m = m[fn.notna() & m[cc].notna()]
    return dict(zip(fn[fn.notna()].astype("int64").astype(str).str.zfill(5), m[cc]))


def p90(s):
    s = s.dropna()
    return float(np.percentile(s, 90)) if len(s) else np.nan


# ---- storm records -> n_storms, storm_hours ----
print("storm records ...")
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
st["s_start"] = pd.to_datetime(st["start_time"], errors="coerce")
st["s_end"] = pd.to_datetime(st["end_time"], errors="coerce")
st["dur_hr"] = (st["s_end"] - st["s_start"]).dt.total_seconds() / 3600.0
st["dur_hr"] = st["dur_hr"].clip(lower=0)
storm_agg = st.groupby("county_fips").agg(
    n_storms=("event_type", "size"),
    storm_hours=("dur_hr", "sum")).reset_index()

# ---- outage table -> n_outage_events, burden, recovery, peak ----
print("outage table ...")
ot = pd.read_csv(OUTAGE_CSV, dtype={"county_fips": str})
ot["county_fips"] = ot["county_fips"].str.zfill(5)
for c in ["peak_customers_out", "auc", "recovery_hr"]:
    ot[c] = pd.to_numeric(ot[c], errors="coerce")
out_agg = ot.groupby("county_fips").agg(
    n_outage_events=("county_fips", "size"),
    p90_peak_outage=("peak_customers_out", p90),
    customer_hours=("auc", "sum"),
    p90_recovery_hr=("recovery_hr", p90)).reset_index()

# ---- linked file -> linkage rate ----
print("linked file ...")
lk = pd.read_csv(LINKED_CSV, dtype={"county_fips": str},
                 usecols=lambda c: c in {"county_fips", "storm_linked_12h"})
lk["county_fips"] = lk["county_fips"].str.zfill(5)
lk["linked"] = lk["storm_linked_12h"].astype(str).str.lower().isin(
    ["true", "1", "1.0", "yes", "t"])
link_agg = lk.groupby("county_fips")["linked"].mean().rename(
    "linkage_rate").reset_index()

# ---- assemble ----
mcc = load_mcc(MCC_CSV)
df = storm_agg.merge(out_agg, on="county_fips", how="outer") \
              .merge(link_agg, on="county_fips", how="outer")
_size = df["county_fips"].map(mcc).astype(float)
_size = _size.where(_size > 0, np.nan)
df["norm_customer_hours"] = df["customer_hours"] / _size

METRICS = ["n_storms", "storm_hours",
           "n_outage_events", "linkage_rate", "p90_peak_outage",
           "customer_hours", "norm_customer_hours",
           "p90_recovery_hr"]
LABELS = {"n_storms": "Number of storms", "storm_hours": "Storm-duration",
          "n_outage_events": "Number of outage events",
          "linkage_rate": "Storm-linkage rate",
          "p90_peak_outage": "P90 peak outage",
          "customer_hours": "Customer-hours",
          "norm_customer_hours": "Normalized customer-hours",
          "p90_recovery_hr": "P90 recovery (hr)"}

present = [m for m in METRICS if m in df.columns and df[m].notna().sum() > 10]
dropped = [m for m in METRICS if m not in present]
if dropped:
    print("DROPPED (missing/too few values):", dropped)

df.to_csv(f"{OUT_DIR}/county_metrics_for_correlation.csv", index=False)
print(f"counties assembled: {len(df):,}")

# ---- Spearman correlation ----
corr = df[present].corr(method="spearman")
corr.to_csv(f"{OUT_DIR}/metric_correlation_matrix.csv")
print("\nSpearman correlation matrix:")
print(corr.round(2).to_string())

# ---- heatmap ----
labels = [LABELS[m] for m in present]
n = len(present)
fig, ax = plt.subplots(figsize=(1.0 * n + 3, 1.0 * n + 2))
im = ax.imshow(corr.values, cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
for i in range(n):
    for j in range(n):
        v = corr.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(v) > 0.55 else "black")
cb = fig.colorbar(im, ax=ax, shrink=0.8)
cb.set_label("Spearman Correlation")
ax.set_title("Storm and Outage Metric Correlations (Spearman, 2015-2025)",
             fontsize=12, pad=12)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_metric_correlation_heatmap.png", dpi=300)
plt.close(fig)
print(f"\nWrote:\n  {OUT_DIR}/metric_correlation_matrix.csv")
print(f"  {OUT_DIR}/county_metrics_for_correlation.csv")
print(f"  {FIG_DIR}/fig_metric_correlation_heatmap.png")

storm records ...
outage table ...
linked file ...
counties assembled: 3,204

Spearman correlation matrix:
                     n_storms  storm_hours  n_outage_events  linkage_rate  p90_peak_outage  customer_hours  norm_customer_hours  p90_recovery_hr
n_storms                 1.00         0.42             0.26          0.68             0.31            0.28                -0.12            -0.10
storm_hours              0.42         1.00             0.11          0.32             0.17            0.13                -0.06            -0.02
n_outage_events          0.26         0.11             1.00          0.02             0.53            0.82                 0.41             0.19
linkage_rate             0.68         0.32             0.02          1.00             0.14            0.08                -0.11            -0.06
p90_peak_outage          0.31         0.17             0.53          0.14             1.00            0.81                 0.23             0.07
customer_hours         

In [43]:
"""
Merged figure — exposure-vs-burden quadrant scatter (left) + metric correlation (right)
=======================================================================================
Left  : storm exposure percentile vs MCC-normalized customer-hour burden
        percentile, split at 50 into four quadrants (cap1 on burden).
Right : Spearman correlation heatmap of 8 county-level storm/outage metrics.

Both are built from the same raw event tables + MCC (pooled 2015-2025).

Reads:  county_level_storm_records_2015_2025.csv, outage_event_table_2015_2025.csv,
        storm_linked_outage_events_2015_2025.csv, MCC.csv
Writes: Results/Start_From_Begining/metric_correlation_matrix.csv
        Results/Start_From_Begining/county_metrics_for_correlation.csv
        Results/Start_From_Begining/figures/fig_scatter_and_correlation.png
Requires: pandas, numpy, matplotlib, scipy
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
OUTAGE_CSV = f"{Q1_3}/outage_event_table_2015_2025.csv"
LINKED_CSV = f"{Q1_3}/storm_linked_outage_events_2015_2025.csv"
SPLIT = 50.0

COL = {"HH": "#8c4a63", "HL": "#6f9bd1", "LH": "#e8695f", "LL": "#d9d9d9"}
LEG = {"HH": "High Exposure - High Burden", "HL": "High Exposure - Low Burden",
       "LH": "Low Exposure - High Burden", "LL": "Low Exposure - Low Burden"}


def load_mcc(path):
    m = pd.read_csv(path); m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    m = m[fn.notna() & m[cc].notna()]
    return dict(zip(fn[fn.notna()].astype("int64").astype(str).str.zfill(5), m[cc]))


def p90(s):
    s = s.dropna()
    return float(np.percentile(s, 90)) if len(s) else np.nan


def pctl(s):
    return s.fillna(0).rank(pct=True, method="average") * 100.0


# ---- storm records ----
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
st["s_start"] = pd.to_datetime(st["start_time"], errors="coerce")
st["s_end"] = pd.to_datetime(st["end_time"], errors="coerce")
st["dur_hr"] = ((st["s_end"] - st["s_start"]).dt.total_seconds() / 3600).clip(lower=0)
storm_agg = st.groupby("county_fips").agg(
    n_storms=("event_type", "size"), storm_hours=("dur_hr", "sum")).reset_index()

# ---- outage table ----
ot = pd.read_csv(OUTAGE_CSV, dtype={"county_fips": str})
ot["county_fips"] = ot["county_fips"].str.zfill(5)
for c in ["peak_customers_out", "auc", "recovery_hr"]:
    ot[c] = pd.to_numeric(ot[c], errors="coerce")
out_agg = ot.groupby("county_fips").agg(
    n_outage_events=("county_fips", "size"),
    p90_peak_outage=("peak_customers_out", p90),
    customer_hours=("auc", "sum"),
    p90_recovery_hr=("recovery_hr", p90)).reset_index()

# ---- linkage rate ----
lk = pd.read_csv(LINKED_CSV, dtype={"county_fips": str},
                 usecols=lambda c: c in {"county_fips", "storm_linked_12h"})
lk["county_fips"] = lk["county_fips"].str.zfill(5)
lk["linked"] = lk["storm_linked_12h"].astype(str).str.lower().isin(
    ["true", "1", "1.0", "yes", "t"])
link_agg = lk.groupby("county_fips")["linked"].mean().rename("linkage_rate").reset_index()

# ---- assemble ----
mcc = load_mcc(MCC_CSV)
df = storm_agg.merge(out_agg, on="county_fips", how="outer").merge(
    link_agg, on="county_fips", how="outer")
served = df["county_fips"].map(mcc).astype(float)
served = served.where(served > 0, np.nan)
df["norm_customer_hours"] = df["customer_hours"] / served
df.to_csv(f"{OUT_DIR}/county_metrics_for_correlation.csv", index=False)

# ---- correlation ----
METRICS = ["n_storms", "storm_hours", "n_outage_events", "linkage_rate",
           "p90_peak_outage", "customer_hours", "norm_customer_hours", "p90_recovery_hr"]
LABELS = {"n_storms": "Number of storms", "storm_hours": "Storm-duration",
          "n_outage_events": "Number of outage events", "linkage_rate": "Storm-linkage rate",
          "p90_peak_outage": "P90 peak outage", "customer_hours": "Customer-hours",
          "norm_customer_hours": "Normalized customer-hours", "p90_recovery_hr": "P90 recovery (hr)"}
present = [m for m in METRICS if df[m].notna().sum() > 10]
corr = df[present].corr(method="spearman")
corr.to_csv(f"{OUT_DIR}/metric_correlation_matrix.csv")

# ---- quadrant scatter data (cap1 burden) ----
q = df[served.notna()].copy()
sv = served[q.index]
flag = (sv < q["p90_peak_outage"]).fillna(False)
norm = (q["customer_hours"] / sv).replace([np.inf, -np.inf], np.nan)
maxb = norm[~flag].max()
norm = norm.where(~flag, maxb)
q["exposure_pct"] = pctl(q["n_storms"].fillna(0))
q["burden_pct"] = pctl(norm)
q.loc[flag.values, "burden_pct"] = 100.0
hx = q["exposure_pct"] > SPLIT; hy = q["burden_pct"] > SPLIT
q["quad"] = np.where(hx & hy, "HH", np.where(hx & ~hy, "HL",
                     np.where(~hx & hy, "LH", "LL")))

# ---- merged figure ----
fig, (axS, axC) = plt.subplots(1, 2, figsize=(20, 9),
                               gridspec_kw={"width_ratios": [1, 1.15]})

# LEFT: quadrant scatter
for kk in ["LL", "HL", "LH", "HH"]:
    s = q[q["quad"] == kk]
    axS.scatter(s["exposure_pct"], s["burden_pct"], s=14, alpha=0.75,
                color=COL[kk], edgecolors="none", label=f"{LEG[kk]} (n={len(s):,})")
axS.axvline(SPLIT, color="gray", lw=0.8, alpha=0.7)
axS.axhline(SPLIT, color="gray", lw=0.8, alpha=0.7)
axS.plot([0, 100], [0, 100], "k--", lw=1, alpha=0.6, label="Equal exposure-burden agreement")
axS.set_xlabel("Storm exposure percentile")
axS.set_ylabel("Normalized customer-hour burden percentile")
axS.set_title("A. Hazard exposure vs Normalized outage burden")
axS.set_xlim(-1, 101); axS.set_ylim(-1, 101)
axS.legend(loc="lower right", fontsize=8, framealpha=0.95)

# RIGHT: correlation heatmap
labs = [LABELS[m] for m in present]
nn = len(present)
im = axC.imshow(corr.values, cmap="RdBu", vmin=-1, vmax=1)
axC.set_xticks(range(nn)); axC.set_yticks(range(nn))
axC.set_xticklabels(labs, rotation=45, ha="right", fontsize=9)
axC.set_yticklabels(labs, fontsize=9)
for i in range(nn):
    for j in range(nn):
        v = corr.values[i, j]
        axC.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                 color="white" if abs(v) > 0.55 else "black")
cb = fig.colorbar(im, ax=axC, shrink=0.8)
cb.set_label("Spearman correlation")
axC.set_title("B. Storm and outage metric correlations (Spearman)")

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_scatter_and_correlation.png", dpi=600, bbox_inches="tight")
plt.close(fig)
print("quadrant counts:\n", q["quad"].value_counts().to_string())
print(f"\nWrote {FIG_DIR}/fig_scatter_and_correlation.png")
print(f"  {OUT_DIR}/metric_correlation_matrix.csv")
print(f"  {OUT_DIR}/county_metrics_for_correlation.csv")

quadrant counts:
 quad
LH    828
HL    827
LL    722
HH    721

Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_scatter_and_correlation.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/metric_correlation_matrix.csv
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/county_metrics_for_correlation.csv


In [57]:
"""
Combined bivariate quartile maps (pooled 2015-2025) — polished layout
======================================================================
Two bivariate choropleths side by side, each with a clean inset 4x4 legend
placed in the open lower-right area of the CONUS map.

  (a) Number of storm exposures vs number of outage events
  (b) Storm duration vs power-outage duration

Reads:  county_level_storm_records_2015_2025.csv, outage_event_table_2015_2025.csv,
        cb_2022_us_county_5m.shp
Writes: Results/Start_From_Begining/figures/fig_bivariate_combined.png
        Results/Start_From_Begining/bivariate_quartile_values.csv
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
OUTAGE_CSV = f"{Q1_3}/outage_event_table_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"

NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING_COLOR = "#ececec"
DURATION_MODE = "total"          # "total" = sum recovery_hr ; "p90" = county P90

# ---- bivariate palette ----
C00 = to_rgb("#e8e8e8")   # low x, low y
Cx0 = to_rgb("#5ac8c8")   # high x, low y
C0y = to_rgb("#be64ac")   # low x, high y
Cxy = to_rgb("#3b4994")   # high x, high y


def bivariate_color(ix, iy, n=4):
    fx, fy = ix / (n - 1), iy / (n - 1)
    bottom = [(1 - fx) * C00[k] + fx * Cx0[k] for k in range(3)]
    top = [(1 - fx) * C0y[k] + fx * Cxy[k] for k in range(3)]
    return tuple((1 - fy) * bottom[k] + fy * top[k] for k in range(3))


PALETTE = {(ix, iy): bivariate_color(ix, iy) for ix in range(4) for iy in range(4)}


def quartile(series):
    r = series.rank(method="average", pct=True)
    q = np.ceil(r * 4).clip(1, 4) - 1
    return q.where(series.notna(), np.nan)


def build_metrics():
    st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
    st["county_fips"] = st["county_fips"].str.zfill(5)
    st["s_start"] = pd.to_datetime(st["start_time"], errors="coerce")
    st["s_end"] = pd.to_datetime(st["end_time"], errors="coerce")
    st["dur_hr"] = ((st["s_end"] - st["s_start"]).dt.total_seconds() / 3600.0).clip(lower=0)
    sg = st.groupby("county_fips").agg(
        n_storms=("event_type", "size"), storm_hours=("dur_hr", "sum")).reset_index()

    ot = pd.read_csv(OUTAGE_CSV, dtype={"county_fips": str})
    ot["county_fips"] = ot["county_fips"].str.zfill(5)
    ot["recovery_hr"] = pd.to_numeric(ot["recovery_hr"], errors="coerce")
    if DURATION_MODE == "p90":
        odur = ot.groupby("county_fips")["recovery_hr"].agg(
            lambda s: np.percentile(s.dropna(), 90) if s.notna().any() else np.nan
        ).rename("outage_hours").reset_index()
    else:
        odur = ot.groupby("county_fips")["recovery_hr"].sum().rename(
            "outage_hours").reset_index()
    og = ot.groupby("county_fips").size().rename("n_outage_events").reset_index()
    return sg.merge(og, on="county_fips", how="outer").merge(
        odur, on="county_fips", how="outer")


def prepare_bivariate(df, xcol, ycol):
    d = df.copy()
    d["qx"] = quartile(d[xcol])
    d["qy"] = quartile(d[ycol])
    d["bi"] = d.apply(lambda r: f"{int(r['qx'])}{int(r['qy'])}"
                      if pd.notna(r["qx"]) and pd.notna(r["qy"]) else np.nan, axis=1)
    return d


def add_bivariate_legend_inset(ax, xlabel, ylabel, box):
    lax = ax.inset_axes(box)
    lax.set_facecolor("white")
    for ix in range(4):
        for iy in range(4):
            lax.add_patch(plt.Rectangle((ix, iy), 1, 1, facecolor=PALETTE[(ix, iy)],
                                        edgecolor="white", linewidth=0.4))
    lax.set_xlim(0, 4); lax.set_ylim(0, 4)
    lax.set_xticks([0.5, 1.5, 2.5, 3.5]); lax.set_yticks([0.5, 1.5, 2.5, 3.5])
    lax.set_xticklabels(["Q1", "Q2", "Q3", "Q4"], fontsize=7)
    lax.set_yticklabels(["Q1", "Q2", "Q3", "Q4"], fontsize=7)
    lax.set_xlabel(f"{xlabel}  \u2192", fontsize=9, fontweight="bold", labelpad=3)
    lax.set_ylabel(f"{ylabel}  \u2192", fontsize=9, fontweight="bold", labelpad=3)
    lax.set_title("Higher \u2192 darker", fontsize=8, style="italic",
                  color="#444444", pad=4)
    lax.set_aspect("equal")
    lax.tick_params(length=0, pad=2)
    for sp in lax.spines.values():
        sp.set_visible(True); sp.set_linewidth(0.8); sp.set_edgecolor("#666666")


def draw_map(ax, geo, title, legend_xlabel, legend_ylabel, legend_box):
    for ix in range(4):
        for iy in range(4):
            sub = geo[geo["bi"] == f"{ix}{iy}"]
            if len(sub):
                sub.plot(ax=ax, color=PALETTE[(ix, iy)], linewidth=0.05,
                         edgecolor="white")
    miss = geo[geo["bi"].isna()]
    if len(miss):
        miss.plot(ax=ax, color=MISSING_COLOR, linewidth=0.05, edgecolor="white")
    ax.set_axis_off()
    ax.set_title(title, fontsize=14, fontweight="semibold", color="#222222", pad=6)
    add_bivariate_legend_inset(ax, legend_xlabel, legend_ylabel, legend_box)


# ---- build ----
df = build_metrics()
r1 = prepare_bivariate(df, "n_storms", "n_outage_events")
r2 = prepare_bivariate(df, "storm_hours", "outage_hours")

gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo1 = gdf.merge(r1, on="county_fips", how="left")
geo2 = gdf.merge(r2, on="county_fips", how="left")

# ---- figure ----
plt.rcParams.update({"figure.facecolor": "white"})
fig = plt.figure(figsize=(22, 9))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1], wspace=0.01,
                      left=0.01, right=0.99, top=0.90, bottom=0.02)

ax1 = fig.add_subplot(gs[0, 0])
draw_map(ax1, geo1, "(a) Storm exposure vs. outage events",
         "Number of storms", "Number of outages",
         legend_box=(0.86, 0.10, 0.17, 0.30))

ax2 = fig.add_subplot(gs[0, 1])
draw_map(ax2, geo2, "(b) Storm duration vs. outage duration",
         "Storm duration", "Outage duration",
         legend_box=(0.86, 0.10, 0.17, 0.30))

fig.suptitle("Bivariate Relationships Between Storm Characteristics "
             "and Power Outages, 2015\u20132025",
             fontsize=18, fontweight="bold", y=0.97)

fname = f"{FIG_DIR}/fig_bivariate_combined.png"
fig.savefig(fname, dpi=600, bbox_inches="tight")
plt.close(fig)

# ---- save county output ----
output = df.copy()
output["storm_count_q"] = r1["qx"]
output["outage_count_q"] = r1["qy"]
output["storm_outage_count_class"] = r1["bi"]
output["storm_duration_q"] = r2["qx"]
output["outage_duration_q"] = r2["qy"]
output["storm_outage_duration_class"] = r2["bi"]
output.to_csv(f"{OUT_DIR}/bivariate_quartile_values.csv", index=False)

print("Counties:", len(df))
print("\nMap 1 class counts:\n", r1["bi"].value_counts().sort_index().to_string())
print("\nMap 2 class counts:\n", r2["bi"].value_counts().sort_index().to_string())
print(f"\nWrote:\n  {fname}\n  {OUT_DIR}/bivariate_quartile_values.csv")

Counties: 3204

Map 1 class counts:
 bi
00    224
01    216
02    173
03     96
10    196
11    228
12    208
13    131
20    196
21    182
22    197
23    208
30    140
31    130
32    185
33    332

Map 2 class counts:
 bi
00    240
01    209
02    185
03    116
10    157
11    190
12    201
13    185
20    156
21    152
22    211
23    250
30    201
31    212
32    165
33    212

Wrote:
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_bivariate_combined.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/bivariate_quartile_values.csv


In [10]:
"""
Figure 1 — Hazard exposure vs MCC-normalized outage burden (clean quadrant scatter)
===================================================================================
Scatter of storm exposure percentile (x) vs normalized customer-hour burden
percentile (y), each split at 50 into four quadrants. No county annotations,
no stats in the title.

    exposure_pct = percentile(n_storms)
    burden_pct   = percentile(customer_hours / customers_served)  [cap1: sat->100]

Reads:  county_level_storm_records_2015_2025.csv, outage_event_table_2015_2025.csv, MCC.csv
Writes: Results/Start_From_Begining/figures/fig1_exposure_vs_burden_scatter.png
Requires: pandas, numpy, matplotlib
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
OUTAGE_CSV = f"{Q1_3}/outage_event_table_2015_2025.csv"
SPLIT = 50.0

COL = {"HH": "#8c4a63",   # high exp, high burden (dark maroon)
       "HL": "#6f9bd1",   # high exp, low burden  (blue)
       "LH": "#e8695f",   # low exp, high burden  (salmon/red)
       "LL": "#d9d9d9"}   # low exp, low burden   (light grey)
LEG = {"HH": "High Exposure - High Burden", "HL": "High Exposure - Low Burden",
       "LH": "Low Exposure - High Burden", "LL": "Low Exposure - Low Burden"}


def load_mcc(path):
    m = pd.read_csv(path); m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    m = m[fn.notna() & m[cc].notna()]
    return dict(zip(fn[fn.notna()].astype("int64").astype(str).str.zfill(5), m[cc]))


def pctl(s):
    return s.fillna(0).rank(pct=True, method="average") * 100.0


# exposure
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
expo = st.groupby("county_fips").size().rename("n_storms").reset_index()

# burden (normalized, cap1 corrected)
ot = pd.read_csv(OUTAGE_CSV, dtype={"county_fips": str})
ot["county_fips"] = ot["county_fips"].str.zfill(5)
for c in ["auc", "peak_customers_out"]:
    ot[c] = pd.to_numeric(ot[c], errors="coerce")
ob = ot.groupby("county_fips").agg(
    customer_hours=("auc", "sum"),
    peak=("peak_customers_out", lambda s: np.percentile(s.dropna(), 90)
          if s.notna().any() else np.nan)).reset_index()

mcc = load_mcc(MCC_CSV)
df = expo.merge(ob, on="county_fips", how="outer")
df["n_storms"] = df["n_storms"].fillna(0)
served = df["county_fips"].map(mcc).astype(float)
served = served.where(served > 0, np.nan)
flag = (served < df["peak"]).fillna(False)
norm = (df["customer_hours"] / served).replace([np.inf, -np.inf], np.nan)
df = df[served.notna()].copy()
served = served[df.index]; flag = flag[df.index]; norm = norm[df.index]
maxb = norm[~flag].max()
norm = norm.where(~flag, maxb)

df["exposure_pct"] = pctl(df["n_storms"])
df["burden_pct"] = pctl(norm)
df.loc[flag.values, "burden_pct"] = 100.0

hx = df["exposure_pct"] > SPLIT
hy = df["burden_pct"] > SPLIT
df["quad"] = np.where(hx & hy, "HH", np.where(hx & ~hy, "HL",
                      np.where(~hx & hy, "LH", "LL")))

# ---- plot ----
fig, ax = plt.subplots(figsize=(9, 9))
for k in ["LL", "HL", "LH", "HH"]:
    s = df[df["quad"] == k]
    ax.scatter(s["exposure_pct"], s["burden_pct"], s=14, alpha=0.75,
               color=COL[k], edgecolors="none",
               label=f"{LEG[k]} (n={len(s):,})")
ax.axvline(SPLIT, color="gray", lw=0.8, alpha=0.7)
ax.axhline(SPLIT, color="gray", lw=0.8, alpha=0.7)
ax.plot([0, 100], [0, 100], "k--", lw=1, alpha=0.6,
        label="Perfect exposure-burden agreement")
ax.set_xlabel("Storm exposure percentile")
ax.set_ylabel("Normalized customer-hour burden percentile (MCC-normalized)")
ax.set_title("Figure 1. Hazard exposure vs MCC-normalized outage burden")
ax.set_xlim(-1, 101); ax.set_ylim(-1, 101)
ax.legend(loc="lower right", fontsize=8, framealpha=0.95)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig1_exposure_vs_burden_scatter.png", dpi=300)
plt.close(fig)
print("quadrant counts:")
print(df["quad"].value_counts().to_string())
print(f"\nWrote {FIG_DIR}/fig1_exposure_vs_burden_scatter.png")

quadrant counts:
quad
LH    828
HL    827
LL    722
HH    721

Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig1_exposure_vs_burden_scatter.png


In [64]:
"""
Outage concentration — composite figure + per-quantity concentration tables
===========================================================================
Composite 2x2 figure:
  A (top-left)   : pooled Lorenz curves, ABSOLUTE vs NORMALIZED burden (Gini in legend)
  B (top-right)  : Gini per year, both metrics
  C (bottom-left): CONUS map of ABSOLUTE burden   -- PERCENTILE + PowerNorm
  D (bottom-right): CONUS map of NORMALIZED burden -- PERCENTILE + PowerNorm

Reads:  outage_event_table_2015_2025.csv, MCC.csv, shapefile
Writes: Results/Start_From_Begining/figures/fig_outage_concentration_composite.png
        Results/Start_From_Begining/concentration_*.csv
Requires: pandas, numpy, matplotlib, geopandas
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

OUTAGE_CSV = f"{Q1_3}/outage_event_table_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING_COLOR = "#d9d9d9"
ABS_COLOR = "#238b45"
NORM_COLOR = "#6a51a3"
GAMMA = 2.5

GREEN_CMAP = LinearSegmentedColormap.from_list("burden_green",
    ["#ffffff", "#f7fcfd", "#e5f5f9", "#ccece6", "#99d8c9",
     "#66c2a4", "#41ae76", "#238b45", "#006d2c", "#00441b"], N=256)
PURPLE_CMAP = LinearSegmentedColormap.from_list("burden_purple",
    ["#ffffff", "#fcfbfd", "#efedf5", "#dadaeb", "#bcbddc",
     "#9e9ac8", "#807dba", "#6a51a3", "#54278f", "#3f007d"], N=256)


def load_mcc(path):
    m = pd.read_csv(path); m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    m = m[fn.notna() & m[cc].notna()]
    return dict(zip(fn[fn.notna()].astype("int64").astype(str).str.zfill(5), m[cc]))


def lorenz_xy(values):
    x = pd.to_numeric(pd.Series(values), errors="coerce").dropna().values
    x = np.sort(x)[::-1]; x = x[x > 0]
    if x.size == 0:
        return None, None
    cum_b = np.cumsum(x) / x.sum() * 100.0
    cum_c = np.arange(1, x.size + 1) / x.size * 100.0
    return np.concatenate([[0], cum_c]), np.concatenate([[0], cum_b])


def gini(values):
    v = np.sort(pd.to_numeric(pd.Series(values), errors="coerce").dropna().values)
    v = v[v >= 0]
    if v.size == 0 or v.sum() == 0:
        return np.nan
    n = v.size; cum = np.cumsum(v)
    return (n + 1 - 2 * (cum / cum[-1]).sum()) / n


def concentration_stats(values):
    v = np.sort(pd.to_numeric(pd.Series(values), errors="coerce").dropna().values)[::-1]
    v = v[v > 0]
    if v.size == 0:
        return None
    tot = v.sum(); n = v.size
    def topsh(f):
        k = max(1, int(np.ceil(f * n))); return v[:k].sum() / tot * 100
    bottom50 = v[int(np.floor(0.5 * n)):].sum() / tot * 100
    cum = np.cumsum(v) / tot
    k50 = int(np.searchsorted(cum, 0.50) + 1); k50 = min(k50, n)
    return {"n_counties": n, "gini": round(gini(v), 4),
            "top1pct_share": round(topsh(0.01), 2),
            "top5pct_share": round(topsh(0.05), 2),
            "top10pct_share": round(topsh(0.10), 2),
            "top20pct_share": round(topsh(0.20), 2),
            "bottom50pct_share": round(bottom50, 2),
            "counties_to_50pct": k50,
            "counties_to_50pct_share_of_counties": round(k50 / n * 100, 2)}


ot = pd.read_csv(OUTAGE_CSV, dtype={"county_fips": str})
ot["county_fips"] = ot["county_fips"].str.zfill(5)
ot["year"] = pd.to_numeric(ot["year"], errors="coerce")
for c in ["auc", "recovery_hr"]:
    ot[c] = pd.to_numeric(ot[c], errors="coerce")
mcc = load_mcc(MCC_CSV)


def county_metrics(sub):
    g = sub.groupby("county_fips").agg(
        abs_burden=("auc", "sum"),
        n_events=("county_fips", "size"),
        p90_recovery=("recovery_hr",
                      lambda s: np.percentile(s.dropna(), 90) if s.notna().any() else 0.0)
    ).reset_index()
    sv = g["county_fips"].map(mcc).astype(float)
    g["norm_burden"] = g["abs_burden"] / sv.where(sv > 0, np.nan)
    return g


pooled = county_metrics(ot)
years = sorted(int(y) for y in ot["year"].dropna().unique())
per_year_g = {yr: county_metrics(ot[ot["year"] == yr]) for yr in years}

QCOLS = {"customer_hour_burden": "abs_burden",
         "normalized_customer_hour_burden": "norm_burden",
         "n_outage_events": "n_events",
         "p90_recovery": "p90_recovery"}
for qname, col in QCOLS.items():
    rows = []
    st = concentration_stats(pooled[col])
    if st: rows.append({"period": "2015_2025", **st})
    for yr in years:
        st = concentration_stats(per_year_g[yr][col])
        if st: rows.append({"period": str(yr), **st})
    pd.DataFrame(rows).to_csv(f"{OUT_DIR}/concentration_{qname}.csv", index=False)
    print(f"wrote concentration_{qname}.csv")

pooled["abs_pct"] = pooled["abs_burden"].rank(pct=True) * 100
pooled["norm_pct"] = pooled["norm_burden"].rank(pct=True) * 100

gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(pooled, on="county_fips", how="left")

TITLE_KW = dict(fontsize=14, fontweight="bold", pad=8)

fig = plt.figure(figsize=(18, 14))

ax1 = fig.add_subplot(2, 2, 1)
ax1.plot([0, 100], [0, 100], "k--", lw=1.2, alpha=0.7, label="Line of equality")
for col, color, name in [("abs_burden", ABS_COLOR, "Absolute burden"),
                         ("norm_burden", NORM_COLOR, "Normalized burden")]:
    cx, cy = lorenz_xy(pooled[col])
    if cx is not None:
        ax1.plot(cx, cy, lw=2.4, color=color,
                 label=f"{name} (Gini = {gini(pooled[col]):.2f})")
ax1.set_xlabel("Cumulative share of counties (%) — highest burden first")
ax1.set_ylabel("Cumulative share of total (%)")
ax1.set_title("A. Outage burden concentration (Lorenz), 2015-2025", **TITLE_KW)
ax1.set_xlim(0, 100); ax1.set_ylim(0, 100); ax1.set_aspect("equal")
ax1.legend(loc="lower right", fontsize=9); ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(2, 2, 2)
ax2.plot(years, [gini(per_year_g[y]["abs_burden"]) for y in years], "-o",
         color=ABS_COLOR, label="Absolute burden")
ax2.plot(years, [gini(per_year_g[y]["norm_burden"]) for y in years], "-o",
         color=NORM_COLOR, label="Normalized burden")
ax2.set_xlabel("Year"); ax2.set_ylabel("Gini (concentration)")
ax2.set_title("B. Concentration over time (Gini per year)", **TITLE_KW)
ax2.set_ylim(0.7, 1.0); ax2.grid(alpha=0.3); ax2.legend(fontsize=9)

norm = PowerNorm(gamma=GAMMA, vmin=0, vmax=100)
ax3 = fig.add_subplot(2, 2, 3)
geo.plot(column="abs_pct", cmap=GREEN_CMAP, norm=norm, linewidth=0.04,
         edgecolor="white", ax=ax3, legend=True,
         missing_kwds={"color": MISSING_COLOR},
         legend_kwds={"shrink": 0.55, "label": "Absolute burden (percentile)"})
ax3.set_axis_off()
ax3.set_title("C. Absolute customer-hour burden (percentile)", **TITLE_KW)

ax4 = fig.add_subplot(2, 2, 4)
geo.plot(column="norm_pct", cmap=PURPLE_CMAP, norm=norm, linewidth=0.04,
         edgecolor="white", ax=ax4, legend=True,
         missing_kwds={"color": MISSING_COLOR},
         legend_kwds={"shrink": 0.55, "label": "Normalized burden (percentile)"})
ax4.set_axis_off()
ax4.set_title("D. Normalized customer-hour burden (percentile)", **TITLE_KW)

fig.suptitle("Outage Burden Concentration, 2015-2025",
             fontsize=20, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.975])
fig.savefig(f"{FIG_DIR}/fig_outage_concentration_composite.png", dpi=600)
plt.close(fig)
print(f"\nWrote {FIG_DIR}/fig_outage_concentration_composite.png")

wrote concentration_customer_hour_burden.csv
wrote concentration_normalized_customer_hour_burden.csv
wrote concentration_n_outage_events.csv
wrote concentration_p90_recovery.csv

Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_outage_concentration_composite.png


In [65]:
"""
Concentration summary table — all four measures in one CSV
=========================================================
Builds a single tidy table (pooled 2015-2025) with rows = county groups
(Top 1%, Top 5%, Top 20%, Bottom 50%) and, for each of the four measures,
the group's SHARE of the national total (%) and the NUMBER OF COUNTIES in
the group.

Measures:
  customer_hour_burden           sum(auc)
  normalized_customer_hour_burden sum(auc) / customers_served (MCC)
  n_outage_events                count of outage events
  p90_recovery                   P90 of recovery_hr per county

Top groups are cumulative and ranked highest-first (Top 5% includes Top 1%).
Bottom 50% = the lower half of counties by that measure.

Reads:  outage_event_table_2015_2025.csv, MCC.csv
Writes: Results/Start_From_Begining/concentration_group_table.csv
Requires: pandas, numpy
"""
import os
import numpy as np
import pandas as pd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
os.makedirs(OUT_DIR, exist_ok=True)
OUTAGE_CSV = f"{Q1_3}/outage_event_table_2015_2025.csv"

# measure key -> pretty column prefix
MEASURES = {
    "abs_burden": "CustomerHourBurden",
    "norm_burden": "NormalizedBurden",
    "n_events": "NumberOfOutages",
    "p90_recovery": "P90Recovery",
}
GROUPS = [("Top 1%", 0.01), ("Top 5%", 0.05), ("Top 20%", 0.20),
          ("Bottom 50%", None)]


def load_mcc(path):
    m = pd.read_csv(path); m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    m = m[fn.notna() & m[cc].notna()]
    return dict(zip(fn[fn.notna()].astype("int64").astype(str).str.zfill(5), m[cc]))


def group_share_count(values, frac):
    """Return (share_pct, n_counties) for a top-frac group, or bottom-50% if
    frac is None. Counties ranked highest-first; only positive values count."""
    v = np.sort(pd.to_numeric(pd.Series(values), errors="coerce").dropna().values)[::-1]
    v = v[v > 0]
    if v.size == 0:
        return np.nan, 0
    tot, n = v.sum(), v.size
    if frac is None:                       # bottom 50%
        start = int(np.floor(0.5 * n))
        grp = v[start:]
        return round(grp.sum() / tot * 100, 2), int(grp.size)
    k = max(1, int(np.ceil(frac * n)))
    return round(v[:k].sum() / tot * 100, 2), int(k)


# ---- build per-county measures (pooled) ----
ot = pd.read_csv(OUTAGE_CSV, dtype={"county_fips": str})
ot["county_fips"] = ot["county_fips"].str.zfill(5)
for c in ["auc", "recovery_hr"]:
    ot[c] = pd.to_numeric(ot[c], errors="coerce")
g = ot.groupby("county_fips").agg(
    abs_burden=("auc", "sum"),
    n_events=("county_fips", "size"),
    p90_recovery=("recovery_hr",
                  lambda s: np.percentile(s.dropna(), 90) if s.notna().any() else 0.0)
).reset_index()
sv = g["county_fips"].map(load_mcc(MCC_CSV)).astype(float)
g["norm_burden"] = g["abs_burden"] / sv.where(sv > 0, np.nan)

# ---- assemble table ----
rows = []
for label, frac in GROUPS:
    row = {"County Group": label}
    for key, pretty in MEASURES.items():
        share, ncnt = group_share_count(g[key], frac)
        row[f"{pretty}_Share_%"] = share
        row[f"{pretty}_nCounties"] = ncnt
    rows.append(row)
table = pd.DataFrame(rows)

# order columns: group, then each measure's share + count
cols = ["County Group"]
for pretty in MEASURES.values():
    cols += [f"{pretty}_Share_%", f"{pretty}_nCounties"]
table = table[cols]

table.to_csv(f"{OUT_DIR}/concentration_group_table.csv", index=False)
print(table.to_string(index=False))
print(f"\nWrote {OUT_DIR}/concentration_group_table.csv")

County Group  CustomerHourBurden_Share_%  CustomerHourBurden_nCounties  NormalizedBurden_Share_%  NormalizedBurden_nCounties  NumberOfOutages_Share_%  NumberOfOutages_nCounties  P90Recovery_Share_%  P90Recovery_nCounties
      Top 1%                       22.36                            31                     80.17                          31                     3.90                         31                 2.82                     31
      Top 5%                       48.40                           153                     84.39                         152                    13.53                        153                 9.56                    153
     Top 20%                       78.07                           611                     91.49                         607                    39.04                        611                28.57                    611
  Bottom 50%                        4.74                          1526                      2.80                    

In [72]:
"""
Hazard-attributed outage burden: 6h vs 12h buffer (absolute only)
=================================================================

Three common hazards:
    Thunderstorm Wind
    Flood
    Tornado

Figure structure:

    rows 1-2 : per-hazard ABSOLUTE burden maps
               row 1 = 6h
               row 2 = 12h

    spacer    : small visual gap

    row 3     : two DOMINANT-hazard maps
               left = 6h
               right = 12h

    spacer    : small visual gap

    row 4     : left  = number of counties per hazard, 6h vs 12h
               right = concentration Gini per hazard, 6h vs 12h

Hazard attribution:
    nearest storm within +/- buffer of the outage peak.

Outputs:
    figures/fig_hazard_outage_6h_12h.png
    hazard_outage_concentration_6h_12h.csv

Requires:
    pandas
    numpy
    matplotlib
    geopandas
"""

import os
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
from matplotlib.patches import Patch
import geopandas as gpd


# ============================================================
# PATHS
# ============================================================

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"

Q1_3 = f"{BASE}/Results/Q1_3"

MCC_CSV = f"{BASE}/MCC.csv"

OUT_DIR = f"{BASE}/Results/Start_From_Begining"

FIG_DIR = f"{OUT_DIR}/figures"

os.makedirs(FIG_DIR, exist_ok=True)


LINKED_CSV = (
    f"{Q1_3}/storm_linked_outage_events_2015_2025.csv"
)

STORM_CSV = (
    f"{Q1_3}/county_level_storm_records_2015_2025.csv"
)

SHP = (
    f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
)


# ============================================================
# SETTINGS
# ============================================================

NON_CONUS = {
    "02",  # Alaska
    "15",  # Hawaii
    "72",  # Puerto Rico
    "66",
    "78",
    "60",
    "69",
}

MISSING_COLOR = "#e6e6e6"

BUFFERS = [6, 12]

HAZARDS = [
    "Thunderstorm Wind",
    "Flood",
    "Tornado",
]

GAMMA = 2.5


# ============================================================
# COLORS
# ============================================================

HAZ_COLOR = {
    "Thunderstorm Wind": "#2166ac",
    "Flood": "#238b45",
    "Tornado": "#d94801",
}

HAZ_DARK = {
    "Thunderstorm Wind": "#08306b",
    "Flood": "#00441b",
    "Tornado": "#7f2704",
}

CMAPS = {
    h: LinearSegmentedColormap.from_list(
        h,
        [
            "#ffffff",
            HAZ_COLOR[h],
            HAZ_DARK[h],
        ],
        N=256,
    )
    for h in HAZARDS
}


# ============================================================
# STORM TYPE GROUPING
# ============================================================

STORM_MAP = {

    "Thunderstorm Wind": "Thunderstorm Wind",
    "Lightning": "Thunderstorm Wind",
    "Hail": "Thunderstorm Wind",

    "Hurricane": "Tropical Cyclone",
    "Hurricane (Typhoon)": "Tropical Cyclone",
    "Tropical Storm": "Tropical Cyclone",
    "Tropical Depression": "Tropical Cyclone",
    "Storm Surge/Tide": "Tropical Cyclone",

    "Flood": "Flood",
    "Flash Flood": "Flood",
    "Coastal Flood": "Flood",
    "Heavy Rain": "Flood",
    "Excessive Rainfall": "Flood",

    "Tornado": "Tornado",
    "Funnel Cloud": "Tornado",
    "Waterspout": "Tornado",

    "Winter Storm": "Winter Storm",
    "Winter Weather": "Winter Storm",
    "Blizzard": "Winter Storm",
    "Ice Storm": "Winter Storm",
    "Heavy Snow": "Winter Storm",
    "Sleet": "Winter Storm",
    "Lake-Effect Snow": "Winter Storm",

    "High Wind": "High Wind",
    "Strong Wind": "High Wind",
}


# ============================================================
# LOAD MCC
# ============================================================

def load_mcc(path):

    m = pd.read_csv(path)

    m.columns = [
        c.strip()
        for c in m.columns
    ]

    fc = (
        "County_FIPS"
        if "County_FIPS" in m.columns
        else m.columns[0]
    )

    cc = (
        "Customers"
        if "Customers" in m.columns
        else m.columns[1]
    )

    m[cc] = pd.to_numeric(
        m[cc],
        errors="coerce",
    )

    fn = pd.to_numeric(
        m[fc],
        errors="coerce",
    )

    m = m[
        fn.notna()
        & m[cc].notna()
    ]

    return dict(
        zip(
            fn[
                fn.notna()
            ]
            .astype("int64")
            .astype(str)
            .str.zfill(5),

            m[cc],
        )
    )


# ============================================================
# GINI
# ============================================================

def gini(v):

    v = np.sort(
        np.asarray(v, float)
    )

    v = v[v >= 0]

    if (
        v.size == 0
        or v.sum() == 0
    ):
        return np.nan

    n = v.size

    cum = np.cumsum(v)

    return (
        n
        + 1
        - 2
        * (
            cum / cum[-1]
        ).sum()
    ) / n


# ============================================================
# CONCENTRATION STATISTICS
# ============================================================

def conc_stats(v):

    v = np.sort(
        np.asarray(v, float)
    )[::-1]

    v = v[v > 0]

    if v.size == 0:
        return None

    tot = v.sum()

    n = v.size

    top = lambda f: (
        v[
            :max(
                1,
                int(
                    np.ceil(
                        f * n
                    )
                ),
            )
        ].sum()
        / tot
        * 100
    )

    cum = (
        np.cumsum(v)
        / tot
    )

    k50 = min(
        int(
            np.searchsorted(
                cum,
                0.5,
            )
            + 1
        ),
        n,
    )

    return {

        "n_counties": n,

        "gini": round(
            gini(v),
            4,
        ),

        "top1pct_share": round(
            top(0.01),
            2,
        ),

        "top5pct_share": round(
            top(0.05),
            2,
        ),

        "top10pct_share": round(
            top(0.10),
            2,
        ),

        "counties_to_50pct": k50,
    }


# ============================================================
# LOAD DATA
# ============================================================

mcc = load_mcc(
    MCC_CSV
)


# ---------------- Storm records ----------------

st = pd.read_csv(
    STORM_CSV,
    dtype={
        "county_fips": str
    },
)

st["county_fips"] = (
    st["county_fips"]
    .str.zfill(5)
)

st["hazard"] = (
    st["event_type"]
    .map(STORM_MAP)
    .fillna("Other")
)

st["s_start"] = pd.to_datetime(
    st["start_time"],
    errors="coerce",
)

st["s_end"] = pd.to_datetime(
    st["end_time"],
    errors="coerce",
)

st = st.dropna(
    subset=[
        "s_start",
        "s_end",
    ]
)

storms_by_c = {
    c: g
    for c, g
    in st.groupby(
        "county_fips"
    )
}


# ---------------- Outage events ----------------

outs = pd.read_csv(
    LINKED_CSV,
    dtype={
        "county_fips": str
    },
)

outs["county_fips"] = (
    outs["county_fips"]
    .str.zfill(5)
)

outs["auc"] = pd.to_numeric(
    outs["auc"],
    errors="coerce",
)

outs["peak_dt"] = pd.to_datetime(
    outs["peak_time"],
    errors="coerce",
).fillna(

    pd.to_datetime(
        outs["event_start"],
        errors="coerce",
    )
)

outs = outs.dropna(
    subset=[
        "peak_dt"
    ]
)


# ---------------- County shapefile ----------------

gdf = gpd.read_file(
    SHP
)

gdf["county_fips"] = (
    gdf["GEOID"]
    .astype(str)
    .str.zfill(5)
)

gdf = gdf[
    ~gdf[
        "county_fips"
    ]
    .str[:2]
    .isin(
        NON_CONUS
    )
]

gdf = gdf.to_crs(
    "EPSG:5070"
)


# ============================================================
# BOOLEAN HELPER
# ============================================================

def truthy(s):

    return (
        s
        .astype(str)
        .str.lower()
        .isin(
            [
                "true",
                "1",
                "1.0",
                "yes",
                "t",
            ]
        )
    )


# ============================================================
# ATTRIBUTE OUTAGE TO NEAREST STORM
# ============================================================

def attribute(
    o_df,
    buffer_hr,
):

    buf = np.timedelta64(
        buffer_hr,
        "h",
    )

    haz = np.array(
        [None] * len(o_df),
        dtype=object,
    )

    pos = {
        ix: i
        for i, ix
        in enumerate(
            o_df.index
        )
    }

    for c, og in o_df.groupby(
        "county_fips"
    ):

        sg = storms_by_c.get(c)

        if (
            sg is None
            or len(sg) == 0
        ):
            continue

        ss = (
            sg["s_start"]
            .values
            .astype(
                "datetime64[ns]"
            )
        )

        se = (
            sg["s_end"]
            .values
            .astype(
                "datetime64[ns]"
            )
        )

        sh = (
            sg["hazard"]
            .values
        )

        peaks = (
            og["peak_dt"]
            .values
            .astype(
                "datetime64[ns]"
            )
        )

        loc = [
            pos[ix]
            for ix
            in og.index
        ]

        ch = max(
            1,
            min(
                4000,
                int(
                    2_000_000
                    / max(
                        1,
                        len(sg),
                    )
                ),
            ),
        )

        for i in range(
            0,
            len(peaks),
            ch,
        ):

            P = peaks[
                i:i + ch
            ][:, None]

            dist = np.maximum(

                np.maximum(
                    ss[None, :]
                    - P,

                    P
                    - se[None, :],
                ),

                np.timedelta64(
                    0,
                    "ns",
                ),
            )

            within = (
                dist <= buf
            )

            df_ = (
                dist
                .astype(
                    "timedelta64[s]"
                )
                .astype(float)
            )

            df_[
                ~within
            ] = np.inf

            j = df_.argmin(
                axis=1
            )

            any_ = within.any(
                axis=1
            )

            sel = np.where(
                any_,
                sh[j],
                None,
            )

            for r, val in enumerate(
                sel
            ):

                haz[
                    loc[i + r]
                ] = val

    return pd.Series(
        haz,
        index=o_df.index,
    )


# ============================================================
# COUNTY-LEVEL HAZARD BURDEN
# ============================================================

def hazard_county(
    buffer_hr
):

    o = outs[
        truthy(
            outs[
                f"storm_linked_{buffer_hr}h"
            ]
        )
    ].copy()

    o["hazard"] = attribute(
        o,
        buffer_hr,
    )

    o = o[
        o["hazard"]
        .isin(
            HAZARDS
        )
    ]

    return (
        o
        .groupby(
            [
                "county_fips",
                "hazard",
            ]
        )["auc"]
        .sum()
        .rename(
            "abs_burden"
        )
        .reset_index()
    )


# ============================================================
# CALCULATE BOTH BUFFERS
# ============================================================

data = {
    b: hazard_county(b)
    for b in BUFFERS
}


for b in BUFFERS:

    print(
        f"\n{b}h county counts:\n",
        data[b]
        .groupby(
            "hazard"
        )["county_fips"]
        .nunique()
        .to_string(),
    )


# ============================================================
# MAP NORMALIZATION
# ============================================================

norm = PowerNorm(
    gamma=GAMMA,
    vmin=0,
    vmax=100,
)

nH = len(
    HAZARDS
)


# ============================================================
# CREATE FIGURE
# ============================================================

fig = plt.figure(
    figsize=(
        19,
        16,
    )
)


# ============================================================
# CUSTOM GRID
#
# Row 0 = 6h hazard maps
# Row 1 = 12h hazard maps
# Row 2 = SMALL SPACER
# Row 3 = dominant-hazard maps
# Row 4 = SMALL SPACER
# Row 5 = bottom statistics
#
# This gives much more direct control over spacing.
# ============================================================

gs = fig.add_gridspec(
    6,
    6,

    height_ratios=[
        1.30,
        1.30,
        0.09,
        1.70,
        0.09,
        0.80,
    ],

    hspace=0.015,
    wspace=0.03,

    # slightly lower first row to leave a small gap below main title
    top=0.945,

    bottom=0.055,
    left=0.035,
    right=0.985,
)


# ============================================================
# ROWS 1-2:
# PER-HAZARD ABSOLUTE BURDEN MAPS
# ============================================================

for row, b in enumerate(
    BUFFERS
):

    for col, h in enumerate(
        HAZARDS
    ):

        sub = data[b][
            data[b]["hazard"] == h
        ].copy()

        sub["abs_pct"] = (
            sub["abs_burden"]
            .rank(
                pct=True
            )
            * 100
        )

        geo = gdf.merge(
            sub,
            on="county_fips",
            how="left",
        )

        ax = fig.add_subplot(
            gs[
                row,
                2 * col:
                2 * col + 2,
            ]
        )

        geo.plot(

            column="abs_pct",

            cmap=CMAPS[h],

            norm=norm,

            linewidth=0.03,

            edgecolor="white",

            ax=ax,

            missing_kwds={
                "color": MISSING_COLOR
            },
        )

        ax.set_axis_off()


        # -------------------------
        # Hazard titles
        # -------------------------

        if row == 0:

            ax.set_title(
                h,
                fontsize=14,
                fontweight="bold",
                color="black",

                # Slightly closer to maps
                pad=3,
            )


        # -------------------------
        # Row labels
        # -------------------------

        if col == 0:

            ax.text(

                -0.04,

                0.5,

                f"{b}h buffer",

                transform=ax.transAxes,

                rotation=90,

                va="center",

                fontsize=13,
            )


# ============================================================
# DOMINANT HAZARD FUNCTION
# ============================================================

def dominant(g):

    piv = g.pivot_table(

        index="county_fips",

        columns="hazard",

        values="abs_burden",

        aggfunc="sum",
    )

    for h in HAZARDS:

        if h not in piv.columns:

            piv[h] = np.nan

    return (

        piv[
            HAZARDS
        ]

        .idxmax(
            axis=1
        )

        .rename(
            "dominant"
        )

        .reset_index()

        .dropna(
            subset=[
                "dominant"
            ]
        )
    )


# ============================================================
# ROW 3:
# DOMINANT HAZARD MAPS
# ============================================================

# Keep references to both map axes so the shared legend can be
# positioned exactly in the gap between the two CONUS maps.
dominant_axes = []

for i, b in enumerate(
    BUFFERS
):

    dom = dominant(
        data[b]
    )

    geo_d = gdf.merge(
        dom,
        on="county_fips",
        how="left",
    )


    if i == 0:

        ax = fig.add_subplot(
            gs[
                3,
                0:3,
            ]
        )

    else:

        ax = fig.add_subplot(
            gs[
                3,
                3:6,
            ]
        )


    # -------------------------
    # No attributed outage
    # -------------------------

    geo_d[
        geo_d[
            "dominant"
        ].isna()
    ].plot(

        ax=ax,

        color=MISSING_COLOR,

        linewidth=0.03,

        edgecolor="white",
    )


    # -------------------------
    # Dominant hazards
    # -------------------------

    for h in HAZARDS:

        s = geo_d[
            geo_d[
                "dominant"
            ] == h
        ]

        if len(s):

            s.plot(

                ax=ax,

                color=HAZ_COLOR[h],

                linewidth=0.03,

                edgecolor="white",
            )


    ax.set_axis_off()


    ax.set_title(

        f"Dominant hazard by county — {b}h buffer",

        fontsize=13,
        fontweight="bold",

        # Reduce title-to-map distance
        pad=3,
    )

    dominant_axes.append(ax)


# -------------------------
# One shared legend centered in the gap between the two maps
# -------------------------
legend_handles = [
    Patch(color=HAZ_COLOR[h], label=h)
    for h in HAZARDS
] + [
    Patch(color=MISSING_COLOR, label="no attributed outage")
]

# Draw once so Matplotlib has finalized the geographic axes positions.
fig.canvas.draw()

left_pos = dominant_axes[0].get_position()
right_pos = dominant_axes[1].get_position()

legend_x = (left_pos.x1 + right_pos.x0) / 2
legend_y = (left_pos.y0 + left_pos.y1) / 2

fig.legend(
    handles=legend_handles,
    loc="center",
    bbox_to_anchor=(legend_x, legend_y),
    bbox_transform=fig.transFigure,
    fontsize=9,
    frameon=False,
    ncol=1,
    handlelength=1.4,
    handletextpad=0.5,
    borderaxespad=0.0,
)


# ============================================================
# CONCENTRATION STATISTICS
# ============================================================

rows = []


for b in BUFFERS:

    for h in HAZARDS:

        cs = conc_stats(

            data[b][
                data[b][
                    "hazard"
                ] == h
            ][
                "abs_burden"
            ]
        )

        if cs:

            rows.append(
                {
                    "buffer_h": b,
                    "hazard": h,
                    **cs,
                }
            )


cdf = pd.DataFrame(
    rows
)


cdf.to_csv(

    f"{OUT_DIR}/hazard_outage_concentration_6h_12h.csv",

    index=False,
)


# ============================================================
# BAR SETTINGS
# ============================================================

x = np.arange(
    nH
)

w = 0.38

hcol = [
    HAZ_COLOR[h]
    for h in HAZARDS
]


def val(
    b,
    h,
    key,
):

    r = cdf[
        (cdf.buffer_h == b)
        &
        (cdf.hazard == h)
    ]

    return (
        r[key].iloc[0]
        if len(r)
        else 0
    )


# ============================================================
# BOTTOM CHART LAYOUT
# Slightly narrower charts with a small gap between them
# ============================================================

bottom_gs = gs[5, :].subgridspec(
    1,
    3,
    width_ratios=[1.0, 0.08, 1.0],
    wspace=0.0,
)


# ============================================================
# BOTTOM LEFT:
# NUMBER OF COUNTIES
# ============================================================

ax1 = fig.add_subplot(bottom_gs[0, 0])


n6 = [
    val(
        6,
        h,
        "n_counties",
    )
    for h in HAZARDS
]


n12 = [
    val(
        12,
        h,
        "n_counties",
    )
    for h in HAZARDS
]


ax1.bar(

    x - w / 2,

    n6,

    w,

    color=hcol,
)


ax1.bar(

    x + w / 2,

    n12,

    w,

    color=hcol,

    hatch="//",

    alpha=0.65,
)


ax1.set_xticks(
    x
)


ax1.set_xticklabels(

    HAZARDS,

    rotation=15,

    ha="right",

    fontsize=10,
)


ax1.set_ylabel(
    "Number of counties"
)


ax1.set_title(

    "Counties with attributed outages",
    fontweight="bold",

    fontsize=12,

    pad=4,
)


ax1.legend(

    handles=[

        Patch(
            facecolor="#888",
            label="6h",
        ),

        Patch(
            facecolor="#888",
            hatch="//",
            alpha=0.65,
            label="12h",
        ),
    ],

    fontsize=9,
)


# ============================================================
# BOTTOM RIGHT:
# GINI
# ============================================================

ax2 = fig.add_subplot(bottom_gs[0, 2])


g6 = [
    val(
        6,
        h,
        "gini",
    )
    for h in HAZARDS
]


g12 = [
    val(
        12,
        h,
        "gini",
    )
    for h in HAZARDS
]


ax2.bar(

    x - w / 2,

    g6,

    w,

    color=hcol,
)


ax2.bar(

    x + w / 2,

    g12,

    w,

    color=hcol,

    hatch="//",

    alpha=0.65,
)


ax2.set_xticks(
    x
)


ax2.set_xticklabels(

    HAZARDS,

    rotation=15,

    ha="right",

    fontsize=10,
)


ax2.set_ylabel(
    "Gini"
)


ax2.set_ylim(
    0,
    1,
)


ax2.set_title(

    "Concentration (Gini)",

    fontsize=12,
    fontweight="bold",

    pad=4,
)


ax2.legend(

    handles=[

        Patch(
            facecolor="#888",
            label="6h",
        ),

        Patch(
            facecolor="#888",
            hatch="//",
            alpha=0.65,
            label="12h",
        ),
    ],

    fontsize=9,
)


# ============================================================
# MAIN TITLE
# ============================================================

fig.suptitle(

    "Hazard-attributed outage burden: 6h vs 12h buffer",

    fontsize=18,          # 2 sizes larger than before
    fontweight="bold",
    # Lower than original 0.995 so it sits closer to first row
    y=0.978,
)


# ============================================================
# SAVE
# ============================================================

fig.savefig(

    f"{FIG_DIR}/fig_hazard_outage_6h_12h.png",

    dpi=600,

    bbox_inches="tight",
)


plt.close(
    fig
)


# ============================================================
# OUTPUT MESSAGE
# ============================================================

print(
    f"\nWrote "
    f"{FIG_DIR}/fig_hazard_outage_6h_12h.png"
)

print(
    f"  "
    f"{OUT_DIR}/hazard_outage_concentration_6h_12h.csv"
)


6h county counts:
 hazard
Flood                2353
Thunderstorm Wind    2419
Tornado               613

12h county counts:
 hazard
Flood                2455
Thunderstorm Wind    2899
Tornado              1540

Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_hazard_outage_6h_12h.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/hazard_outage_concentration_6h_12h.csv


In [75]:
"""
Disproportionate disruption maps (two panels)
=============================================
Where does realized outage impact exceed what storm exposure alone predicts,
and where is it the reverse?

  Panel A: DR_burden = B_pct - E_pct
           B_pct = percentile of normalized customer-hour burden (sum auc / MCC)
           E_pct = percentile of storm exposure (n_storms)
           red  -> burden rank exceeds exposure rank (worse than exposure predicts)
           blue -> exposure rank exceeds burden rank (relatively resilient)

  Panel B: DR_count = Ocount_pct - E_pct
           Ocount_pct = percentile of number of outage events
           red  -> more outages than exposure predicts; blue -> fewer

Both computed per county, pooled 2015-2025, all counties with both values.

Reads:  county_level_storm_records_2015_2025.csv, outage_event_table_2015_2025.csv,
        MCC.csv, shapefile
Writes: Results/Start_From_Begining/figures/fig_disproportionate_two_panel.png
        Results/Start_From_Begining/disproportionate_disruption.csv
Requires: pandas, numpy, matplotlib, geopandas
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
OUTAGE_CSV = f"{Q1_3}/outage_event_table_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING_COLOR = "#e6e6e6"


def load_mcc(path):
    m = pd.read_csv(path); m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    m = m[fn.notna() & m[cc].notna()]
    return dict(zip(fn[fn.notna()].astype("int64").astype(str).str.zfill(5), m[cc]))


def pctl(s):
    return s.rank(pct=True) * 100.0


# exposure
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
expo = st.groupby("county_fips").size().rename("n_storms").reset_index()

# outage burden + count
ot = pd.read_csv(OUTAGE_CSV, dtype={"county_fips": str})
ot["county_fips"] = ot["county_fips"].str.zfill(5)
ot["auc"] = pd.to_numeric(ot["auc"], errors="coerce")
ob = ot.groupby("county_fips").agg(
    customer_hours=("auc", "sum"),
    n_outage_events=("county_fips", "size")).reset_index()

mcc = load_mcc(MCC_CSV)
d = expo.merge(ob, on="county_fips", how="outer")
sv = d["county_fips"].map(mcc).astype(float)
sv = sv.where(sv > 0, np.nan)
d["norm_burden"] = d["customer_hours"] / sv

d["E_pct"] = pctl(d["n_storms"])
d["B_pct"] = pctl(d["norm_burden"])
d["Ocount_pct"] = pctl(d["n_outage_events"])
d["DR_burden"] = d["B_pct"] - d["E_pct"]
d["DR_count"] = d["Ocount_pct"] - d["E_pct"]
d.to_csv(f"{OUT_DIR}/disproportionate_disruption.csv", index=False)

gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d, on="county_fips", how="left")

fig, (axA, axB) = plt.subplots(1, 2, figsize=(22, 8))
norm = TwoSlopeNorm(vmin=-80, vcenter=0, vmax=80)

geo.plot(column="DR_burden", cmap="RdBu_r", norm=norm, linewidth=0.04,
         edgecolor="white", legend=True, ax=axA,
         missing_kwds={"color": MISSING_COLOR, "label": "No data"},
         legend_kwds={"shrink": 0.55, "label": "burden pct - exposure pct"})
axA.set_axis_off()
axA.set_title("A. Burden vs exposure\n"
              "red = burden exceeds exposure (worse than hazard predicts); "
              "blue = exposure exceeds burden", fontsize=12)

geo.plot(column="DR_count", cmap="RdBu_r", norm=norm, linewidth=0.04,
         edgecolor="white", legend=True, ax=axB,
         missing_kwds={"color": MISSING_COLOR, "label": "No data"},
         legend_kwds={"shrink": 0.55, "label": "outage-count pct - exposure pct"})
axB.set_axis_off()
axB.set_title("B. Outage count vs exposure\n"
              "red = more outages than exposure predicts; blue = fewer",
              fontsize=12)

fig.suptitle("Disproportionate disruption, 2015-2025", fontsize=16, y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(f"{FIG_DIR}/fig_disproportionate_two_panel.png", dpi=600,
            bbox_inches="tight")
plt.close(fig)
print("counties:", d["DR_burden"].notna().sum())
print(f"Wrote {FIG_DIR}/fig_disproportionate_two_panel.png")
print(f"  {OUT_DIR}/disproportionate_disruption.csv")

counties: 3032
Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_disproportionate_two_panel.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/disproportionate_disruption.csv


In [74]:
"""
Top-tier burden concentration map
=================================
CONUS map classifying counties by where they fall in the distribution of
customer-hour burden (sum of auc), pooled 2015-2025:

    Top 1%      (>= 99th percentile)
    Top 1-5%    (95th-99th)
    Top 5-20%   (80th-95th)
    Bottom 80%  (< 80th)
    No burden / no data

This makes the concentration visible: the handful of top-1% counties that hold
a large share of the national burden stand out in the darkest colour.

Reads:  outage_event_table_2015_2025.csv, shapefile
        (concentration_customer_hour_burden.csv only for reference thresholds)
Writes: Results/Start_From_Begining/figures/fig_top_tier_burden_map.png
        Results/Start_From_Begining/burden_tier_by_county.csv
Requires: pandas, numpy, matplotlib, geopandas
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

OUTAGE_CSV = f"{Q1_3}/outage_event_table_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING_COLOR = "#e6e6e6"

# tiers (top-down) and their colours: darkest = most concentrated
TIERS = ["Top 1%", "Top 1-5%", "Top 5-20%", "Bottom 80%"]
TIER_COLOR = {"Top 1%": "#67000d", "Top 1-5%": "#cb181d",
              "Top 5-20%": "#fc9272", "Bottom 80%": "#fee0d2"}


# ---- per-county burden ----
ot = pd.read_csv(OUTAGE_CSV, dtype={"county_fips": str})
ot["county_fips"] = ot["county_fips"].str.zfill(5)
ot["auc"] = pd.to_numeric(ot["auc"], errors="coerce")
d = ot.groupby("county_fips")["auc"].sum().rename("customer_hours").reset_index()
d = d[d["customer_hours"] > 0].copy()

pct = d["customer_hours"].rank(pct=True) * 100.0
d["burden_percentile"] = pct
d["tier"] = np.select(
    [pct >= 99, pct >= 95, pct >= 80],
    ["Top 1%", "Top 1-5%", "Top 5-20%"],
    default="Bottom 80%")
d.to_csv(f"{OUT_DIR}/burden_tier_by_county.csv", index=False)

# report share of total burden held by each tier
tot = d["customer_hours"].sum()
print("Tier | n_counties | share of total burden")
for t in TIERS:
    sub = d[d["tier"] == t]
    print(f"  {t:11s} {len(sub):5d}   {sub['customer_hours'].sum()/tot*100:5.1f}%")

# ---- map ----
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d[["county_fips", "tier"]], on="county_fips", how="left")

fig, ax = plt.subplots(figsize=(15, 9))
geo[geo["tier"].isna()].plot(ax=ax, color=MISSING_COLOR, linewidth=0.05,
                             edgecolor="white")
for t in TIERS:
    sub = geo[geo["tier"] == t]
    if len(sub):
        sub.plot(ax=ax, color=TIER_COLOR[t], linewidth=0.05, edgecolor="white")
ax.set_axis_off()
ax.set_title("Counties by customer-hour burden tier, 2015-2025\n"
             "darkest = top 1% of counties by total burden", fontsize=14, pad=10)
handles = [Patch(color=TIER_COLOR[t],
           label=f"{t} (n={int((d['tier'] == t).sum()):,})") for t in TIERS]
handles.append(Patch(color=MISSING_COLOR, label="No burden / no data"))
ax.legend(handles=handles, loc="lower left", fontsize=10, frameon=False)

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_top_tier_burden_map.png", dpi=600, bbox_inches="tight")
plt.close(fig)
print(f"\nWrote {FIG_DIR}/fig_top_tier_burden_map.png")
print(f"  {OUT_DIR}/burden_tier_by_county.csv")

Tier | n_counties | share of total burden
  Top 1%         31    22.4%
  Top 1-5%      122    26.0%
  Top 5-20%     458    29.7%
  Bottom 80%   2441    21.9%

Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_top_tier_burden_map.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/burden_tier_by_county.csv


In [76]:
"""
Seasonal hazard tracking (two panels)
=====================================
When in the year does each hazard drive outages? Uses hazard-attributed outage
burden (12h buffer), aggregated by calendar month over 2015-2025.

  Panel A: month x hazard HEATMAP of total customer-hours (auc), each hazard
           row normalized to its own peak month (so seasonality is comparable
           across hazards of different size). Raw totals also written to CSV.
  Panel B: STACKED AREA of monthly customer-hours by hazard (absolute), showing
           both the seasonal shape and each hazard's contribution.

Hazard attribution: nearest storm within +/- 12h of the outage peak.

Reads:  storm_linked_outage_events_2015_2025.csv, county_level_storm_records_2015_2025.csv
Writes: Results/Start_From_Begining/figures/fig_seasonal_hazards.png
        Results/Start_From_Begining/seasonal_hazard_burden.csv
Requires: pandas, numpy, matplotlib
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

LINKED_CSV = f"{Q1_3}/storm_linked_outage_events_2015_2025.csv"
STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
BUFFER = 12
HAZARDS = ["Thunderstorm Wind", "Flood", "Tornado"]
HAZ_COLOR = {"Thunderstorm Wind": "#2166ac", "Flood": "#238b45", "Tornado": "#d94801"}
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

STORM_MAP = {
    "Thunderstorm Wind": "Thunderstorm Wind", "Lightning": "Thunderstorm Wind",
    "Hail": "Thunderstorm Wind",
    "Hurricane": "Tropical Cyclone", "Hurricane (Typhoon)": "Tropical Cyclone",
    "Tropical Storm": "Tropical Cyclone", "Tropical Depression": "Tropical Cyclone",
    "Storm Surge/Tide": "Tropical Cyclone",
    "Flood": "Flood", "Flash Flood": "Flood", "Coastal Flood": "Flood",
    "Heavy Rain": "Flood", "Excessive Rainfall": "Flood",
    "Tornado": "Tornado", "Funnel Cloud": "Tornado", "Waterspout": "Tornado",
    "Winter Storm": "Winter Storm", "Winter Weather": "Winter Storm",
    "Blizzard": "Winter Storm", "Ice Storm": "Winter Storm",
    "Heavy Snow": "Winter Storm", "Sleet": "Winter Storm",
    "Lake-Effect Snow": "Winter Storm", "High Wind": "High Wind",
    "Strong Wind": "High Wind",
}


def truthy(s):
    return s.astype(str).str.lower().isin(["true", "1", "1.0", "yes", "t"])


# storm records for attribution
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
st["hazard"] = st["event_type"].map(STORM_MAP).fillna("Other")
st["s_start"] = pd.to_datetime(st["start_time"], errors="coerce")
st["s_end"] = pd.to_datetime(st["end_time"], errors="coerce")
st = st.dropna(subset=["s_start", "s_end"])
storms_by_c = {c: g for c, g in st.groupby("county_fips")}

outs = pd.read_csv(LINKED_CSV, dtype={"county_fips": str})
outs["county_fips"] = outs["county_fips"].str.zfill(5)
outs["auc"] = pd.to_numeric(outs["auc"], errors="coerce")
outs["peak_dt"] = pd.to_datetime(outs["peak_time"], errors="coerce").fillna(
    pd.to_datetime(outs["event_start"], errors="coerce"))
outs = outs.dropna(subset=["peak_dt"])
outs = outs[truthy(outs[f"storm_linked_{BUFFER}h"])].copy()


def attribute(o_df, buffer_hr):
    buf = np.timedelta64(buffer_hr, "h")
    haz = np.array([None] * len(o_df), dtype=object)
    pos = {ix: i for i, ix in enumerate(o_df.index)}
    for c, og in o_df.groupby("county_fips"):
        sg = storms_by_c.get(c)
        if sg is None or len(sg) == 0:
            continue
        ss = sg["s_start"].values.astype("datetime64[ns]")
        se = sg["s_end"].values.astype("datetime64[ns]")
        sh = sg["hazard"].values
        peaks = og["peak_dt"].values.astype("datetime64[ns]")
        loc = [pos[ix] for ix in og.index]
        ch = max(1, min(4000, int(2_000_000 / max(1, len(sg)))))
        for i in range(0, len(peaks), ch):
            P = peaks[i:i+ch][:, None]
            dist = np.maximum(np.maximum(ss[None, :] - P, P - se[None, :]),
                              np.timedelta64(0, "ns"))
            within = dist <= buf
            df_ = dist.astype("timedelta64[s]").astype(float); df_[~within] = np.inf
            j = df_.argmin(axis=1); any_ = within.any(axis=1)
            sel = np.where(any_, sh[j], None)
            for r, val in enumerate(sel):
                haz[loc[i+r]] = val
    return pd.Series(haz, index=o_df.index)


outs["hazard"] = attribute(outs, BUFFER)
outs = outs[outs["hazard"].isin(HAZARDS)]
outs["month"] = outs["peak_dt"].dt.month

# month x hazard burden matrix (customer-hours)
mat = (outs.groupby(["hazard", "month"])["auc"].sum()
       .unstack("month").reindex(index=HAZARDS, columns=range(1, 13))
       .fillna(0.0))
mat.columns = MONTHS
mat.to_csv(f"{OUT_DIR}/seasonal_hazard_burden.csv")
print("Monthly burden by hazard (customer-hours):")
print(mat.round(0).to_string())

# ---- figure ----
fig, (axA, axB) = plt.subplots(1, 2, figsize=(20, 6.5))

# Panel A: heatmap, each hazard row scaled to its own max
row_norm = mat.div(mat.max(axis=1).replace(0, np.nan), axis=0)
im = axA.imshow(row_norm.values, cmap="magma_r", aspect="auto", vmin=0, vmax=1)
axA.set_xticks(range(12)); axA.set_xticklabels(MONTHS)
axA.set_yticks(range(len(HAZARDS))); axA.set_yticklabels(HAZARDS)
for i in range(len(HAZARDS)):
    for j in range(12):
        v = row_norm.values[i, j]
        if not np.isnan(v):
            axA.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                     color="white" if v > 0.55 else "black")
cb = fig.colorbar(im, ax=axA, shrink=0.7)
cb.set_label("share of hazard's peak month")
axA.set_title("A. Seasonality by hazard\n(each row scaled to its own peak month)",
              fontsize=12)

# Panel B: stacked area, absolute customer-hours
bottom = np.zeros(12)
xs = np.arange(12)
for h in HAZARDS:
    y = mat.loc[h].values
    axB.fill_between(xs, bottom, bottom + y, color=HAZ_COLOR[h], alpha=0.85, label=h)
    bottom = bottom + y
axB.set_xticks(range(12)); axB.set_xticklabels(MONTHS)
axB.set_ylabel("Customer-hours (absolute)")
axB.set_title("B. Monthly outage burden by hazard (stacked)", fontsize=12)
axB.legend(loc="upper right", fontsize=9)
axB.margins(x=0)

fig.suptitle("Seasonal hazard tracking (12h buffer), 2015-2025", fontsize=15, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{FIG_DIR}/fig_seasonal_hazards.png", dpi=600, bbox_inches="tight")
plt.close(fig)
print(f"\nWrote {FIG_DIR}/fig_seasonal_hazards.png")
print(f"  {OUT_DIR}/seasonal_hazard_burden.csv")

Monthly burden by hazard (customer-hours):
                          Jan         Feb         Mar          Apr          May          Jun          Jul          Aug          Sep          Oct         Nov         Dec
hazard                                                                                                                                                                  
Thunderstorm Wind  13119106.0  18649688.0  59470498.0  120363292.0  189286484.0  275023651.0  210763476.0  215379078.0   31973210.0   29268076.0  16548135.0  15929873.0
Flood              32095031.0  22925742.0  12112974.0   22437483.0   59953398.0   36457127.0   52894988.0  113931888.0  360940889.0  164349566.0  15088399.0  27291034.0
Tornado             5144709.0   3203418.0  14257814.0   16991173.0   18535134.0   10191636.0    8322032.0   17745873.0    2010254.0    4224653.0   2210726.0   9097873.0

Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/f

In [90]:
"""
Seasonal hazard tracking — national timing + hazard mix + regional detail
=========================================================================
Hazard-attributed outage burden (12h buffer), by calendar month, 2015-2025.

  A : month x hazard heatmap (each row scaled to its own peak month) -- TIMING
  B : 100%-normalized hazard-mix area (share of each month's burden by hazard)
      with a black line showing total monthly burden -- MIX + OVERALL SEASON
  C : four regional timing heatmaps (Northeast / Midwest / South / West) --
      how the seasonal timing shifts geographically

Reads:  storm_linked_outage_events_2015_2025.csv, county_level_storm_records_2015_2025.csv
Writes: Results/Start_From_Begining/figures/fig_seasonal_hazards.png
        Results/Start_From_Begining/seasonal_hazard_burden.csv
Requires: pandas, numpy, matplotlib
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

LINKED_CSV = f"{Q1_3}/storm_linked_outage_events_2015_2025.csv"
STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
BUFFER = 12
HAZARDS = ["Thunderstorm Wind", "Flood", "Tornado"]
HAZ_COLOR = {"Thunderstorm Wind": "#2166ac", "Flood": "#238b45", "Tornado": "#d94801"}
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
M1 = [m[0] for m in MONTHS]
HEAT_CMAP = "YlGnBu"

# census regions by 2-digit state FIPS
REGIONS = {
    "Northeast": {"09", "23", "25", "33", "44", "50", "34", "36", "42"},
    "Midwest": {"17", "18", "26", "39", "55", "19", "20", "27", "29", "31", "38", "46"},
    "South": {"10", "11", "12", "13", "24", "37", "45", "51", "54",
              "01", "21", "28", "47", "05", "22", "40", "48"},
    "West": {"04", "08", "16", "30", "32", "35", "49", "56",
             "06", "41", "53"},
}

STORM_MAP = {
    "Thunderstorm Wind": "Thunderstorm Wind", "Lightning": "Thunderstorm Wind",
    "Hail": "Thunderstorm Wind",
    "Hurricane": "Tropical Cyclone", "Hurricane (Typhoon)": "Tropical Cyclone",
    "Tropical Storm": "Tropical Cyclone", "Tropical Depression": "Tropical Cyclone",
    "Storm Surge/Tide": "Tropical Cyclone",
    "Flood": "Flood", "Flash Flood": "Flood", "Coastal Flood": "Flood",
    "Heavy Rain": "Flood", "Excessive Rainfall": "Flood",
    "Tornado": "Tornado", "Funnel Cloud": "Tornado", "Waterspout": "Tornado",
    "Winter Storm": "Winter Storm", "Winter Weather": "Winter Storm",
    "Blizzard": "Winter Storm", "Ice Storm": "Winter Storm",
    "Heavy Snow": "Winter Storm", "Sleet": "Winter Storm",
    "Lake-Effect Snow": "Winter Storm", "High Wind": "High Wind", "Strong Wind": "High Wind",
}


def truthy(s):
    return s.astype(str).str.lower().isin(["true", "1", "1.0", "yes", "t"])


st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
st["hazard"] = st["event_type"].map(STORM_MAP).fillna("Other")
st["s_start"] = pd.to_datetime(st["start_time"], errors="coerce")
st["s_end"] = pd.to_datetime(st["end_time"], errors="coerce")
st = st.dropna(subset=["s_start", "s_end"])
storms_by_c = {c: g for c, g in st.groupby("county_fips")}

outs = pd.read_csv(LINKED_CSV, dtype={"county_fips": str})
outs["county_fips"] = outs["county_fips"].str.zfill(5)
outs["auc"] = pd.to_numeric(outs["auc"], errors="coerce")
outs["peak_dt"] = pd.to_datetime(outs["peak_time"], errors="coerce").fillna(
    pd.to_datetime(outs["event_start"], errors="coerce"))
outs = outs.dropna(subset=["peak_dt"])
outs = outs[truthy(outs[f"storm_linked_{BUFFER}h"])].copy()


def attribute(o_df, buffer_hr):
    buf = np.timedelta64(buffer_hr, "h")
    haz = np.array([None] * len(o_df), dtype=object)
    pos = {ix: i for i, ix in enumerate(o_df.index)}
    for c, og in o_df.groupby("county_fips"):
        sg = storms_by_c.get(c)
        if sg is None or len(sg) == 0:
            continue
        ss = sg["s_start"].values.astype("datetime64[ns]")
        se = sg["s_end"].values.astype("datetime64[ns]")
        sh = sg["hazard"].values
        peaks = og["peak_dt"].values.astype("datetime64[ns]")
        loc = [pos[ix] for ix in og.index]
        ch = max(1, min(4000, int(2_000_000 / max(1, len(sg)))))
        for i in range(0, len(peaks), ch):
            P = peaks[i:i+ch][:, None]
            dist = np.maximum(np.maximum(ss[None, :] - P, P - se[None, :]),
                              np.timedelta64(0, "ns"))
            within = dist <= buf
            df_ = dist.astype("timedelta64[s]").astype(float); df_[~within] = np.inf
            j = df_.argmin(axis=1); any_ = within.any(axis=1)
            sel = np.where(any_, sh[j], None)
            for r, val in enumerate(sel):
                haz[loc[i+r]] = val
    return pd.Series(haz, index=o_df.index)


outs["hazard"] = attribute(outs, BUFFER)
outs = outs[outs["hazard"].isin(HAZARDS)]
outs["month"] = outs["peak_dt"].dt.month
outs["st2"] = outs["county_fips"].str[:2]
r2region = {s: r for r, ss in REGIONS.items() for s in ss}
outs["region"] = outs["st2"].map(r2region)


def haz_month_matrix(df):
    return (df.groupby(["hazard", "month"])["auc"].sum().unstack("month")
            .reindex(index=HAZARDS, columns=range(1, 13)).fillna(0.0))


mat = haz_month_matrix(outs)
mat_out = mat.copy(); mat_out.columns = MONTHS
mat_out.to_csv(f"{OUT_DIR}/seasonal_hazard_burden.csv")

# top-3-month concentration per hazard (share of annual burden in top 3 months)
top3 = {h: (np.sort(mat.loc[h].values)[::-1][:3].sum() / mat.loc[h].sum() * 100)
        if mat.loc[h].sum() > 0 else np.nan for h in HAZARDS}


def draw_heatmap(ax, m, title, month_lab=MONTHS, annotate=True, fs=7):
    rn = m.div(m.max(axis=1).replace(0, np.nan), axis=0)
    ax.imshow(rn.values, cmap=HEAT_CMAP, aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(12)); ax.set_xticklabels(month_lab, fontsize=8)
    ax.set_yticks(range(len(HAZARDS))); ax.set_yticklabels(HAZARDS, fontsize=8)
    # white cell separators
    ax.set_xticks(np.arange(-.5, 12, 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(HAZARDS), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.4)
    ax.tick_params(which="minor", length=0)
    ax.tick_params(which="major", length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    if annotate:
        for i in range(len(HAZARDS)):
            for j in range(12):
                v = rn.values[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                            fontsize=fs, color="white" if v > 0.55 else "#333333")
    ax.set_title(title, fontsize=12, fontweight="bold")
    return rn


# ---- figure ----
fig = plt.figure(figsize=(20, 11))
gs = fig.add_gridspec(2, 4, height_ratios=[1.15, 0.9], hspace=0.32, wspace=0.28)

# A: national heatmap
axA = fig.add_subplot(gs[0, 0:2])
rnA = draw_heatmap(axA, mat, "A. Seasonal timing by hazard (row-scaled to peak month)")
sm = plt.cm.ScalarMappable(cmap=HEAT_CMAP, norm=plt.Normalize(0, 1))
cb = fig.colorbar(sm, ax=axA, shrink=0.7, pad=0.02)
cb.set_label("Share of hazard's peak month", fontsize=9)
# annotate top-3-month concentration on the right
txt = "  |  ".join([f"{h.split()[0]}: {top3[h]:.0f}%" for h in HAZARDS])
axA.text(0.0, -0.22, f"Share of annual burden in each hazard's top 3 months —  {txt}",
         transform=axA.transAxes, fontsize=8.5, style="italic", color="#444")

# B: 100% hazard-mix area + total-burden line
axB = fig.add_subplot(gs[0, 2:4])
totals = mat.sum(axis=0).values
shares = np.divide(mat.values, totals, out=np.zeros_like(mat.values),
                   where=totals > 0) * 100
xs = np.arange(12); bottom = np.zeros(12)
for i, h in enumerate(HAZARDS):
    axB.fill_between(xs, bottom, bottom + shares[i], color=HAZ_COLOR[h],
                     alpha=0.85, label=h)
    bottom += shares[i]
axB.set_xticks(range(12)); axB.set_xticklabels(MONTHS, fontsize=8)
axB.set_ylabel("Share of monthly burden (%)"); axB.set_ylim(0, 100); axB.margins(x=0)
axB.set_title("B. Hazard mix by month, with total burden", fontsize=12, fontweight="bold")
axT = axB.twinx()
axT.plot(xs, totals, color="black", lw=2.2, marker="o", ms=4, label="Total burden")
axT.set_ylabel("Total customer-hours", fontsize=9)
axT.set_ylim(0, totals.max() * 1.15)
h1, l1 = axB.get_legend_handles_labels(); h2, l2 = axT.get_legend_handles_labels()
axB.legend(h1 + h2, l1 + l2, loc="upper right", fontsize=8, framealpha=0.95)

# C: regional heatmaps
for k, region in enumerate(["Northeast", "Midwest", "South", "West"]):
    axR = fig.add_subplot(gs[1, k])
    rmat = haz_month_matrix(outs[outs["region"] == region])
    draw_heatmap(axR, rmat, region, month_lab=M1, annotate=False)
    if k == 0:
        axR.set_ylabel("")  # keep hazard labels only on first via yticklabels

fig.suptitle("Seasonal Hazard Tracking (12h buffer), 2015-2025",
             fontsize=17, fontweight="bold", y=0.98)
fig.text(0.5, 0.46, "C. Regional seasonal timing (each row scaled to its own peak month)",
         ha="center", fontsize=12, fontweight="bold")
fig.savefig(f"{FIG_DIR}/fig_seasonal_hazards.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print("top-3-month concentration:", {h: round(top3[h], 1) for h in HAZARDS})
print(f"Wrote {FIG_DIR}/fig_seasonal_hazards.png")
print(f"  {OUT_DIR}/seasonal_hazard_burden.csv")

top-3-month concentration: {'Thunderstorm Wind': np.float64(58.6), 'Flood': np.float64(69.4), 'Tornado': np.float64(47.6)}
Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_seasonal_hazards.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/seasonal_hazard_burden.csv


In [82]:
"""
Attribution-buffer threshold justification
==========================================
Shows how much outage impact gets attributed to storms as the temporal buffer
widens, to justify 12h as a sensible cutoff. For each window in
{1, 3, 6, 12, 24, 48} h, an outage is "linked" if a storm in the same county
falls within +/- window of the outage peak. We measure:

  - linkage_rate    : fraction of outage EVENTS linked
  - burden_captured : fraction of total outage CUSTOMER-HOURS (auc) linked
  - marginal_gain   : increase in burden_captured vs previous window (pp)
  - gain_per_hour   : marginal_gain divided by the step width (pp per hour)
                      -- fair comparison because the windows are unevenly spaced

Efficiency (gain per hour) declines as the window widens; the point past which
extra hours add little per hour supports the chosen threshold.

Reads:  storm_linked_outage_events_2015_2025.csv, county_level_storm_records_2015_2025.csv
Writes: Results/Start_From_Begining/figures/fig_buffer_threshold.png
        Results/Start_From_Begining/buffer_threshold_capture.csv
Requires: pandas, numpy, matplotlib
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

LINKED_CSV = f"{Q1_3}/storm_linked_outage_events_2015_2025.csv"
STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
WINDOWS = [1, 3, 6, 12, 24, 48]
HIGHLIGHT = 12

# ---- storms ----
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
st["s_start"] = pd.to_datetime(st["start_time"], errors="coerce")
st["s_end"] = pd.to_datetime(st["end_time"], errors="coerce")
st = st.dropna(subset=["s_start", "s_end"])
storms_by_c = {c: (g["s_start"].values.astype("datetime64[ns]"),
                   g["s_end"].values.astype("datetime64[ns]"))
               for c, g in st.groupby("county_fips")}

# ---- outages ----
o = pd.read_csv(LINKED_CSV, dtype={"county_fips": str})
o["county_fips"] = o["county_fips"].str.zfill(5)
o["auc"] = pd.to_numeric(o["auc"], errors="coerce").fillna(0)
o["peak_dt"] = pd.to_datetime(o["peak_time"], errors="coerce").fillna(
    pd.to_datetime(o["event_start"], errors="coerce"))
o = o.dropna(subset=["peak_dt"])
print(f"outages: {len(o):,} | total customer-hours: {o['auc'].sum():,.0f}")


def min_dist_hours(o_df):
    out = np.full(len(o_df), np.nan)
    pos = {ix: i for i, ix in enumerate(o_df.index)}
    for c, og in o_df.groupby("county_fips"):
        sc = storms_by_c.get(c)
        if sc is None:
            continue
        ss, se = sc
        peaks = og["peak_dt"].values.astype("datetime64[ns]")
        loc = [pos[ix] for ix in og.index]
        chunk = max(1, min(4000, int(2_000_000 / max(1, len(ss)))))
        for i in range(0, len(peaks), chunk):
            P = peaks[i:i+chunk][:, None]
            dist = np.maximum(np.maximum(ss[None, :] - P, P - se[None, :]),
                              np.timedelta64(0, "ns"))
            dmin = dist.min(axis=1).astype("timedelta64[s]").astype(float) / 3600.0
            for r, val in enumerate(dmin):
                out[loc[i+r]] = val
    return out


print("computing nearest-storm distance per outage (once) ...")
o["dist_hr"] = min_dist_hours(o)

# ---- capture per window ----
tot_burden = o["auc"].sum()
rows = []
prev_cap, prev_w = 0.0, 0
for w in WINDOWS:
    linked = o["dist_hr"] <= w
    lr = linked.mean() * 100
    cap = o.loc[linked, "auc"].sum() / tot_burden * 100
    step = w - prev_w
    rows.append({"window_h": w, "linkage_rate_pct": round(lr, 2),
                 "burden_captured_pct": round(cap, 2),
                 "marginal_gain_pct": round(cap - prev_cap, 2),
                 "step_hours": step,
                 "gain_per_hour_pp": round((cap - prev_cap) / step, 3)})
    prev_cap, prev_w = cap, w
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/buffer_threshold_capture.csv", index=False)
print(res.to_string(index=False))

ceiling = res["burden_captured_pct"].iloc[-1]

# ---- figure ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# A: capture curves
ax1.plot(res["window_h"], res["burden_captured_pct"], "-o", color="#b2182b",
         lw=2.4, label="Customer-hours captured")
ax1.plot(res["window_h"], res["linkage_rate_pct"], "-o", color="#2166ac",
         lw=2.4, label="Outage events linked")
ax1.axvline(HIGHLIGHT, color="gray", ls="--", lw=1.5, label=f"{HIGHLIGHT}h threshold")
ax1.axhline(ceiling, color="#b2182b", ls=":", lw=1, alpha=0.6)
ax1.text(48, ceiling + 1.5, f"ceiling ~{ceiling:.0f}% at 48h",
         ha="right", va="bottom", fontsize=8, color="#b2182b")
ax1.set_xlabel("Attribution buffer window (± hours)")
ax1.set_ylabel("Percent captured")
ax1.set_title("A. Storm attribution vs buffer window")
ax1.set_xticks(WINDOWS); ax1.set_ylim(0, 100)
ax1.grid(alpha=0.3); ax1.legend(loc="upper left", fontsize=9)

# B: gain PER HOUR (fair across unequal steps)
ax2.bar([str(w) for w in res["window_h"]], res["gain_per_hour_pp"],
        color=["#999" if w != HIGHLIGHT else "#b2182b" for w in res["window_h"]])
ax2.set_xlabel("Buffer window (± hours)")
ax2.set_ylabel("Capture efficiency (pp per extra hour)")
ax2.set_title("B. Marginal gain per hour of widening")
ax2.grid(alpha=0.3, axis="y")
for i, v in enumerate(res["gain_per_hour_pp"]):
    ax2.text(i, v + max(res["gain_per_hour_pp"]) * 0.02, f"{v:.2f}",
             ha="center", fontsize=8)

fig.suptitle("Attribution-buffer threshold justification, 2015-2025",
             fontweight="bold", fontsize=15, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{FIG_DIR}/fig_buffer_threshold.png", dpi=600, bbox_inches="tight")
plt.close(fig)
print(f"\nWrote {FIG_DIR}/fig_buffer_threshold.png")
print(f"  {OUT_DIR}/buffer_threshold_capture.csv")

outages: 630,724 | total customer-hours: 8,350,681,843
computing nearest-storm distance per outage (once) ...
 window_h  linkage_rate_pct  burden_captured_pct  marginal_gain_pct  step_hours  gain_per_hour_pp
        1              1.28                 5.35               5.35           1             5.353
        3              1.98                 6.73               1.37           2             0.687
        6              4.47                11.03               4.30           3             1.433
       12             11.08                26.72              15.70           6             2.616
       24             13.54                33.67               6.95          12             0.579
       48             16.73                39.94               6.27          24             0.261

Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_buffer_threshold.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFil

In [83]:
"""
Eleven-year trends (2015-2025)
==============================
Six-panel view of how storm-outage impact and its structure changed over time.

  A total outage customer-hours per year
  B number of outage events per year
  C storm-linkage rate per year (share of outages linked at 12h)
  D customer-hours by hazard per year (stacked; 12h attribution)
  E concentration: Gini of county customer-hour burden per year
  F concentration: number of counties holding 50% of that year's burden

Reads:  storm_linked_outage_events_2015_2025.csv, county_level_storm_records_2015_2025.csv
Writes: Results/Start_From_Begining/figures/fig_trends_11yr.png
        Results/Start_From_Begining/trends_11yr.csv
Requires: pandas, numpy, matplotlib
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

LINKED_CSV = f"{Q1_3}/storm_linked_outage_events_2015_2025.csv"
STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
BUFFER = 12
HAZARDS = ["Thunderstorm Wind", "Flood", "Tornado"]
HAZ_COLOR = {"Thunderstorm Wind": "#2166ac", "Flood": "#238b45", "Tornado": "#d94801"}
ACCENT = "#6a51a3"

STORM_MAP = {
    "Thunderstorm Wind": "Thunderstorm Wind", "Lightning": "Thunderstorm Wind",
    "Hail": "Thunderstorm Wind",
    "Hurricane": "Tropical Cyclone", "Hurricane (Typhoon)": "Tropical Cyclone",
    "Tropical Storm": "Tropical Cyclone", "Tropical Depression": "Tropical Cyclone",
    "Storm Surge/Tide": "Tropical Cyclone",
    "Flood": "Flood", "Flash Flood": "Flood", "Coastal Flood": "Flood",
    "Heavy Rain": "Flood", "Excessive Rainfall": "Flood",
    "Tornado": "Tornado", "Funnel Cloud": "Tornado", "Waterspout": "Tornado",
    "Winter Storm": "Winter Storm", "Winter Weather": "Winter Storm",
    "Blizzard": "Winter Storm", "Ice Storm": "Winter Storm",
    "Heavy Snow": "Winter Storm", "Sleet": "Winter Storm",
    "Lake-Effect Snow": "Winter Storm", "High Wind": "High Wind",
    "Strong Wind": "High Wind",
}


def truthy(s):
    return s.astype(str).str.lower().isin(["true", "1", "1.0", "yes", "t"])


def gini(v):
    v = np.sort(np.asarray(v, float)); v = v[v >= 0]
    if v.size == 0 or v.sum() == 0:
        return np.nan
    n = v.size; cum = np.cumsum(v)
    return (n + 1 - 2 * (cum / cum[-1]).sum()) / n


def counties_to_50(v):
    v = np.sort(np.asarray(v, float))[::-1]; v = v[v > 0]
    if v.size == 0:
        return np.nan
    cum = np.cumsum(v) / v.sum()
    return int(np.searchsorted(cum, 0.5) + 1)


# storms for attribution
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
st["hazard"] = st["event_type"].map(STORM_MAP).fillna("Other")
st["s_start"] = pd.to_datetime(st["start_time"], errors="coerce")
st["s_end"] = pd.to_datetime(st["end_time"], errors="coerce")
st = st.dropna(subset=["s_start", "s_end"])
storms_by_c = {c: g for c, g in st.groupby("county_fips")}

o = pd.read_csv(LINKED_CSV, dtype={"county_fips": str})
o["county_fips"] = o["county_fips"].str.zfill(5)
o["auc"] = pd.to_numeric(o["auc"], errors="coerce").fillna(0)
o["year"] = pd.to_numeric(o["year"], errors="coerce")
o["peak_dt"] = pd.to_datetime(o["peak_time"], errors="coerce").fillna(
    pd.to_datetime(o["event_start"], errors="coerce"))
o = o.dropna(subset=["peak_dt", "year"])
o["year"] = o["year"].astype(int)
o["linked12"] = truthy(o[f"storm_linked_{BUFFER}h"])

years = sorted(o["year"].unique())

# per-year basics
per_year = o.groupby("year").agg(
    total_customer_hours=("auc", "sum"),
    n_outage_events=("county_fips", "size"),
    linkage_rate=("linked12", "mean")).reindex(years)
per_year["linkage_rate"] *= 100

# per-year concentration (county burden within the year)
gini_y, c50_y = [], []
for y in years:
    cb = o[o["year"] == y].groupby("county_fips")["auc"].sum()
    gini_y.append(gini(cb.values))
    c50_y.append(counties_to_50(cb.values))
per_year["gini"] = gini_y
per_year["counties_to_50pct"] = c50_y


# per-year burden by hazard (attribute linked outages)
def attribute(o_df, buffer_hr):
    buf = np.timedelta64(buffer_hr, "h")
    haz = np.array([None] * len(o_df), dtype=object)
    pos = {ix: i for i, ix in enumerate(o_df.index)}
    for c, og in o_df.groupby("county_fips"):
        sg = storms_by_c.get(c)
        if sg is None or len(sg) == 0:
            continue
        ss = sg["s_start"].values.astype("datetime64[ns]")
        se = sg["s_end"].values.astype("datetime64[ns]")
        sh = sg["hazard"].values
        peaks = og["peak_dt"].values.astype("datetime64[ns]")
        loc = [pos[ix] for ix in og.index]
        ch = max(1, min(4000, int(2_000_000 / max(1, len(sg)))))
        for i in range(0, len(peaks), ch):
            P = peaks[i:i+ch][:, None]
            dist = np.maximum(np.maximum(ss[None, :] - P, P - se[None, :]),
                              np.timedelta64(0, "ns"))
            within = dist <= buf
            df_ = dist.astype("timedelta64[s]").astype(float); df_[~within] = np.inf
            j = df_.argmin(axis=1); any_ = within.any(axis=1)
            sel = np.where(any_, sh[j], None)
            for r, val in enumerate(sel):
                haz[loc[i+r]] = val
    return pd.Series(haz, index=o_df.index)


ol = o[o["linked12"]].copy()
ol["hazard"] = attribute(ol, BUFFER)
hz = (ol[ol["hazard"].isin(HAZARDS)]
      .groupby(["year", "hazard"])["auc"].sum().unstack("hazard")
      .reindex(index=years, columns=HAZARDS).fillna(0.0))

per_year.to_csv(f"{OUT_DIR}/trends_11yr.csv")
print(per_year.round(3).to_string())

# ---- figure ----
fig, axes = plt.subplots(2, 3, figsize=(19, 10))
yr = np.array(years)

def line(ax, y, color, title, ylab, fmt=None):
    ax.plot(yr, y, "-o", color=color, lw=2.2)
    ax.set_title(title, fontsize=12); ax.set_ylabel(ylab)
    ax.grid(alpha=0.3); ax.set_xticks(yr); ax.tick_params(axis="x", rotation=45)

line(axes[0, 0], per_year["total_customer_hours"].values, "#b2182b",
     "A. Total outage customer-hours", "customer-hours")
line(axes[0, 1], per_year["n_outage_events"].values, "#2166ac",
     "B. Number of outage events", "events")
line(axes[0, 2], per_year["linkage_rate"].values, "#238b45",
     "C. Storm-linkage rate (12h)", "% outages linked")
axes[0, 2].set_ylim(0, 100)

# D: stacked burden by hazard
axD = axes[1, 0]
bottom = np.zeros(len(yr))
for h in HAZARDS:
    axD.bar(yr, hz[h].values, bottom=bottom, color=HAZ_COLOR[h], label=h)
    bottom += hz[h].values
axD.set_title("D. Customer-hours by hazard", fontsize=12)
axD.set_ylabel("customer-hours"); axD.set_xticks(yr)
axD.tick_params(axis="x", rotation=45); axD.legend(fontsize=8)

line(axes[1, 1], per_year["gini"].values, ACCENT,
     "E. Concentration (Gini of county burden)", "Gini")
axes[1, 1].set_ylim(0, 1)
line(axes[1, 2], per_year["counties_to_50pct"].values, "#d94801",
     "F. Counties holding 50% of burden", "# counties")

fig.suptitle("Eleven-year trends in storm-induced outage impact, 2015-2025",
             fontsize=16, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{FIG_DIR}/fig_trends_11yr.png", dpi=250, bbox_inches="tight")
plt.close(fig)
print(f"\nWrote {FIG_DIR}/fig_trends_11yr.png")
print(f"  {OUT_DIR}/trends_11yr.csv")

      total_customer_hours  n_outage_events  linkage_rate   gini  counties_to_50pct
year                                                                               
2015          2.804017e+08            44303        12.108  0.772                116
2016          3.884331e+08            42643        12.342  0.794                 97
2017          9.424452e+08            44269        12.833  0.882                 21
2018          6.350398e+08            54849        11.231  0.804                100
2019          5.086797e+08            63073        12.782  0.777                110
2020          1.059693e+09            56370        12.750  0.836                 80
2021          1.036557e+09            60859        10.600  0.841                 52
2022          7.106935e+08            69015        10.127  0.792                 96
2023          7.647468e+08            61242        13.889  0.781                113
2024          1.487744e+09            63290        12.239  0.853            

In [84]:
"""
Risk framework: four dimensions, LHS-weighted, with correlation + maps
======================================================================
Dimensions (each an equal-average of its sub-elements, all as percentiles):
  Hazard   = mean( storm exposure count* , storm duration* )
  Severity = mean( normalized customer-hours* , P90 recovery* )
  Fragility= DR*   where DR = norm-burden pct - exposure pct
  Recovery = storm-linkage share*   (share of outages storm-linked at 12h)
(* = percentile 0-100)

Baseline risk = mean of the four dimensions (equal weight).
LHS risk      = weights sampled on the 4-D simplex; R_lhs_mean and P_VH
                (fraction of scenarios a county is top-10%) give robustness.

INTERPRETATION SWITCHES (edit if you meant something else):
  SEVERITY_P90 = "recovery"      -> p90_recovery_hr in Severity
  RECOVERY_METRIC = "linkage"    -> storm-linkage share as the Recovery dimension
Burden/recovery are computed over STORM-LINKED outages (12h) so the index is
storm-induced; linkage share is over all outages.

Outputs (Start_From_Begining) -- a CSV per stage:
  risk_elements_by_county.csv          sub-elements, percentiles, dimensions
  correlation_subelements.csv          Spearman among the 6 sub-elements
  correlation_dimensions.csv           Spearman among the 4 dimensions
  lhs_weight_scenarios.csv             sampled weight vectors
  risk_scores_by_county.csv            R_base, R_lhs_mean, P_VH, class, dims
  risk_class_counts.csv                robustness class counts
Figures:
  fig_risk_correlation.png, fig_risk_dimension_maps.png, fig_risk_score_maps.png
Requires: pandas, numpy, matplotlib, geopandas, scipy
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, PowerNorm
from matplotlib.patches import Patch
from scipy.stats import spearmanr
from scipy.stats.qmc import LatinHypercube
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

LINKED_CSV = f"{Q1_3}/storm_linked_outage_events_2015_2025.csv"
STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#e6e6e6"

BUFFER = 12
K = 1000
SEED = 42
TOP_FRAC = 0.10
SEVERITY_P90 = "recovery"        # p90_recovery_hr
RECOVERY_METRIC = "linkage"      # storm-linkage share
GAMMA = 2.5


def load_mcc(path):
    m = pd.read_csv(path); m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    m = m[fn.notna() & m[cc].notna()]
    return dict(zip(fn[fn.notna()].astype("int64").astype(str).str.zfill(5), m[cc]))


def pctl(s):
    return s.rank(pct=True, method="average") * 100.0


def truthy(s):
    return s.astype(str).str.lower().isin(["true", "1", "1.0", "yes", "t"])


def p90(s):
    s = s.dropna()
    return float(np.percentile(s, 90)) if len(s) else np.nan


# ---- sub-elements per county ----
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
st["s_start"] = pd.to_datetime(st["start_time"], errors="coerce")
st["s_end"] = pd.to_datetime(st["end_time"], errors="coerce")
st["dur"] = ((st["s_end"] - st["s_start"]).dt.total_seconds() / 3600).clip(lower=0)
se = st.groupby("county_fips").agg(exposure=("event_type", "size"),
                                   duration=("dur", "sum")).reset_index()

o = pd.read_csv(LINKED_CSV, dtype={"county_fips": str})
o["county_fips"] = o["county_fips"].str.zfill(5)
o["auc"] = pd.to_numeric(o["auc"], errors="coerce")
o["recovery_hr"] = pd.to_numeric(o["recovery_hr"], errors="coerce")
o["linked"] = truthy(o[f"storm_linked_{BUFFER}h"])
link = o.groupby("county_fips")["linked"].mean().rename("linkage_share").reset_index()
ol = o[o["linked"]]
burden = ol.groupby("county_fips").agg(
    customer_hours=("auc", "sum"),
    p90_recovery=("recovery_hr", p90)).reset_index()

mcc = load_mcc(MCC_CSV)
d = se.merge(burden, on="county_fips", how="outer").merge(link, on="county_fips", how="outer")
sv = d["county_fips"].map(mcc).astype(float); sv = sv.where(sv > 0, np.nan)
d["norm_burden"] = d["customer_hours"] / sv
for c in ["exposure", "duration", "customer_hours", "p90_recovery",
          "linkage_share", "norm_burden"]:
    d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)

# percentiles
d["exposure_pct"] = pctl(d["exposure"])
d["duration_pct"] = pctl(d["duration"])
d["norm_burden_pct"] = pctl(d["norm_burden"])
d["p90_recovery_pct"] = pctl(d["p90_recovery"])
d["linkage_pct"] = pctl(d["linkage_share"])
d["DR"] = d["norm_burden_pct"] - d["exposure_pct"]
d["DR_pct"] = pctl(d["DR"])

# dimensions
d["Hazard"] = d[["exposure_pct", "duration_pct"]].mean(axis=1)
d["Severity"] = d[["norm_burden_pct", "p90_recovery_pct"]].mean(axis=1)
d["Fragility"] = d["DR_pct"]
d["Recovery"] = d["linkage_pct"]
DIMS = ["Hazard", "Severity", "Fragility", "Recovery"]
d.to_csv(f"{OUT_DIR}/risk_elements_by_county.csv", index=False)
print(f"counties: {len(d):,}")

# ---- pairwise correlation ----
SUBS = ["exposure_pct", "duration_pct", "norm_burden_pct", "p90_recovery_pct",
        "DR_pct", "linkage_pct"]
SUBLAB = ["Exposure", "Duration", "Norm burden", "P90 recovery",
          "DR (fragility)", "Linkage share"]
corr_sub = d[SUBS].corr(method="spearman")
corr_sub.to_csv(f"{OUT_DIR}/correlation_subelements.csv")
corr_dim = d[DIMS].corr(method="spearman")
corr_dim.to_csv(f"{OUT_DIR}/correlation_dimensions.csv")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(18, 8))
for ax, corr, labs, ttl in [(a1, corr_sub, SUBLAB, "Sub-elements"),
                            (a2, corr_dim, DIMS, "Dimensions")]:
    im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(labs))); ax.set_yticks(range(len(labs)))
    ax.set_xticklabels(labs, rotation=40, ha="right"); ax.set_yticklabels(labs)
    for i in range(len(labs)):
        for j in range(len(labs)):
            v = corr.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if abs(v) > 0.55 else "black")
    ax.set_title(f"Spearman correlation — {ttl}", fontsize=13)
    fig.colorbar(im, ax=ax, shrink=0.7)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_correlation.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# ---- baseline + LHS ----
X = d[DIMS].to_numpy()
d["R_base"] = X.mean(axis=1)
d["R_base_pct"] = pctl(d["R_base"])
u = LatinHypercube(d=4, seed=SEED).random(n=K)
e = -np.log(1.0 - u); W = e / e.sum(axis=1, keepdims=True)
pd.DataFrame(W, columns=[f"w_{c}" for c in DIMS]).assign(
    scenario=np.arange(1, K+1)).to_csv(f"{OUT_DIR}/lhs_weight_scenarios.csv", index=False)
scores = W @ X.T
n = len(d); order = np.argsort(-scores, axis=1)
ranks = np.empty_like(order); ranks[np.arange(K)[:, None], order] = np.arange(1, n+1)
kv = max(1, int(np.ceil(TOP_FRAC * n)))
d["P_VH"] = (ranks <= kv).mean(axis=0)
d["R_lhs_mean"] = scores.mean(axis=0)
d["robustness_class"] = pd.cut(d["P_VH"], bins=[-1, .0999, .4999, .8999, 2],
    labels=["rare", "weight-sensitive", "moderately robust", "robust"])

keep = ["county_fips"] + DIMS + ["R_base", "R_base_pct", "R_lhs_mean",
        "P_VH", "robustness_class"]
d[keep].to_csv(f"{OUT_DIR}/risk_scores_by_county.csv", index=False)
cc = d["robustness_class"].value_counts().reindex(
    ["robust", "moderately robust", "weight-sensitive", "rare"]).rename("n")
cc.to_frame().to_csv(f"{OUT_DIR}/risk_class_counts.csv")
print("robustness classes:\n", cc.to_string())

# ---- maps ----
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d, on="county_fips", how="left")
pn = PowerNorm(gamma=GAMMA, vmin=0, vmax=100)

# dimension maps
fig, axes = plt.subplots(2, 2, figsize=(18, 11))
for ax, dim in zip(axes.ravel(), DIMS):
    geo.plot(column=dim, cmap="magma_r", norm=pn, linewidth=0.03, edgecolor="white",
             ax=ax, legend=True, missing_kwds={"color": MISSING},
             legend_kwds={"shrink": 0.6, "label": f"{dim} (0-100)"})
    ax.set_axis_off(); ax.set_title(dim, fontsize=13)
fig.suptitle("Risk dimensions", fontsize=16, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{FIG_DIR}/fig_risk_dimension_maps.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# score maps: baseline, LHS mean, robustness class
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
geo.plot(column="R_base_pct", cmap="magma_r", norm=pn, linewidth=0.03,
         edgecolor="white", ax=axes[0], legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6, "label": "baseline risk percentile"})
axes[0].set_axis_off(); axes[0].set_title("Baseline risk (equal weight)", fontsize=13)
geo["R_lhs_pct"] = pctl(geo["R_lhs_mean"])
geo.plot(column="R_lhs_pct", cmap="magma_r", norm=pn, linewidth=0.03,
         edgecolor="white", ax=axes[1], legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6, "label": "LHS-mean risk percentile"})
axes[1].set_axis_off(); axes[1].set_title("LHS-mean risk", fontsize=13)
RC = ["robust", "moderately robust", "weight-sensitive", "rare"]
RCC = {"robust": "#67000d", "moderately robust": "#cb181d",
       "weight-sensitive": "#fc9272", "rare": "#fee0d2"}
geo[geo["robustness_class"].isna()].plot(ax=axes[2], color=MISSING,
                                         linewidth=0.03, edgecolor="white")
for c in RC:
    s = geo[geo["robustness_class"] == c]
    if len(s):
        s.plot(ax=axes[2], color=RCC[c], linewidth=0.03, edgecolor="white")
axes[2].set_axis_off(); axes[2].set_title("Robustness class (P_VH)", fontsize=13)
axes[2].legend(handles=[Patch(color=RCC[c], label=c) for c in RC],
               loc="lower left", fontsize=9, frameon=False)
fig.suptitle("Risk score maps", fontsize=16, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(f"{FIG_DIR}/fig_risk_score_maps.png", dpi=250, bbox_inches="tight")
plt.close(fig)

print(f"\nWrote CSVs + 3 figures to {OUT_DIR}")

counties: 3,204
robustness classes:
 robustness_class
robust                 15
moderately robust     214
weight-sensitive      578
rare                 2397


/local/tmp/ipykernel_4102714/51277303.py:220: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  geo[geo["robustness_class"].isna()].plot(ax=axes[2], color=MISSING,



Wrote CSVs + 3 figures to /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining


In [85]:
"""
Weight-robustness checking for the four-dimension risk model
============================================================
Reads the risk scores and LHS outputs, then checks how sensitive the county
ranking is to the choice of weights on the four dimensions
(Hazard, Severity, Fragility, Recovery):

  1. Overall stability : Spearman(R_base, R_lhs_mean) and top-10% overlap.
  2. Rank shift        : DeltaRank = Rank(R_base) - Rank(R_lhs_mean)
                         (rank 1 = highest risk; +ve = higher risk under LHS).
  3. Named scenarios   : equal vs each dimension-heavy weighting (0.4 on one,
                         0.2 on the rest) -- Spearman, top-10% overlap, and how
                         many counties enter/leave the top 10%.
  4. P_VH robustness   : counts of robust / moderately robust / weight-sensitive
                         / rare, and which baseline-high counties are unstable.

Outputs (Start_From_Begining):
  risk_weight_robustness_summary.csv     headline metrics + class counts
  risk_named_scenarios.csv               each dimension-heavy scenario vs equal
  risk_rank_shift_by_county.csv          per-county ranks and DeltaRank
Figure:
  fig_risk_weight_robustness.png         4-panel diagnostic
Requires: pandas, numpy, matplotlib, scipy
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

DIMS = ["Hazard", "Severity", "Fragility", "Recovery"]
DIM_COLOR = {"Hazard": "#2166ac", "Severity": "#b2182b",
             "Fragility": "#6a51a3", "Recovery": "#238b45"}
TOP_FRAC = 0.10


def pctl(s):
    return s.rank(pct=True, method="average") * 100.0


d = pd.read_csv(f"{OUT_DIR}/risk_scores_by_county.csv", dtype={"county_fips": str})
d["county_fips"] = d["county_fips"].str.zfill(5)
n = len(d)
k = int(np.ceil(TOP_FRAC * n))
X = d[DIMS].to_numpy()

# ---- 1. overall stability ----
rho, _ = spearmanr(d["R_base"], d["R_lhs_mean"], nan_policy="omit")
T_base = set(d.nlargest(k, "R_base")["county_fips"])
T_lhs = set(d.nlargest(k, "R_lhs_mean")["county_fips"])
overlap = len(T_base & T_lhs) / k

# ---- 2. rank shift ----
d["rank_base"] = d["R_base"].rank(ascending=False, method="min")
d["rank_lhs"] = d["R_lhs_mean"].rank(ascending=False, method="min")
d["DeltaRank"] = d["rank_base"] - d["rank_lhs"]      # + = higher risk under LHS
d["abs_DeltaRank"] = d["DeltaRank"].abs()
d[["county_fips", "R_base", "R_lhs_mean", "rank_base", "rank_lhs",
   "DeltaRank", "P_VH"]].to_csv(f"{OUT_DIR}/risk_rank_shift_by_county.csv", index=False)

# ---- 3. named scenarios ----
SCEN = {"equal": [.25, .25, .25, .25]}
for i, dim in enumerate(DIMS):
    w = [.2, .2, .2, .2]; w[i] = .4
    SCEN[f"{dim}-heavy"] = w
scores = {name: X @ np.array(w) for name, w in SCEN.items()}
ref = scores["equal"]
T_ref = set(d.iloc[np.argsort(-ref)[:k]]["county_fips"])
rows = []
for name, sc in scores.items():
    r, _ = spearmanr(sc, ref, nan_policy="omit")
    T = set(d.iloc[np.argsort(-sc)[:k]]["county_fips"])
    rows.append({"scenario": name, "w_Hazard": SCEN[name][0],
                 "w_Severity": SCEN[name][1], "w_Fragility": SCEN[name][2],
                 "w_Recovery": SCEN[name][3],
                 "spearman_vs_equal": round(r, 4),
                 "top10_overlap_pct": round(len(T & T_ref) / k * 100, 1),
                 "n_entering_top10": len(T - T_ref),
                 "n_leaving_top10": len(T_ref - T)})
scen = pd.DataFrame(rows)
scen.to_csv(f"{OUT_DIR}/risk_named_scenarios.csv", index=False)
print(scen.to_string(index=False))

# ---- 4. robustness classes ----
cls = d["robustness_class"].value_counts().reindex(
    ["robust", "moderately robust", "weight-sensitive", "rare"]).fillna(0).astype(int)
n_robust = int((d["P_VH"] >= 0.90).sum())
unstable_top = d[(d["R_base_pct"] >= 90) & (d["P_VH"] < 0.50)]

summary = pd.DataFrame([
    {"metric": "Spearman R_base vs R_lhs_mean", "value": f"{rho:.4f}"},
    {"metric": "Top-10% overlap (base vs LHS mean)",
     "value": f"{len(T_base & T_lhs)}/{k} ({overlap*100:.1f}%)"},
    {"metric": "Robust (P_VH>=0.90)", "value": str(int(cls['robust']))},
    {"metric": "Moderately robust (0.50-0.90)", "value": str(int(cls['moderately robust']))},
    {"metric": "Weight-sensitive (0.10-0.50)", "value": str(int(cls['weight-sensitive']))},
    {"metric": "Rare (<0.10)", "value": str(int(cls['rare']))},
    {"metric": "Baseline-top-10% but unstable (P_VH<0.50)", "value": str(len(unstable_top))},
    {"metric": "Most disruptive dimension (lowest Spearman vs equal)",
     "value": scen.iloc[1:].sort_values("spearman_vs_equal").iloc[0]["scenario"]},
])
summary.to_csv(f"{OUT_DIR}/risk_weight_robustness_summary.csv", index=False)
print("\n", summary.to_string(index=False))

# ---- figure ----
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# A: rank shift histogram
ax = axes[0, 0]
ax.hist(d["DeltaRank"], bins=60, color="#4575b4", edgecolor="white", linewidth=0.3)
ax.axvline(0, color="black", lw=1.2, label="No shift")
for q in (0.05, 0.95):
    v = d["DeltaRank"].quantile(q)
    ax.axvline(v, color="#d73027", ls="--", lw=1, label=f"{int(q*100)}th = {v:.0f}")
ax.set_xlabel("DeltaRank = Rank(base) - Rank(LHS mean)\n+ = higher risk under LHS")
ax.set_ylabel("Counties"); ax.set_title("A. Rank shift under reweighting")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# B: scatter base vs lhs mean
ax = axes[0, 1]
sc = ax.scatter(d["R_base"], d["R_lhs_mean"], c=d["abs_DeltaRank"], cmap="viridis_r",
                s=12, alpha=0.7, edgecolors="none",
                vmax=d["abs_DeltaRank"].quantile(0.95))
ax.plot([0, 100], [0, 100], "k--", lw=1, alpha=0.6)
fig.colorbar(sc, ax=ax, shrink=0.8, label="|DeltaRank|")
ax.set_xlabel("R_base (equal weight)"); ax.set_ylabel("R_lhs_mean")
ax.set_title(f"B. Baseline vs LHS mean\nSpearman={rho:.3f}, overlap={overlap*100:.1f}%")

# C: named scenarios bars
ax = axes[1, 0]
plot = scen[scen.scenario != "equal"]
x = np.arange(len(plot)); w = 0.4
ax.bar(x - w/2, plot["spearman_vs_equal"], w, color="#4575b4", label="Spearman vs equal")
ax2 = ax.twinx()
ax2.bar(x + w/2, plot["top10_overlap_pct"], w, color="#d73027", label="top-10% overlap %")
ax.set_xticks(x); ax.set_xticklabels(plot["scenario"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Spearman", color="#4575b4"); ax.set_ylim(0, 1.02)
ax2.set_ylabel("overlap %", color="#d73027"); ax2.set_ylim(0, 102)
ax.set_title("C. Dimension-heavy scenarios vs equal weight")

# D: baseline percentile vs P_VH
ax = axes[1, 1]
ax.scatter(d["R_base_pct"], d["P_VH"], s=12, alpha=0.5, color="#6a51a3", edgecolors="none")
ax.axvline(90, color="gray", ls="--", lw=1); ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.axhline(0.9, color="gray", ls=":", lw=1)
ax.set_xlabel("Baseline risk percentile"); ax.set_ylabel("P_VH (prob. top-10% under LHS)")
ax.set_title("D. Baseline rank vs robustness\n(bottom-right = high but weight-sensitive)")
ax.grid(alpha=0.3)

fig.suptitle("Weight-robustness of the four-dimension risk score", fontsize=16, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(f"{FIG_DIR}/fig_risk_weight_robustness.png", dpi=250, bbox_inches="tight")
plt.close(fig)
print(f"\nWrote CSVs + {FIG_DIR}/fig_risk_weight_robustness.png")

       scenario  w_Hazard  w_Severity  w_Fragility  w_Recovery  spearman_vs_equal  top10_overlap_pct  n_entering_top10  n_leaving_top10
          equal      0.25        0.25         0.25        0.25             1.0000              100.0                 0                0
   Hazard-heavy      0.40        0.20         0.20        0.20             0.9585               80.4                63               63
 Severity-heavy      0.20        0.40         0.20        0.20             0.9855               83.5                53               53
Fragility-heavy      0.20        0.20         0.40        0.20             0.9391               71.3                92               92
 Recovery-heavy      0.20        0.20         0.20        0.40             0.9571               79.1                67               67

                                               metric           value
                       Spearman R_base vs R_lhs_mean          1.0000
                  Top-10% overlap (base vs L

In [86]:
"""
Weight-robustness checking for the four-dimension risk model (honest version)
=============================================================================
Panels A and B now use the ACTUAL spread of each county's rank across the 1,000
LHS weightings -- not the LHS *mean* (which is ~equal-weight by construction and
therefore trivially agrees with the baseline). This measures real weight
sensitivity.

  A: distribution of per-county rank uncertainty = rank_p95 - rank_p5 across the
     1,000 weightings (how much a county's rank can move).
  B: rank-uncertainty band -- counties sorted by median rank, showing the
     5th-95th percentile rank band. Wide band = weight-sensitive position.
  C: dimension-heavy named scenarios vs equal weight (Spearman, overlap).
  D: baseline percentile vs P_VH.

Also prints the DIMENSION correlation matrix and flags any dimension pair with
|Spearman| >= 0.7 (redundancy that flattens weight sensitivity).

Reads: risk_scores_by_county.csv, lhs_weight_scenarios.csv, correlation_dimensions.csv
Outputs (Start_From_Begining):
  risk_rank_uncertainty_by_county.csv, risk_named_scenarios.csv,
  risk_weight_robustness_summary.csv
Figure: fig_risk_weight_robustness.png
Requires: pandas, numpy, matplotlib, scipy
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

DIMS = ["Hazard", "Severity", "Fragility", "Recovery"]
TOP_FRAC = 0.10

d = pd.read_csv(f"{OUT_DIR}/risk_scores_by_county.csv", dtype={"county_fips": str})
d["county_fips"] = d["county_fips"].str.zfill(5)
n = len(d); k = int(np.ceil(TOP_FRAC * n))
X = d[DIMS].to_numpy()

W = pd.read_csv(f"{OUT_DIR}/lhs_weight_scenarios.csv")[
    [f"w_{c}" for c in DIMS]].to_numpy()
K = len(W)

# ---- rank of every county in every scenario ----
scores = W @ X.T                                  # (K, n)
order = np.argsort(-scores, axis=1)
ranks = np.empty_like(order)
ranks[np.arange(K)[:, None], order] = np.arange(1, n + 1)   # 1 = highest risk

r5, r50, r95 = np.percentile(ranks, [5, 50, 95], axis=0)
rmin, rmax = ranks.min(axis=0), ranks.max(axis=0)
d["rank_median"] = r50
d["rank_p5"] = r5
d["rank_p95"] = r95
d["rank_width"] = r95 - r5
d["baseline_rank"] = d["R_base"].rank(ascending=False, method="min")
d[["county_fips", "baseline_rank", "rank_median", "rank_p5", "rank_p95",
   "rank_width", "P_VH"]].sort_values("rank_median").to_csv(
    f"{OUT_DIR}/risk_rank_uncertainty_by_county.csv", index=False)

top_mask = d["baseline_rank"] <= k
med_width_all = d["rank_width"].median()
med_width_top = d.loc[top_mask, "rank_width"].median()

# ---- dimension correlation ----
corr = pd.read_csv(f"{OUT_DIR}/correlation_dimensions.csv", index_col=0)
print("Dimension correlation (Spearman):")
print(corr.round(3).to_string())
pairs = []
for i in range(len(DIMS)):
    for j in range(i + 1, len(DIMS)):
        pairs.append((DIMS[i], DIMS[j], corr.iloc[i, j]))
flag = [(a, b, round(v, 3)) for a, b, v in pairs if abs(v) >= 0.7]
print("\nHighly correlated dimension pairs (|rho|>=0.7):", flag if flag else "none")

# ---- named dimension-heavy scenarios ----
SCEN = {"equal": [.25, .25, .25, .25]}
for i, dim in enumerate(DIMS):
    w = [.2, .2, .2, .2]; w[i] = .4
    SCEN[f"{dim}-heavy"] = w
sc_scores = {name: X @ np.array(w) for name, w in SCEN.items()}
ref = sc_scores["equal"]
T_ref = set(d.iloc[np.argsort(-ref)[:k]]["county_fips"])
rows = []
for name, sc in sc_scores.items():
    r, _ = spearmanr(sc, ref, nan_policy="omit")
    T = set(d.iloc[np.argsort(-sc)[:k]]["county_fips"])
    rows.append({"scenario": name, "spearman_vs_equal": round(r, 4),
                 "top10_overlap_pct": round(len(T & T_ref) / k * 100, 1),
                 "n_entering_top10": len(T - T_ref), "n_leaving_top10": len(T_ref - T)})
scen = pd.DataFrame(rows)
scen.to_csv(f"{OUT_DIR}/risk_named_scenarios.csv", index=False)
print("\n", scen.to_string(index=False))

# ---- summary ----
cls = d["robustness_class"].value_counts().reindex(
    ["robust", "moderately robust", "weight-sensitive", "rare"]).fillna(0).astype(int)
worst = scen.iloc[1:].sort_values("spearman_vs_equal").iloc[0]
summary = pd.DataFrame([
    {"metric": "Median rank uncertainty (all counties, p95-p5)",
     "value": f"{med_width_all:.0f} ranks"},
    {"metric": "Median rank uncertainty (baseline top-10% counties)",
     "value": f"{med_width_top:.0f} ranks"},
    {"metric": "Robust (P_VH>=0.90)", "value": str(int(cls["robust"]))},
    {"metric": "Moderately robust", "value": str(int(cls["moderately robust"]))},
    {"metric": "Weight-sensitive", "value": str(int(cls["weight-sensitive"]))},
    {"metric": "Rare", "value": str(int(cls["rare"]))},
    {"metric": "Most disruptive dimension (lowest Spearman vs equal)",
     "value": f"{worst['scenario']} ({worst['spearman_vs_equal']})"},
    {"metric": "Highly correlated dimension pairs (|rho|>=0.7)",
     "value": "; ".join(f"{a}-{b}={v}" for a, b, v in flag) if flag else "none"},
])
summary.to_csv(f"{OUT_DIR}/risk_weight_robustness_summary.csv", index=False)
print("\n", summary.to_string(index=False))

# ---- figure ----
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# A: rank uncertainty histogram
ax = axes[0, 0]
ax.hist(d["rank_width"], bins=50, color="#4575b4", edgecolor="white", linewidth=0.3)
ax.axvline(med_width_all, color="black", lw=1.5, label=f"median = {med_width_all:.0f}")
ax.set_xlabel("Rank uncertainty across 1,000 weightings (p95 - p5 rank)")
ax.set_ylabel("Counties")
ax.set_title("A. How much each county's rank moves under reweighting")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# B: rank band, counties sorted by median rank
ax = axes[0, 1]
srt = d.sort_values("rank_median")
x = np.arange(len(srt))
ax.fill_between(x, srt["rank_p5"], srt["rank_p95"], color="#4575b4", alpha=0.3,
                label="5th-95th pct rank band")
ax.plot(x, srt["rank_median"], color="#08306b", lw=1, label="median rank")
ax.axhline(k, color="#d73027", ls="--", lw=1, label=f"top-10% cutoff (rank {k})")
ax.set_xlabel("Counties sorted by median rank")
ax.set_ylabel("Rank across scenarios (1 = highest risk)")
ax.set_title("B. Rank-uncertainty band (wide = weight-sensitive)")
ax.invert_yaxis(); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# C: dimension-heavy scenarios
ax = axes[1, 0]
plot = scen[scen.scenario != "equal"]
xx = np.arange(len(plot)); w = 0.4
ax.bar(xx - w/2, plot["spearman_vs_equal"], w, color="#4575b4", label="Spearman vs equal")
ax2 = ax.twinx()
ax2.bar(xx + w/2, plot["top10_overlap_pct"], w, color="#d73027", label="top-10% overlap %")
ax.set_xticks(xx); ax.set_xticklabels(plot["scenario"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Spearman", color="#4575b4"); ax.set_ylim(0, 1.02)
ax2.set_ylabel("overlap %", color="#d73027"); ax2.set_ylim(0, 102)
ax.set_title("C. Dimension-heavy scenarios vs equal weight")

# D: baseline percentile vs P_VH
ax = axes[1, 1]
ax.scatter(d["R_base_pct"], d["P_VH"], s=12, alpha=0.5, color="#6a51a3", edgecolors="none")
ax.axvline(90, color="gray", ls="--", lw=1); ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.axhline(0.9, color="gray", ls=":", lw=1)
ax.set_xlabel("Baseline risk percentile"); ax.set_ylabel("P_VH (prob. top-10% under LHS)")
ax.set_title("D. Baseline rank vs robustness\n(bottom-right = high but weight-sensitive)")
ax.grid(alpha=0.3)

fig.suptitle("Weight-robustness of the four-dimension risk score", fontsize=16, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(f"{FIG_DIR}/fig_risk_weight_robustness.png", dpi=250, bbox_inches="tight")
plt.close(fig)
print(f"\nWrote CSVs + {FIG_DIR}/fig_risk_weight_robustness.png")

Dimension correlation (Spearman):
           Hazard  Severity  Fragility  Recovery
Hazard      1.000     0.163     -0.490     0.603
Severity    0.163     1.000      0.597     0.282
Fragility  -0.490     0.597      1.000    -0.249
Recovery    0.603     0.282     -0.249     1.000

Highly correlated dimension pairs (|rho|>=0.7): none

        scenario  spearman_vs_equal  top10_overlap_pct  n_entering_top10  n_leaving_top10
          equal             1.0000              100.0                 0                0
   Hazard-heavy             0.9585               80.4                63               63
 Severity-heavy             0.9855               83.5                53               53
Fragility-heavy             0.9391               71.3                92               92
 Recovery-heavy             0.9571               79.1                67               67

                                               metric                    value
      Median rank uncertainty (all counties, p95-p5

In [36]:
"""
Robust-core hotspots (P_VH >= 0.80)
===================================
Isolates the counties that remain top-10% risk under almost any weighting -- the
"counties we can stand behind" deliverable. Two tiers plus context:

    Robust core     P_VH >= 0.80   (top-10% in >=80% of the 1,000 weightings)
    Usually top-tier 0.50 <= P_VH < 0.80
    (context)       everything else, faint

Reads:  risk_scores_by_county.csv, shapefile
Writes: Results/Start_From_Begining/figures/fig_robust_core_map.png
        Results/Start_From_Begining/robust_core_counties.csv
Requires: pandas, numpy, matplotlib, geopandas
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}

CORE_T = 0.80
SEC_T = 0.50
CORE_COLOR = "#67000d"
SEC_COLOR = "#fb6a4a"
CONTEXT_COLOR = "#f0f0f0"
MISSING = "#d9d9d9"

STATE_ABBR = {
 "01":"AL","04":"AZ","05":"AR","06":"CA","08":"CO","09":"CT","10":"DE","11":"DC",
 "12":"FL","13":"GA","16":"ID","17":"IL","18":"IN","19":"IA","20":"KS","21":"KY",
 "22":"LA","23":"ME","24":"MD","25":"MA","26":"MI","27":"MN","28":"MS","29":"MO",
 "30":"MT","31":"NE","32":"NV","33":"NH","34":"NJ","35":"NM","36":"NY","37":"NC",
 "38":"ND","39":"OH","40":"OK","41":"OR","42":"PA","44":"RI","45":"SC","46":"SD",
 "47":"TN","48":"TX","49":"UT","50":"VT","51":"VA","53":"WA","54":"WV","55":"WI","56":"WY"}

d = pd.read_csv(f"{OUT_DIR}/risk_scores_by_county.csv", dtype={"county_fips": str})
d["county_fips"] = d["county_fips"].str.zfill(5)

d["tier"] = np.where(d["P_VH"] >= CORE_T, "Robust core",
            np.where(d["P_VH"] >= SEC_T, "Usually top-tier", "Context"))

gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
name_col = "NAME" if "NAME" in gdf.columns else gdf.columns[0]
names = gdf[["county_fips", name_col]].rename(columns={name_col: "county_name"})
names["state"] = names["county_fips"].str[:2].map(STATE_ABBR)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d, on="county_fips", how="left")

# ---- CSV: core + usually-top-tier, named, sorted ----
out = d[d["tier"] != "Context"].merge(names, on="county_fips", how="left")
out = out[["county_fips", "county_name", "state", "tier", "P_VH", "R_base_pct",
           "Hazard", "Severity", "Fragility", "Recovery"]].sort_values(
    ["tier", "P_VH"], ascending=[True, False])
out.to_csv(f"{OUT_DIR}/robust_core_counties.csv", index=False)

n_core = int((d["tier"] == "Robust core").sum())
n_sec = int((d["tier"] == "Usually top-tier").sum())
print(f"Robust core (P_VH>=0.80): {n_core}")
print(f"Usually top-tier (0.50-0.80): {n_sec}")
print("\nTop robust-core counties:")
print(out[out["tier"] == "Robust core"].head(20)[
    ["county_fips", "county_name", "state", "P_VH", "R_base_pct"]].to_string(index=False))

# ---- map ----
fig, ax = plt.subplots(figsize=(15, 9))
geo[geo["tier"].isna()].plot(ax=ax, color=MISSING, linewidth=0.04, edgecolor="white")
geo[geo["tier"] == "Context"].plot(ax=ax, color=CONTEXT_COLOR, linewidth=0.04,
                                   edgecolor="white")
geo[geo["tier"] == "Usually top-tier"].plot(ax=ax, color=SEC_COLOR, linewidth=0.06,
                                            edgecolor="white")
geo[geo["tier"] == "Robust core"].plot(ax=ax, color=CORE_COLOR, linewidth=0.15,
                                       edgecolor="black")
ax.set_axis_off()
ax.set_title("Robust-core storm-outage risk hotspots\n"
             "counties that stay top-10% under almost any weighting",
             fontsize=14, pad=10)
ax.legend(handles=[
    Patch(color=CORE_COLOR, label=f"Robust core  P_VH>=0.80  (n={n_core})"),
    Patch(color=SEC_COLOR, label=f"Usually top-tier  0.50-0.80  (n={n_sec})"),
    Patch(color=CONTEXT_COLOR, label="Other counties"),
    Patch(color=MISSING, label="No score")],
    loc="lower left", fontsize=10, frameon=False)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_robust_core_map.png", dpi=250, bbox_inches="tight")
plt.close(fig)
print(f"\nWrote {FIG_DIR}/fig_robust_core_map.png")
print(f"  {OUT_DIR}/robust_core_counties.csv")

Robust core (P_VH>=0.80): 40
Usually top-tier (0.50-0.80): 189

Top robust-core counties:
county_fips  county_name state  P_VH  R_base_pct
      05103     Ouachita    AR 0.977  100.000000
      05117      Prairie    AR 0.946   99.875156
      21177   Muhlenberg    KY 0.944   99.937578
      48183        Gregg    TX 0.938   99.750312
      21075       Fulton    KY 0.932   99.828340
      19099       Jasper    IA 0.930   99.781523
      26163        Wayne    MI 0.930   99.968789
      37163      Sampson    NC 0.929   99.469413
      05003       Ashley    AR 0.917   99.375780
      51037    Charlotte    VA 0.911   99.282147
      17085   Jo Daviess    IL 0.908   99.313358
      22069 Natchitoches    LA 0.904   99.719101
      37019    Brunswick    NC 0.904   99.609863
      05017       Chicot    AR 0.903   99.157303
      05149         Yell    AR 0.902   99.188514
      05019        Clark    AR 0.899   99.219725
      22111        Union    LA 0.891   99.500624
      22031      De Soto    

/local/tmp/ipykernel_4102714/2055566413.py:78: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  geo[geo["tier"].isna()].plot(ax=ax, color=MISSING, linewidth=0.04, edgecolor="white")



Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_robust_core_map.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/robust_core_counties.csv


In [37]:
import os

In [93]:
"""
Risk framework: four dimensions, LHS-weighted, with correlation + maps
======================================================================
Dimensions (each an equal-average of its sub-elements, all as percentiles):
  Hazard   = mean( storm exposure count* , storm duration* )
  Severity = mean( normalized customer-hours* , P90 recovery* )
  Fragility= DR*   where DR = norm-burden pct - exposure pct
  Recovery = storm-linkage share*   (share of outages storm-linked at 12h)
(* = percentile 0-100)

Baseline risk = mean of the four dimensions (equal weight).
LHS risk      = weights sampled on the 4-D simplex; R_lhs_mean and P_VH
                (fraction of scenarios a county is top-10%) give robustness.

INTERPRETATION SWITCHES (edit if you meant something else):
  SEVERITY_P90 = "recovery"      -> p90_recovery_hr in Severity
  RECOVERY_METRIC = "linkage"    -> storm-linkage share as the Recovery dimension
Burden/recovery are computed over STORM-LINKED outages (12h) so the index is
storm-induced; linkage share is over all outages.

Outputs (Start_From_Begining) -- a CSV per stage:
  risk_elements_by_county.csv          sub-elements, percentiles, dimensions
  correlation_subelements.csv          Spearman among the 6 sub-elements
  correlation_dimensions.csv           Spearman among the 4 dimensions
  lhs_weight_scenarios.csv             sampled weight vectors
  risk_scores_by_county.csv            R_base, R_lhs_mean, P_VH, class, dims
  risk_class_counts.csv                robustness class counts
Figures:
  fig_risk_correlation.png, fig_risk_dimension_maps.png, fig_risk_score_maps.png
Requires: pandas, numpy, matplotlib, geopandas, scipy
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, PowerNorm
from matplotlib.patches import Patch
from scipy.stats import spearmanr
from scipy.stats.qmc import LatinHypercube
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

LINKED_CSV = f"{Q1_3}/storm_linked_outage_events_2015_2025.csv"
STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#e6e6e6"

BUFFER = 12
K = 10000
SEED = 42
TOP_FRAC = 0.10
SEVERITY_P90 = "recovery"        # p90_recovery_hr
RECOVERY_METRIC = "linkage"      # storm-linkage share
GAMMA = 2.5


def load_mcc(path):
    m = pd.read_csv(path); m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    m = m[fn.notna() & m[cc].notna()]
    return dict(zip(fn[fn.notna()].astype("int64").astype(str).str.zfill(5), m[cc]))


def pctl(s):
    return s.rank(pct=True, method="average") * 100.0


def truthy(s):
    return s.astype(str).str.lower().isin(["true", "1", "1.0", "yes", "t"])


def p90(s):
    s = s.dropna()
    return float(np.percentile(s, 90)) if len(s) else np.nan


# ---- sub-elements per county ----
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
st["s_start"] = pd.to_datetime(st["start_time"], errors="coerce")
st["s_end"] = pd.to_datetime(st["end_time"], errors="coerce")
st["dur"] = ((st["s_end"] - st["s_start"]).dt.total_seconds() / 3600).clip(lower=0)
se = st.groupby("county_fips").agg(exposure=("event_type", "size"),
                                   duration=("dur", "sum")).reset_index()

o = pd.read_csv(LINKED_CSV, dtype={"county_fips": str})
o["county_fips"] = o["county_fips"].str.zfill(5)
o["auc"] = pd.to_numeric(o["auc"], errors="coerce")
o["recovery_hr"] = pd.to_numeric(o["recovery_hr"], errors="coerce")
o["linked"] = truthy(o[f"storm_linked_{BUFFER}h"])
link = o.groupby("county_fips")["linked"].mean().rename("linkage_share").reset_index()
ol = o[o["linked"]]
burden = ol.groupby("county_fips").agg(
    customer_hours=("auc", "sum"),
    p90_recovery=("recovery_hr", p90)).reset_index()

mcc = load_mcc(MCC_CSV)
d = se.merge(burden, on="county_fips", how="outer").merge(link, on="county_fips", how="outer")
sv = d["county_fips"].map(mcc).astype(float); sv = sv.where(sv > 0, np.nan)
d["norm_burden"] = d["customer_hours"] / sv
for c in ["exposure", "duration", "customer_hours", "p90_recovery",
          "linkage_share", "norm_burden"]:
    d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)

# percentiles
d["exposure_pct"] = pctl(d["exposure"])
d["duration_pct"] = pctl(d["duration"])
d["norm_burden_pct"] = pctl(d["norm_burden"])
d["p90_recovery_pct"] = pctl(d["p90_recovery"])
d["linkage_pct"] = pctl(d["linkage_share"])
d["DR"] = d["norm_burden_pct"] - d["exposure_pct"]
d["DR_pct"] = pctl(d["DR"])

# dimensions
d["Hazard"] = d[["exposure_pct", "duration_pct"]].mean(axis=1)
d["Severity"] = d[["norm_burden_pct", "p90_recovery_pct"]].mean(axis=1)
d["Fragility"] = d["DR_pct"]
d["Recovery"] = d["linkage_pct"]
DIMS = ["Hazard", "Severity", "Fragility", "Recovery"]
d.to_csv(f"{OUT_DIR}/risk_elements_by_county.csv", index=False)
print(f"counties: {len(d):,}")

# ---- pairwise correlation ----
SUBS = ["exposure_pct", "duration_pct", "norm_burden_pct", "p90_recovery_pct",
        "DR_pct", "linkage_pct"]
SUBLAB = ["Exposure", "Duration", "Norm burden", "P90 recovery",
          "DR (fragility)", "Linkage share"]
corr_sub = d[SUBS].corr(method="spearman")
corr_sub.to_csv(f"{OUT_DIR}/correlation_subelements.csv")
corr_dim = d[DIMS].corr(method="spearman")
corr_dim.to_csv(f"{OUT_DIR}/correlation_dimensions.csv")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(18, 8))
for ax, corr, labs, ttl in [(a1, corr_sub, SUBLAB, "Sub-elements"),
                            (a2, corr_dim, DIMS, "Dimensions")]:
    im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(labs))); ax.set_yticks(range(len(labs)))
    ax.set_xticklabels(labs, rotation=40, ha="right"); ax.set_yticklabels(labs)
    for i in range(len(labs)):
        for j in range(len(labs)):
            v = corr.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if abs(v) > 0.55 else "black")
    ax.set_title(f"Spearman correlation — {ttl}", fontsize=13)
    fig.colorbar(im, ax=ax, shrink=0.7)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_correlation.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# ---- baseline + LHS ----
X = d[DIMS].to_numpy()
d["R_base"] = X.mean(axis=1)
d["R_base_pct"] = pctl(d["R_base"])
u = LatinHypercube(d=4, seed=SEED).random(n=K)
e = -np.log(1.0 - u); W = e / e.sum(axis=1, keepdims=True)
pd.DataFrame(W, columns=[f"w_{c}" for c in DIMS]).assign(
    scenario=np.arange(1, K+1)).to_csv(f"{OUT_DIR}/lhs_weight_scenarios.csv", index=False)
scores = W @ X.T
n = len(d); order = np.argsort(-scores, axis=1)
ranks = np.empty_like(order); ranks[np.arange(K)[:, None], order] = np.arange(1, n+1)
kv = max(1, int(np.ceil(TOP_FRAC * n)))
d["P_VH"] = (ranks <= kv).mean(axis=0)
d["R_lhs_mean"] = scores.mean(axis=0)
d["robustness_class"] = pd.cut(d["P_VH"], bins=[-1, .0999, .4999, .8999, 2],
    labels=["rare", "weight-sensitive", "moderately robust", "robust"])

keep = ["county_fips"] + DIMS + ["R_base", "R_base_pct", "R_lhs_mean",
        "P_VH", "robustness_class"]
d[keep].to_csv(f"{OUT_DIR}/risk_scores_by_county.csv", index=False)
cc = d["robustness_class"].value_counts().reindex(
    ["robust", "moderately robust", "weight-sensitive", "rare"]).rename("n")
cc.to_frame().to_csv(f"{OUT_DIR}/risk_class_counts.csv")
print("robustness classes:\n", cc.to_string())

# ---- maps ----
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d, on="county_fips", how="left")
pn = PowerNorm(gamma=GAMMA, vmin=0, vmax=100)

# dimension maps
fig, axes = plt.subplots(2, 2, figsize=(18, 11))
for ax, dim in zip(axes.ravel(), DIMS):
    geo.plot(column=dim, cmap="magma_r", norm=pn, linewidth=0.03, edgecolor="white",
             ax=ax, legend=True, missing_kwds={"color": MISSING},
             legend_kwds={"shrink": 0.6, "label": f"{dim} (0-100)"})
    ax.set_axis_off(); ax.set_title(dim, fontsize=13)
fig.suptitle("Risk dimensions", fontsize=16, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{FIG_DIR}/fig_risk_dimension_maps.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# score maps: baseline, LHS mean, robustness class
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
geo.plot(column="R_base_pct", cmap="magma_r", norm=pn, linewidth=0.03,
         edgecolor="white", ax=axes[0], legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6, "label": "baseline risk percentile"})
axes[0].set_axis_off(); axes[0].set_title("Baseline risk (equal weight)", fontsize=13)
geo["R_lhs_pct"] = pctl(geo["R_lhs_mean"])
geo.plot(column="R_lhs_pct", cmap="magma_r", norm=pn, linewidth=0.03,
         edgecolor="white", ax=axes[1], legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6, "label": "LHS-mean risk percentile"})
axes[1].set_axis_off(); axes[1].set_title("LHS-mean risk", fontsize=13)
RC = ["robust", "moderately robust", "weight-sensitive", "rare"]
RCC = {"robust": "#67000d", "moderately robust": "#cb181d",
       "weight-sensitive": "#fc9272", "rare": "#fee0d2"}
geo[geo["robustness_class"].isna()].plot(ax=axes[2], color=MISSING,
                                         linewidth=0.03, edgecolor="white")
for c in RC:
    s = geo[geo["robustness_class"] == c]
    if len(s):
        s.plot(ax=axes[2], color=RCC[c], linewidth=0.03, edgecolor="white")
axes[2].set_axis_off(); axes[2].set_title("Robustness class (P_VH)", fontsize=13)
axes[2].legend(handles=[Patch(color=RCC[c], label=c) for c in RC],
               loc="lower left", fontsize=9, frameon=False)
fig.suptitle("Risk score maps", fontsize=16, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(f"{FIG_DIR}/fig_risk_score_maps.png", dpi=250, bbox_inches="tight")
plt.close(fig)

print(f"\nWrote CSVs + 3 figures to {OUT_DIR}")

counties: 3,204
robustness classes:
 robustness_class
robust                 10
moderately robust     216
weight-sensitive      585
rare                 2393


/local/tmp/ipykernel_4102714/2190268622.py:220: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  geo[geo["robustness_class"].isna()].plot(ax=axes[2], color=MISSING,



Wrote CSVs + 3 figures to /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining


In [92]:
"""
Weight-robustness checking for the four-dimension risk model (honest version)
=============================================================================
Panels A and B now use the ACTUAL spread of each county's rank across the 1,000
LHS weightings -- not the LHS *mean* (which is ~equal-weight by construction and
therefore trivially agrees with the baseline). This measures real weight
sensitivity.

  A: distribution of per-county rank uncertainty = rank_p95 - rank_p5 across the
     1,000 weightings (how much a county's rank can move).
  B: rank-uncertainty band -- counties sorted by median rank, showing the
     5th-95th percentile rank band. Wide band = weight-sensitive position.
  C: dimension-heavy named scenarios vs equal weight (Spearman, overlap).
  D: baseline percentile vs P_VH.

Also prints the DIMENSION correlation matrix and flags any dimension pair with
|Spearman| >= 0.7 (redundancy that flattens weight sensitivity).

Reads: risk_scores_by_county.csv, lhs_weight_scenarios.csv, correlation_dimensions.csv
Outputs (Start_From_Begining):
  risk_rank_uncertainty_by_county.csv, risk_named_scenarios.csv,
  risk_weight_robustness_summary.csv
Figure: fig_risk_weight_robustness.png
Requires: pandas, numpy, matplotlib, scipy
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

DIMS = ["Hazard", "Severity", "Fragility", "Recovery"]
TOP_FRAC = 0.10

d = pd.read_csv(f"{OUT_DIR}/risk_scores_by_county.csv", dtype={"county_fips": str})
d["county_fips"] = d["county_fips"].str.zfill(5)
n = len(d); k = int(np.ceil(TOP_FRAC * n))
X = d[DIMS].to_numpy()

W = pd.read_csv(f"{OUT_DIR}/lhs_weight_scenarios.csv")[
    [f"w_{c}" for c in DIMS]].to_numpy()
K = len(W)

# ---- rank of every county in every scenario ----
scores = W @ X.T                                  # (K, n)
order = np.argsort(-scores, axis=1)
ranks = np.empty_like(order)
ranks[np.arange(K)[:, None], order] = np.arange(1, n + 1)   # 1 = highest risk

r5, r50, r95 = np.percentile(ranks, [5, 50, 95], axis=0)
rmin, rmax = ranks.min(axis=0), ranks.max(axis=0)
d["rank_median"] = r50
d["rank_p5"] = r5
d["rank_p95"] = r95
d["rank_width"] = r95 - r5
d["baseline_rank"] = d["R_base"].rank(ascending=False, method="min")
d[["county_fips", "baseline_rank", "rank_median", "rank_p5", "rank_p95",
   "rank_width", "P_VH"]].sort_values("rank_median").to_csv(
    f"{OUT_DIR}/risk_rank_uncertainty_by_county.csv", index=False)

top_mask = d["baseline_rank"] <= k
med_width_all = d["rank_width"].median()
med_width_top = d.loc[top_mask, "rank_width"].median()

# ---- dimension correlation ----
corr = pd.read_csv(f"{OUT_DIR}/correlation_dimensions.csv", index_col=0)
print("Dimension correlation (Spearman):")
print(corr.round(3).to_string())
pairs = []
for i in range(len(DIMS)):
    for j in range(i + 1, len(DIMS)):
        pairs.append((DIMS[i], DIMS[j], corr.iloc[i, j]))
flag = [(a, b, round(v, 3)) for a, b, v in pairs if abs(v) >= 0.7]
print("\nHighly correlated dimension pairs (|rho|>=0.7):", flag if flag else "none")

# ---- named dimension-heavy scenarios ----
SCEN = {"equal": [.25, .25, .25, .25]}
for i, dim in enumerate(DIMS):
    w = [.2, .2, .2, .2]; w[i] = .4
    SCEN[f"{dim}-heavy"] = w
sc_scores = {name: X @ np.array(w) for name, w in SCEN.items()}
ref = sc_scores["equal"]
T_ref = set(d.iloc[np.argsort(-ref)[:k]]["county_fips"])
rows = []
for name, sc in sc_scores.items():
    r, _ = spearmanr(sc, ref, nan_policy="omit")
    T = set(d.iloc[np.argsort(-sc)[:k]]["county_fips"])
    rows.append({"scenario": name, "spearman_vs_equal": round(r, 4),
                 "top10_overlap_pct": round(len(T & T_ref) / k * 100, 1),
                 "n_entering_top10": len(T - T_ref), "n_leaving_top10": len(T_ref - T)})
scen = pd.DataFrame(rows)
scen.to_csv(f"{OUT_DIR}/risk_named_scenarios.csv", index=False)
print("\n", scen.to_string(index=False))

# ---- summary ----
cls = d["robustness_class"].value_counts().reindex(
    ["robust", "moderately robust", "weight-sensitive", "rare"]).fillna(0).astype(int)
worst = scen.iloc[1:].sort_values("spearman_vs_equal").iloc[0]
summary = pd.DataFrame([
    {"metric": "Median rank uncertainty (all counties, p95-p5)",
     "value": f"{med_width_all:.0f} ranks"},
    {"metric": "Median rank uncertainty (baseline top-10% counties)",
     "value": f"{med_width_top:.0f} ranks"},
    {"metric": "Robust (P_VH>=0.90)", "value": str(int(cls["robust"]))},
    {"metric": "Moderately robust", "value": str(int(cls["moderately robust"]))},
    {"metric": "Weight-sensitive", "value": str(int(cls["weight-sensitive"]))},
    {"metric": "Rare", "value": str(int(cls["rare"]))},
    {"metric": "Most disruptive dimension (lowest Spearman vs equal)",
     "value": f"{worst['scenario']} ({worst['spearman_vs_equal']})"},
    {"metric": "Highly correlated dimension pairs (|rho|>=0.7)",
     "value": "; ".join(f"{a}-{b}={v}" for a, b, v in flag) if flag else "none"},
])
summary.to_csv(f"{OUT_DIR}/risk_weight_robustness_summary.csv", index=False)
print("\n", summary.to_string(index=False))

# ---- figure ----
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# A: rank uncertainty histogram
ax = axes[0, 0]
ax.hist(d["rank_width"], bins=50, color="#4575b4", edgecolor="white", linewidth=0.3)
ax.axvline(med_width_all, color="black", lw=1.5, label=f"median = {med_width_all:.0f}")
ax.set_xlabel("Rank uncertainty across 1,000 weightings (p95 - p5 rank)")
ax.set_ylabel("Counties")
ax.set_title("A. How much each county's rank moves under reweighting")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# B: rank band for the top ~15% counties only (the decision-relevant zone)
ax = axes[0, 1]
TOPN = int(np.ceil(0.15 * n))
srt = d.sort_values("rank_median").head(TOPN).reset_index(drop=True)
x = np.arange(len(srt))
ax.fill_between(x, srt["rank_p5"], srt["rank_p95"], color="#9ecae1", alpha=0.6,
                label="5th-95th pct rank band")
ax.plot(x, srt["rank_median"], color="#08306b", lw=1.6, label="median rank")
ax.axhline(k, color="#d73027", ls="--", lw=1.5, label=f"top-10% cutoff (rank {k})")
ax.set_xlabel("Counties ranked by median risk (top 15% shown)")
ax.set_ylabel("Rank across scenarios (1 = highest risk)")
ax.set_title("B. Rank uncertainty near the hotspot cutoff\n"
             "bands below the red line = reliably top-10%")
ax.set_ylim(0, min(len(d), int(k * 3)))
ax.invert_yaxis(); ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=0.3)

# C: dimension-heavy scenarios
ax = axes[1, 0]
plot = scen[scen.scenario != "equal"]
xx = np.arange(len(plot)); w = 0.4
ax.bar(xx - w/2, plot["spearman_vs_equal"], w, color="#4575b4", label="Spearman vs equal")
ax2 = ax.twinx()
ax2.bar(xx + w/2, plot["top10_overlap_pct"], w, color="#d73027", label="top-10% overlap %")
ax.set_xticks(xx); ax.set_xticklabels(plot["scenario"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Spearman", color="#4575b4"); ax.set_ylim(0, 1.02)
ax2.set_ylabel("overlap %", color="#d73027"); ax2.set_ylim(0, 102)
ax.set_title("C. Dimension-heavy scenarios vs equal weight")

# D: P_VH vs baseline percentile, zoomed to the informative region + binned mean
ax = axes[1, 1]
hi = d[d["R_base_pct"] >= 50]
ax.scatter(hi["R_base_pct"], hi["P_VH"], s=14, alpha=0.35, color="#b3a2d4",
           edgecolors="none")
bins = np.arange(50, 101, 5)
d["_bin"] = pd.cut(d["R_base_pct"], bins=bins)
gm = d.groupby("_bin")["P_VH"].mean()
centers = [(iv.left + iv.right) / 2 for iv in gm.index.categories]
ax.plot(centers, gm.values, "-o", color="#54278f", lw=2.4, label="mean P_VH per 5-pct bin")
ax.axhline(0.8, color="#d73027", ls="--", lw=1.2, label="robust core (0.80)")
ax.axhline(0.5, color="gray", ls=":", lw=1)
ax.set_xlim(50, 100); ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("Baseline risk percentile (>=50 shown)")
ax.set_ylabel("P_VH (prob. top-10% under LHS)")
ax.set_title("D. Robustness rises steeply only at the very top")
ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3)

fig.suptitle("Weight-robustness of the four-dimension risk score", fontsize=16, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(f"{FIG_DIR}/fig_risk_weight_robustness.png", dpi=250, bbox_inches="tight")
plt.close(fig)
print(f"\nWrote CSVs + {FIG_DIR}/fig_risk_weight_robustness.png")

Dimension correlation (Spearman):
           Hazard  Severity  Fragility  Recovery
Hazard      1.000     0.163     -0.490     0.603
Severity    0.163     1.000      0.597     0.282
Fragility  -0.490     0.597      1.000    -0.249
Recovery    0.603     0.282     -0.249     1.000

Highly correlated dimension pairs (|rho|>=0.7): none

        scenario  spearman_vs_equal  top10_overlap_pct  n_entering_top10  n_leaving_top10
          equal             1.0000              100.0                 0                0
   Hazard-heavy             0.9585               80.4                63               63
 Severity-heavy             0.9855               83.5                53               53
Fragility-heavy             0.9391               71.3                92               92
 Recovery-heavy             0.9571               79.1                67               67

                                               metric                    value
      Median rank uncertainty (all counties, p95-p5

/local/tmp/ipykernel_4102714/4105308508.py:170: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  gm = d.groupby("_bin")["P_VH"].mean()



Wrote CSVs + /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_risk_weight_robustness.png


In [91]:
"""
Robust-core hotspots (P_VH >= 0.80)
===================================
Isolates the counties that remain top-10% risk under almost any weighting -- the
"counties we can stand behind" deliverable. Two tiers plus context:

    Robust core     P_VH >= 0.80   (top-10% in >=80% of the 1,000 weightings)
    Usually top-tier 0.50 <= P_VH < 0.80
    (context)       everything else, faint

Reads:  risk_scores_by_county.csv, shapefile
Writes: Results/Start_From_Begining/figures/fig_robust_core_map.png
        Results/Start_From_Begining/robust_core_counties.csv
Requires: pandas, numpy, matplotlib, geopandas
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}

CORE_T = 0.80
SEC_T = 0.50
CORE_COLOR = "#67000d"
SEC_COLOR = "#fb6a4a"
CONTEXT_COLOR = "#f0f0f0"
MISSING = "#d9d9d9"

STATE_ABBR = {
 "01":"AL","04":"AZ","05":"AR","06":"CA","08":"CO","09":"CT","10":"DE","11":"DC",
 "12":"FL","13":"GA","16":"ID","17":"IL","18":"IN","19":"IA","20":"KS","21":"KY",
 "22":"LA","23":"ME","24":"MD","25":"MA","26":"MI","27":"MN","28":"MS","29":"MO",
 "30":"MT","31":"NE","32":"NV","33":"NH","34":"NJ","35":"NM","36":"NY","37":"NC",
 "38":"ND","39":"OH","40":"OK","41":"OR","42":"PA","44":"RI","45":"SC","46":"SD",
 "47":"TN","48":"TX","49":"UT","50":"VT","51":"VA","53":"WA","54":"WV","55":"WI","56":"WY"}

d = pd.read_csv(f"{OUT_DIR}/risk_scores_by_county.csv", dtype={"county_fips": str})
d["county_fips"] = d["county_fips"].str.zfill(5)

d["tier"] = np.where(d["P_VH"] >= CORE_T, "Robust core",
            np.where(d["P_VH"] >= SEC_T, "Usually top-tier", "Context"))

gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
name_col = "NAME" if "NAME" in gdf.columns else gdf.columns[0]
names = gdf[["county_fips", name_col]].rename(columns={name_col: "county_name"})
names["state"] = names["county_fips"].str[:2].map(STATE_ABBR)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d, on="county_fips", how="left")

# ---- CSV: core + usually-top-tier, named, sorted ----
out = d[d["tier"] != "Context"].merge(names, on="county_fips", how="left")
out = out[["county_fips", "county_name", "state", "tier", "P_VH", "R_base_pct",
           "Hazard", "Severity", "Fragility", "Recovery"]].sort_values(
    ["tier", "P_VH"], ascending=[True, False])
out.to_csv(f"{OUT_DIR}/robust_core_counties.csv", index=False)

n_core = int((d["tier"] == "Robust core").sum())
n_sec = int((d["tier"] == "Usually top-tier").sum())
print(f"Robust core (P_VH>=0.80): {n_core}")
print(f"Usually top-tier (0.50-0.80): {n_sec}")
print("\nTop robust-core counties:")
print(out[out["tier"] == "Robust core"].head(20)[
    ["county_fips", "county_name", "state", "P_VH", "R_base_pct"]].to_string(index=False))

# ---- map ----
fig, ax = plt.subplots(figsize=(16, 9.5))
# faint base + state outlines for geographic context
geo.plot(ax=ax, color=CONTEXT_COLOR, linewidth=0.03, edgecolor="white")
states = geo.dissolve(by=geo["county_fips"].str[:2])
states.boundary.plot(ax=ax, color="#9e9e9e", linewidth=0.5)
# tiers
geo[geo["tier"] == "Usually top-tier"].plot(ax=ax, color=SEC_COLOR, linewidth=0.1,
                                            edgecolor="white")
geo[geo["tier"] == "Robust core"].plot(ax=ax, color=CORE_COLOR, linewidth=0.3,
                                       edgecolor="black")
# centroid dots so small core counties are visible
core = geo[geo["tier"] == "Robust core"]
if len(core):
    cent = core.geometry.representative_point()
    ax.scatter(cent.x, cent.y, s=55, facecolor=CORE_COLOR, edgecolor="black",
               linewidth=0.8, zorder=6)
sec = geo[geo["tier"] == "Usually top-tier"]
if len(sec):
    cs = sec.geometry.representative_point()
    ax.scatter(cs.x, cs.y, s=12, facecolor=SEC_COLOR, edgecolor="none",
               alpha=0.7, zorder=5)
ax.set_axis_off()
ax.set_title("Robust-core storm-outage risk hotspots\n"
             "counties that stay top-10% under almost any weighting",
             fontsize=14, pad=10)
ax.legend(handles=[
    Patch(color=CORE_COLOR, label=f"Robust core  P_VH>=0.80  (n={n_core})"),
    Patch(color=SEC_COLOR, label=f"Usually top-tier  0.50-0.80  (n={n_sec})"),
    Patch(color=CONTEXT_COLOR, label="Other counties"),
    Patch(color=MISSING, label="No score")],
    loc="lower left", fontsize=10, frameon=False)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_robust_core_map.png", dpi=250, bbox_inches="tight")
plt.close(fig)
print(f"\nWrote {FIG_DIR}/fig_robust_core_map.png")
print(f"  {OUT_DIR}/robust_core_counties.csv")

Robust core (P_VH>=0.80): 39
Usually top-tier (0.50-0.80): 187

Top robust-core counties:
county_fips  county_name state   P_VH  R_base_pct
      05103     Ouachita    AR 0.9686  100.000000
      21177   Muhlenberg    KY 0.9369   99.937578
      48183        Gregg    TX 0.9351   99.750312
      05117      Prairie    AR 0.9347   99.875156
      21075       Fulton    KY 0.9257   99.828340
      19099       Jasper    IA 0.9188   99.781523
      26163        Wayne    MI 0.9184   99.968789
      37163      Sampson    NC 0.9169   99.469413
      05003       Ashley    AR 0.9027   99.375780
      17085   Jo Daviess    IL 0.9013   99.313358
      22069 Natchitoches    LA 0.8946   99.719101
      37019    Brunswick    NC 0.8932   99.609863
      51037    Charlotte    VA 0.8925   99.282147
      05017       Chicot    AR 0.8894   99.157303
      05149         Yell    AR 0.8886   99.188514
      05095       Monroe    AR 0.8864   99.828340
      05019        Clark    AR 0.8806   99.219725
      2211

In [118]:
"""
Risk framework: four dimensions, LHS-weighted (top-50% target), maps
====================================================================
Dimensions (each an equal-average of its sub-elements, all as percentiles):
  Hazard   = mean( storm exposure count* , storm duration* )
  Severity = mean( normalized customer-hours* , P90 recovery* )
  Fragility= DR*   where DR = norm-burden pct - exposure pct
  Recovery = storm-linkage share*   (share of outages storm-linked at 12h)
(* = percentile 0-100)

Risk score = R_lhs_mean, the mean weighted score across K LHS weightings on the
4-D simplex. (Baseline equal-weight risk is dropped: it is ~identical to the
LHS mean by construction.)

P_ATL = fraction of weightings in which a county is in the TOP TARGET_FRAC of
counties. With TARGET_FRAC = 0.50 this is "reliably above-median risk", which
fills the map instead of leaving most counties in a 'rare' bucket.

Six robustness tiers on P_ATL:
  >=0.90, 0.75-0.90, 0.50-0.75, 0.25-0.50, 0.10-0.25, <0.10

Outputs (Start_From_Begining) -- a CSV per stage:
  risk_elements_by_county.csv, correlation_subelements.csv,
  correlation_dimensions.csv, lhs_weight_scenarios.csv,
  risk_scores_by_county.csv, risk_class_counts.csv
Figures:
  fig_risk_correlation.png, fig_risk_dimension_maps.png,
  fig_risk_pvh_surface.png       (continuous P_ATL choropleth)
  fig_risk_pvh_tiers.png         (6-tier classified map)
  fig_risk_lhs_mean.png          (LHS-mean risk map)
Requires: pandas, numpy, matplotlib, geopandas, scipy
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, PowerNorm, BoundaryNorm
from matplotlib.patches import Patch
from scipy.stats.qmc import LatinHypercube
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

LINKED_CSV = f"{Q1_3}/storm_linked_outage_events_2015_2025.csv"
STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#e6e6e6"

BUFFER = 12
K = 10000
SEED = 42
TARGET_FRAC = 0.50          # "reliably above-median" target (was 0.10)
GAMMA = 2.5

# six robustness tiers on P_ATL
TIER_BINS = [-0.001, 0.10, 0.25, 0.50, 0.75, 0.90, 1.001]
TIER_LABELS = ["<0.10", "0.10-0.25", "0.25-0.50", "0.50-0.75", "0.75-0.90", ">=0.90"]
TIER_COLORS = ["#f7f7f7", "#fddbc7", "#f4a582", "#d6604d", "#b2182b", "#67001f"]


def load_mcc(path):
    m = pd.read_csv(path); m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    m = m[fn.notna() & m[cc].notna()]
    return dict(zip(fn[fn.notna()].astype("int64").astype(str).str.zfill(5), m[cc]))


def pctl(s):
    return s.rank(pct=True, method="average") * 100.0


def truthy(s):
    return s.astype(str).str.lower().isin(["true", "1", "1.0", "yes", "t"])


def p90(s):
    s = s.dropna()
    return float(np.percentile(s, 90)) if len(s) else np.nan


# ---- sub-elements per county ----
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
st["s_start"] = pd.to_datetime(st["start_time"], errors="coerce")
st["s_end"] = pd.to_datetime(st["end_time"], errors="coerce")
st["dur"] = ((st["s_end"] - st["s_start"]).dt.total_seconds() / 3600).clip(lower=0)
se = st.groupby("county_fips").agg(exposure=("event_type", "size"),
                                   duration=("dur", "sum")).reset_index()

o = pd.read_csv(LINKED_CSV, dtype={"county_fips": str})
o["county_fips"] = o["county_fips"].str.zfill(5)
o["auc"] = pd.to_numeric(o["auc"], errors="coerce")
o["recovery_hr"] = pd.to_numeric(o["recovery_hr"], errors="coerce")
o["linked"] = truthy(o[f"storm_linked_{BUFFER}h"])
link = o.groupby("county_fips")["linked"].mean().rename("linkage_share").reset_index()
ol = o[o["linked"]]
burden = ol.groupby("county_fips").agg(
    customer_hours=("auc", "sum"),
    p90_recovery=("recovery_hr", p90)).reset_index()

mcc = load_mcc(MCC_CSV)
d = se.merge(burden, on="county_fips", how="outer").merge(link, on="county_fips", how="outer")
sv = d["county_fips"].map(mcc).astype(float); sv = sv.where(sv > 0, np.nan)
d["norm_burden"] = d["customer_hours"] / sv
for c in ["exposure", "duration", "customer_hours", "p90_recovery",
          "linkage_share", "norm_burden"]:
    d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)

d["exposure_pct"] = pctl(d["exposure"])
d["duration_pct"] = pctl(d["duration"])
d["norm_burden_pct"] = pctl(d["norm_burden"])
d["p90_recovery_pct"] = pctl(d["p90_recovery"])
d["linkage_pct"] = pctl(d["linkage_share"])
d["DR"] = d["norm_burden_pct"] - d["exposure_pct"]
d["DR_pct"] = pctl(d["DR"])

d["Hazard"] = d[["exposure_pct", "duration_pct"]].mean(axis=1)
d["Severity"] = d[["norm_burden_pct", "p90_recovery_pct"]].mean(axis=1)
d["Fragility"] = d["DR_pct"]
d["Recovery"] = d["linkage_pct"]
DIMS = ["Hazard", "Severity", "Fragility", "Recovery"]
d.to_csv(f"{OUT_DIR}/risk_elements_by_county.csv", index=False)
print(f"counties: {len(d):,}")

# ---- correlation ----
SUBS = ["exposure_pct", "duration_pct", "norm_burden_pct", "p90_recovery_pct",
        "DR_pct", "linkage_pct"]
SUBLAB = ["Exposure", "Duration", "Norm burden", "P90 recovery",
          "DR (fragility)", "Linkage share"]
corr_sub = d[SUBS].corr(method="spearman")
corr_sub.to_csv(f"{OUT_DIR}/correlation_subelements.csv")
corr_dim = d[DIMS].corr(method="spearman")
corr_dim.to_csv(f"{OUT_DIR}/correlation_dimensions.csv")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(18, 8))
for ax, corr, labs, ttl in [(a1, corr_sub, SUBLAB, "Sub-elements"),
                            (a2, corr_dim, DIMS, "Dimensions")]:
    im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(labs))); ax.set_yticks(range(len(labs)))
    ax.set_xticklabels(labs, rotation=40, ha="right"); ax.set_yticklabels(labs)
    for i in range(len(labs)):
        for j in range(len(labs)):
            v = corr.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if abs(v) > 0.55 else "black")
    ax.set_title(f"Spearman correlation — {ttl}", fontsize=13, fontweight="bold")
    fig.colorbar(im, ax=ax, shrink=0.7)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_correlation.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# ---- LHS ----
X = d[DIMS].to_numpy()
u = LatinHypercube(d=4, seed=SEED).random(n=K)
e = -np.log(1.0 - u); W = e / e.sum(axis=1, keepdims=True)
pd.DataFrame(W, columns=[f"w_{c}" for c in DIMS]).assign(
    scenario=np.arange(1, K + 1)).to_csv(
    f"{OUT_DIR}/lhs_weight_scenarios.csv", index=False)
scores = W @ X.T
n = len(d)
order = np.argsort(-scores, axis=1)
ranks = np.empty_like(order); ranks[np.arange(K)[:, None], order] = np.arange(1, n + 1)
kv = max(1, int(np.ceil(TARGET_FRAC * n)))          # top-50% cutoff rank
d["P_ATL"] = (ranks <= kv).mean(axis=0)             # prob. above target line
d["R_lhs_mean"] = scores.mean(axis=0)
d["R_lhs_pct"] = pctl(d["R_lhs_mean"])
d["robustness_tier"] = pd.cut(d["P_ATL"], bins=TIER_BINS, labels=TIER_LABELS)

keep = ["county_fips"] + DIMS + ["R_lhs_mean", "R_lhs_pct", "P_ATL",
        "robustness_tier"]
d[keep].to_csv(f"{OUT_DIR}/risk_scores_by_county.csv", index=False)
cc = d["robustness_tier"].value_counts().reindex(TIER_LABELS).rename("n")
cc.to_frame().to_csv(f"{OUT_DIR}/risk_class_counts.csv")
print(f"target = top {int(TARGET_FRAC*100)}% (cutoff rank {kv})")
print("robustness tiers:\n", cc.to_string())

# ---- maps ----
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d, on="county_fips", how="left")
pn = PowerNorm(gamma=GAMMA, vmin=0, vmax=100)

# dimension maps
fig, axes = plt.subplots(2, 2, figsize=(18, 11))
for ax, dim in zip(axes.ravel(), DIMS):
    geo.plot(column=dim, cmap="magma_r", norm=pn, linewidth=0.03, edgecolor="white",
             ax=ax, legend=True, missing_kwds={"color": MISSING},
             legend_kwds={"shrink": 0.6, "label": f"{dim} (0-100)"})
    ax.set_axis_off(); ax.set_title(dim, fontsize=13, fontweight="bold")
fig.suptitle("Risk dimensions", fontsize=16, fontweight="bold", y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{FIG_DIR}/fig_risk_dimension_maps.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# LHS-mean risk map (single)
fig, ax = plt.subplots(figsize=(15, 9))
geo.plot(column="R_lhs_pct", cmap="magma_r", norm=pn, linewidth=0.03,
         edgecolor="white", ax=ax, legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6, "label": "LHS-mean risk percentile"})
ax.set_axis_off()
ax.set_title("Storm-Outage Disruption Inde", fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_lhs_mean.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# continuous P_ATL surface
fig, ax = plt.subplots(figsize=(15, 9))
geo.plot(column="P_ATL", cmap="magma_r", vmin=0, vmax=1, linewidth=0.03,
         edgecolor="white", ax=ax, legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6,
                      "label": f"P(top {int(TARGET_FRAC*100)}%) across weightings"})
ax.set_axis_off()
ax.set_title(f"Robustness surface: probability of top-{int(TARGET_FRAC*100)}% risk\n"
             "under 10,000 weightings", fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_pvh_surface.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# six-tier classified map
fig, ax = plt.subplots(figsize=(15, 9))
tier_cmap = ListedColormap(TIER_COLORS)
geo["_tier_code"] = pd.Categorical(geo["robustness_tier"],
                                   categories=TIER_LABELS).codes
geo.loc[geo["_tier_code"] < 0, "_tier_code"] = np.nan
geo[geo["_tier_code"].isna()].plot(ax=ax, color=MISSING, linewidth=0.03,
                                   edgecolor="white")
geo.dropna(subset=["_tier_code"]).plot(
    column="_tier_code", cmap=tier_cmap,
    norm=BoundaryNorm(np.arange(-0.5, len(TIER_LABELS) + 0.5), tier_cmap.N),
    linewidth=0.03, edgecolor="white", ax=ax)
ax.set_axis_off()
ax.set_title(f"Robustness tiers: P(top {int(TARGET_FRAC*100)}%) across weightings",
             fontsize=15, fontweight="bold")
handles = [Patch(color=TIER_COLORS[i],
           label=f"{TIER_LABELS[i]}  (n={int((d['robustness_tier']==TIER_LABELS[i]).sum())})")
           for i in range(len(TIER_LABELS))]
handles.append(Patch(color=MISSING, label="No score"))
ax.legend(handles=handles, loc="lower left", fontsize=9, frameon=False,
          title="P(above target)")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_pvh_tiers.png", dpi=250, bbox_inches="tight")
plt.close(fig)

print(f"\nWrote CSVs + figures to {OUT_DIR}")

counties: 3,204
target = top 50% (cutoff rank 1602)
robustness tiers:
 robustness_tier
<0.10        806
0.10-0.25    308
0.25-0.50    458
0.50-0.75    516
0.75-0.90    340
>=0.90       776


/local/tmp/ipykernel_4102714/1218262691.py:237: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  geo[geo["_tier_code"].isna()].plot(ax=ax, color=MISSING, linewidth=0.03,



Wrote CSVs + figures to /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining


In [95]:
"""
Weight-robustness checking for the four-dimension risk model
===========================================================
Uses the ACTUAL spread of each county's rank across the LHS weightings.
Target tier is TOP 50% ("reliably above-median"), matching the framework.

  A: distribution of per-county rank uncertainty (p95 - p5 rank across weightings)
  B: rank-uncertainty band for the top counties, with the top-50% cutoff line
  C: dimension-heavy named scenarios vs equal weight (Spearman, overlap)
  D: LHS-mean risk percentile vs P_ATL (binned mean) -- robustness vs risk

Reads: risk_scores_by_county.csv, lhs_weight_scenarios.csv, correlation_dimensions.csv
Outputs (Start_From_Begining):
  risk_rank_uncertainty_by_county.csv, risk_named_scenarios.csv,
  risk_weight_robustness_summary.csv
Figure: fig_risk_weight_robustness.png
Requires: pandas, numpy, matplotlib, scipy
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

DIMS = ["Hazard", "Severity", "Fragility", "Recovery"]
TARGET_FRAC = 0.50
TIER_LABELS = ["<0.10", "0.10-0.25", "0.25-0.50", "0.50-0.75", "0.75-0.90", ">=0.90"]

d = pd.read_csv(f"{OUT_DIR}/risk_scores_by_county.csv", dtype={"county_fips": str})
d["county_fips"] = d["county_fips"].str.zfill(5)
n = len(d)
k = int(np.ceil(TARGET_FRAC * n))                 # top-50% cutoff rank
X = d[DIMS].to_numpy()

W = pd.read_csv(f"{OUT_DIR}/lhs_weight_scenarios.csv")[
    [f"w_{c}" for c in DIMS]].to_numpy()
K = len(W)

# ---- rank of every county in every scenario ----
scores = W @ X.T
order = np.argsort(-scores, axis=1)
ranks = np.empty_like(order)
ranks[np.arange(K)[:, None], order] = np.arange(1, n + 1)   # 1 = highest risk
r5, r50, r95 = np.percentile(ranks, [5, 50, 95], axis=0)
d["rank_median"] = r50
d["rank_p5"] = r5
d["rank_p95"] = r95
d["rank_width"] = r95 - r5
d[["county_fips", "rank_median", "rank_p5", "rank_p95", "rank_width",
   "R_lhs_pct", "P_ATL"]].sort_values("rank_median").to_csv(
    f"{OUT_DIR}/risk_rank_uncertainty_by_county.csv", index=False)

med_width_all = d["rank_width"].median()
top_mask = d["R_lhs_pct"] >= (100 * (1 - TARGET_FRAC))
med_width_top = d.loc[top_mask, "rank_width"].median()

# ---- dimension correlation ----
corr = pd.read_csv(f"{OUT_DIR}/correlation_dimensions.csv", index_col=0)
print("Dimension correlation (Spearman):")
print(corr.round(3).to_string())
flag = []
for i in range(len(DIMS)):
    for j in range(i + 1, len(DIMS)):
        v = corr.iloc[i, j]
        if abs(v) >= 0.7:
            flag.append((DIMS[i], DIMS[j], round(v, 3)))
print("\nHighly correlated dimension pairs (|rho|>=0.7):", flag if flag else "none")

# ---- named dimension-heavy scenarios (vs equal), measured on the top-50% set ----
SCEN = {"equal": [.25, .25, .25, .25]}
for i, dim in enumerate(DIMS):
    w = [.2, .2, .2, .2]; w[i] = .4
    SCEN[f"{dim}-heavy"] = w
sc_scores = {name: X @ np.array(w) for name, w in SCEN.items()}
ref = sc_scores["equal"]
T_ref = set(d.iloc[np.argsort(-ref)[:k]]["county_fips"])
rows = []
for name, sc in sc_scores.items():
    r, _ = spearmanr(sc, ref, nan_policy="omit")
    T = set(d.iloc[np.argsort(-sc)[:k]]["county_fips"])
    rows.append({"scenario": name, "spearman_vs_equal": round(r, 4),
                 "top50_overlap_pct": round(len(T & T_ref) / k * 100, 1),
                 "n_entering": len(T - T_ref), "n_leaving": len(T_ref - T)})
scen = pd.DataFrame(rows)
scen.to_csv(f"{OUT_DIR}/risk_named_scenarios.csv", index=False)
print("\n", scen.to_string(index=False))

# ---- summary ----
tiers = d["robustness_tier"].value_counts().reindex(TIER_LABELS).fillna(0).astype(int)
worst = scen.iloc[1:].sort_values("spearman_vs_equal").iloc[0]
summary_rows = [
    {"metric": "Target tier", "value": f"top {int(TARGET_FRAC*100)}%"},
    {"metric": "Median rank uncertainty (all counties, p95-p5)",
     "value": f"{med_width_all:.0f} ranks"},
    {"metric": "Median rank uncertainty (above-target counties)",
     "value": f"{med_width_top:.0f} ranks"},
    {"metric": "Most disruptive dimension (lowest Spearman vs equal)",
     "value": f"{worst['scenario']} ({worst['spearman_vs_equal']})"},
    {"metric": "Highly correlated dimension pairs (|rho|>=0.7)",
     "value": "; ".join(f"{a}-{b}={v}" for a, b, v in flag) if flag else "none"},
]
for t in TIER_LABELS:
    summary_rows.append({"metric": f"P_ATL tier {t}", "value": str(int(tiers[t]))})
pd.DataFrame(summary_rows).to_csv(
    f"{OUT_DIR}/risk_weight_robustness_summary.csv", index=False)
print("\n", pd.DataFrame(summary_rows).to_string(index=False))

# ---- figure ----
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# A: rank uncertainty histogram
ax = axes[0, 0]
ax.hist(d["rank_width"], bins=50, color="#4575b4", edgecolor="white", linewidth=0.3)
ax.axvline(med_width_all, color="black", lw=1.5, label=f"median = {med_width_all:.0f}")
ax.set_xlabel("Rank uncertainty across weightings (p95 - p5 rank)")
ax.set_ylabel("Counties")
ax.set_title("A. How much each county's rank moves under reweighting",
             fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# B: rank band for the top ~50% counties, with the top-50% cutoff
ax = axes[0, 1]
TOPN = int(np.ceil(0.55 * n))
srt = d.sort_values("rank_median").head(TOPN).reset_index(drop=True)
x = np.arange(len(srt))
ax.fill_between(x, srt["rank_p5"], srt["rank_p95"], color="#9ecae1", alpha=0.6,
                label="5th-95th pct rank band")
ax.plot(x, srt["rank_median"], color="#08306b", lw=1.6, label="median rank")
ax.axhline(k, color="#d73027", ls="--", lw=1.5, label=f"top-50% cutoff (rank {k})")
ax.set_xlabel("Counties ranked by median risk (upper ~55% shown)")
ax.set_ylabel("Rank across scenarios (1 = highest risk)")
ax.set_title("B. Rank uncertainty near the top-50% cutoff\n"
             "bands above the red line = reliably above median", fontweight="bold")
ax.invert_yaxis(); ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=0.3)

# C: dimension-heavy scenarios
ax = axes[1, 0]
plot = scen[scen.scenario != "equal"]
xx = np.arange(len(plot)); w = 0.4
ax.bar(xx - w/2, plot["spearman_vs_equal"], w, color="#4575b4", label="Spearman vs equal")
ax2 = ax.twinx()
ax2.bar(xx + w/2, plot["top50_overlap_pct"], w, color="#d73027", label="top-50% overlap %")
ax.set_xticks(xx); ax.set_xticklabels(plot["scenario"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Spearman", color="#4575b4"); ax.set_ylim(0, 1.02)
ax2.set_ylabel("overlap %", color="#d73027"); ax2.set_ylim(0, 102)
ax.set_title("C. Dimension-heavy scenarios vs equal weight", fontweight="bold")

# D: LHS-mean risk percentile vs P_ATL (binned mean)
ax = axes[1, 1]
ax.scatter(d["R_lhs_pct"], d["P_ATL"], s=12, alpha=0.3, color="#b3a2d4",
           edgecolors="none")
bins = np.arange(0, 101, 5)
d["_bin"] = pd.cut(d["R_lhs_pct"], bins=bins)
gm = d.groupby("_bin")["P_ATL"].mean()
centers = [(iv.left + iv.right) / 2 for iv in gm.index.categories]
ax.plot(centers, gm.values, "-o", color="#54278f", lw=2.4, label="mean P_ATL per 5-pct bin")
ax.axhline(0.90, color="#d73027", ls="--", lw=1.2, label="robust (0.90)")
ax.axhline(0.50, color="gray", ls=":", lw=1)
ax.set_xlim(0, 100); ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("LHS-mean risk percentile")
ax.set_ylabel(f"P(top {int(TARGET_FRAC*100)}%) across weightings")
ax.set_title("D. Robustness vs risk score", fontweight="bold")
ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3)

fig.suptitle("Weight-robustness of the four-dimension risk score",
             fontsize=16, fontweight="bold", y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(f"{FIG_DIR}/fig_risk_weight_robustness.png", dpi=250, bbox_inches="tight")
plt.close(fig)
print(f"\nWrote CSVs + {FIG_DIR}/fig_risk_weight_robustness.png")

Dimension correlation (Spearman):
           Hazard  Severity  Fragility  Recovery
Hazard      1.000     0.163     -0.490     0.603
Severity    0.163     1.000      0.597     0.282
Fragility  -0.490     0.597      1.000    -0.249
Recovery    0.603     0.282     -0.249     1.000

Highly correlated dimension pairs (|rho|>=0.7): none

        scenario  spearman_vs_equal  top50_overlap_pct  n_entering  n_leaving
          equal             1.0000              100.0           0          0
   Hazard-heavy             0.9585               90.1         158        158
 Severity-heavy             0.9855               94.8          84         84
Fragility-heavy             0.9391               87.8         196        196
 Recovery-heavy             0.9571               89.6         167        167

                                               metric                    value
                                         Target tier                  top 50%
      Median rank uncertainty (all counties, 

/local/tmp/ipykernel_4102714/2730200081.py:161: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  gm = d.groupby("_bin")["P_ATL"].mean()



Wrote CSVs + /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_risk_weight_robustness.png


In [96]:
"""
Supplementary figure — 4 dimension maps + 4 robustness diagnostics (8 panels)
=============================================================================
Top 2x2 : the four risk-dimension maps (Hazard, Severity, Fragility, Recovery)
Bottom  : the four weight-robustness diagnostics (A rank-shift hist, B rank
          band, C dimension-heavy scenarios, D robustness-vs-risk)

Reads only already-saved outputs (fast, no recompute):
  risk_scores_by_county.csv, lhs_weight_scenarios.csv,
  risk_rank_uncertainty_by_county.csv, risk_named_scenarios.csv, shapefile
Writes: Results/Start_From_Begining/figures/fig_supp_dims_robustness.png
Requires: pandas, numpy, matplotlib, geopandas
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#e6e6e6"
DIMS = ["Hazard", "Severity", "Fragility", "Recovery"]
GAMMA = 2.5
TARGET_FRAC = 0.50

d = pd.read_csv(f"{OUT_DIR}/risk_scores_by_county.csv", dtype={"county_fips": str})
d["county_fips"] = d["county_fips"].str.zfill(5)
n = len(d); k = int(np.ceil(TARGET_FRAC * n))
unc = pd.read_csv(f"{OUT_DIR}/risk_rank_uncertainty_by_county.csv",
                  dtype={"county_fips": str})
scen = pd.read_csv(f"{OUT_DIR}/risk_named_scenarios.csv")

gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d, on="county_fips", how="left")
pn = PowerNorm(gamma=GAMMA, vmin=0, vmax=100)

fig = plt.figure(figsize=(20, 20))
gs = fig.add_gridspec(4, 2, height_ratios=[1, 1, 0.85, 0.85], hspace=0.28, wspace=0.08)

# ---- top: four dimension maps ----
for i, dim in enumerate(DIMS):
    ax = fig.add_subplot(gs[i // 2, i % 2])
    geo.plot(column=dim, cmap="magma_r", norm=pn, linewidth=0.03, edgecolor="white",
             ax=ax, legend=True, missing_kwds={"color": MISSING},
             legend_kwds={"shrink": 0.6, "label": f"{dim} (0-100)"})
    ax.set_axis_off()
    ax.set_title(f"{chr(65+i)}. {dim}", fontsize=13, fontweight="bold")

# ---- bottom: robustness diagnostics ----
# E: rank-shift histogram
axE = fig.add_subplot(gs[2, 0])
med = unc["rank_width"].median()
axE.hist(unc["rank_width"], bins=50, color="#4575b4", edgecolor="white", linewidth=0.3)
axE.axvline(med, color="black", lw=1.5, label=f"median = {med:.0f}")
axE.set_xlabel("Rank uncertainty (p95 - p5 rank)"); axE.set_ylabel("Counties")
axE.set_title("E. Rank movement under reweighting", fontsize=12, fontweight="bold")
axE.legend(fontsize=8); axE.grid(alpha=0.3)

# F: rank band (upper ~55%)
axF = fig.add_subplot(gs[2, 1])
srt = unc.sort_values("rank_median").head(int(np.ceil(0.55 * n))).reset_index(drop=True)
x = np.arange(len(srt))
axF.fill_between(x, srt["rank_p5"], srt["rank_p95"], color="#9ecae1", alpha=0.6,
                 label="5th-95th pct band")
axF.plot(x, srt["rank_median"], color="#08306b", lw=1.4, label="median rank")
axF.axhline(k, color="#d73027", ls="--", lw=1.4, label=f"top-50% cutoff (rank {k})")
axF.set_xlabel("Counties by median risk (upper ~55%)")
axF.set_ylabel("Rank across scenarios"); axF.invert_yaxis()
axF.set_title("F. Rank uncertainty near the cutoff", fontsize=12, fontweight="bold")
axF.legend(fontsize=8, loc="lower right"); axF.grid(alpha=0.3)

# G: dimension-heavy scenarios
axG = fig.add_subplot(gs[3, 0])
plot = scen[scen.scenario != "equal"]
xx = np.arange(len(plot)); w = 0.4
ov_col = "top50_overlap_pct" if "top50_overlap_pct" in plot.columns else "top10_overlap_pct"
axG.bar(xx - w/2, plot["spearman_vs_equal"], w, color="#4575b4", label="Spearman")
axG2 = axG.twinx()
axG2.bar(xx + w/2, plot[ov_col], w, color="#d73027", label="overlap %")
axG.set_xticks(xx); axG.set_xticklabels(plot["scenario"], rotation=20, ha="right", fontsize=8)
axG.set_ylabel("Spearman", color="#4575b4"); axG.set_ylim(0, 1.02)
axG2.set_ylabel("overlap %", color="#d73027"); axG2.set_ylim(0, 102)
axG.set_title("G. Dimension-heavy scenarios vs equal", fontsize=12, fontweight="bold")

# H: robustness vs risk score
axH = fig.add_subplot(gs[3, 1])
axH.scatter(d["R_lhs_pct"], d["P_ATL"], s=10, alpha=0.3, color="#b3a2d4", edgecolors="none")
bins = np.arange(0, 101, 5); d["_b"] = pd.cut(d["R_lhs_pct"], bins=bins)
gm = d.groupby("_b")["P_ATL"].mean()
centers = [(iv.left + iv.right) / 2 for iv in gm.index.categories]
axH.plot(centers, gm.values, "-o", color="#54278f", lw=2.2, label="mean P_ATL per bin")
axH.axhline(0.9, color="#d73027", ls="--", lw=1.1, label="robust (0.90)")
axH.axhline(0.5, color="gray", ls=":", lw=1)
axH.set_xlim(0, 100); axH.set_ylim(-0.02, 1.02)
axH.set_xlabel("LHS-mean risk percentile"); axH.set_ylabel(f"P(top {int(TARGET_FRAC*100)}%)")
axH.set_title("H. Robustness vs risk score", fontsize=12, fontweight="bold")
axH.legend(fontsize=8, loc="upper left"); axH.grid(alpha=0.3)

fig.suptitle("Supplementary: risk dimensions (A-D) and weight-robustness (E-H)",
             fontsize=17, fontweight="bold", y=0.995)
fig.savefig(f"{FIG_DIR}/fig_supp_dims_robustness.png", dpi=250, bbox_inches="tight")
plt.close(fig)
print(f"Wrote {FIG_DIR}/fig_supp_dims_robustness.png")

/local/tmp/ipykernel_4102714/3187445142.py:98: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  gm = d.groupby("_b")["P_ATL"].mean()


Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_supp_dims_robustness.png


In [102]:
"""
Weight-robustness of the four-dimension storm-outage risk framework
===================================================================
Layout (3x2):
  A: LHS-mean risk percentile map        B: robustness map P(top 50%)
  C: county rank-uncertainty histogram   D: dimension correlation (blue = high)
  E: dimension-heavy scenarios vs equal  F: robustness vs risk score

COLORS: map colormap is set once via MAP_CMAP. Swap it to taste, e.g.
  "YlOrRd" (light->dark red, current), "cividis", "viridis", "YlGnBu",
  "rocket"* (*needs seaborn). Correlation heatmap uses RdBu (blue = high +).

Reads:  risk_scores_by_county.csv, lhs_weight_scenarios.csv,
        correlation_dimensions.csv, shapefile
Outputs: risk_rank_uncertainty_by_county.csv, risk_named_scenarios.csv,
         risk_weight_robustness_summary.csv, fig_risk_weight_robustness.png
Requires: pandas, numpy, matplotlib, scipy, geopandas
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#e6e6e6"

DIMS = ["Hazard", "Severity", "Fragility", "Recovery"]
TARGET_FRAC = 0.50
TIER_LABELS = ["<0.10", "0.10-0.25", "0.25-0.50", "0.50-0.75", "0.75-0.90", ">=0.90"]

# ---- MAP COLORS (two distinct maps) ----
RISK_CMAP = "cividis"        # A: risk magnitude (standalone figure)
ROBUST_CMAP = "plasma"       # B: robustness probability (try: Purples, Blues, cividis)
CORR_CMAP = "RdBu"           # blue = high positive correlation

d = pd.read_csv(f"{OUT_DIR}/risk_scores_by_county.csv", dtype={"county_fips": str})
d["county_fips"] = d["county_fips"].str.zfill(5)
n = len(d); k = int(np.ceil(TARGET_FRAC * n))
X = d[DIMS].to_numpy()
W = pd.read_csv(f"{OUT_DIR}/lhs_weight_scenarios.csv")[[f"w_{c}" for c in DIMS]].to_numpy()
K = len(W)

# ---- ranks across weightings ----
scores = W @ X.T
order = np.argsort(-scores, axis=1)
ranks = np.empty_like(order); ranks[np.arange(K)[:, None], order] = np.arange(1, n + 1)
r5, r50, r95 = np.percentile(ranks, [5, 50, 95], axis=0)
d["rank_median"], d["rank_p5"], d["rank_p95"] = r50, r5, r95
d["rank_width"] = d["rank_p95"] - d["rank_p5"]
d[["county_fips", "rank_median", "rank_p5", "rank_p95", "rank_width",
   "R_lhs_pct", "P_ATL"]].sort_values("rank_median").to_csv(
    f"{OUT_DIR}/risk_rank_uncertainty_by_county.csv", index=False)
med_width_all = d["rank_width"].median()
med_width_top = d.loc[d["R_lhs_pct"] >= 100 * (1 - TARGET_FRAC), "rank_width"].median()

# ---- dimension correlation ----
corr = pd.read_csv(f"{OUT_DIR}/correlation_dimensions.csv", index_col=0).loc[DIMS, DIMS]
flag = [(DIMS[i], DIMS[j], round(corr.iloc[i, j], 3))
        for i in range(len(DIMS)) for j in range(i + 1, len(DIMS))
        if abs(corr.iloc[i, j]) >= 0.7]

# ---- dimension-heavy scenarios ----
SCEN = {"equal": [.25, .25, .25, .25]}
for i, dim in enumerate(DIMS):
    w = [.2, .2, .2, .2]; w[i] = .4; SCEN[f"{dim}-heavy"] = w
sc_scores = {name: X @ np.array(w) for name, w in SCEN.items()}
ref = sc_scores["equal"]
T_ref = set(d.iloc[np.argsort(-ref)[:k]]["county_fips"])
rows = []
for name, sc in sc_scores.items():
    r, _ = spearmanr(sc, ref, nan_policy="omit")
    T = set(d.iloc[np.argsort(-sc)[:k]]["county_fips"])
    rows.append({"scenario": name, "spearman_vs_equal": round(r, 4),
                 "top50_overlap_pct": round(len(T & T_ref) / k * 100, 1),
                 "n_entering": len(T - T_ref), "n_leaving": len(T_ref - T)})
scen = pd.DataFrame(rows)
scen.to_csv(f"{OUT_DIR}/risk_named_scenarios.csv", index=False)

# ---- summary ----
tiers = d["robustness_tier"].value_counts().reindex(TIER_LABELS).fillna(0).astype(int)
worst = scen.iloc[1:].sort_values("spearman_vs_equal").iloc[0]
srows = [{"metric": "Target tier", "value": f"top {int(TARGET_FRAC*100)}%"},
         {"metric": "Median rank uncertainty (all)", "value": f"{med_width_all:.0f}"},
         {"metric": "Median rank uncertainty (above-target)", "value": f"{med_width_top:.0f}"},
         {"metric": "Most disruptive dimension", "value": f"{worst['scenario']} ({worst['spearman_vs_equal']})"},
         {"metric": "Highly correlated pairs (|rho|>=0.7)",
          "value": "; ".join(f"{a}-{b}={v}" for a, b, v in flag) if flag else "none"}]
for t in TIER_LABELS:
    srows.append({"metric": f"P_ATL tier {t}", "value": str(int(tiers[t]))})
pd.DataFrame(srows).to_csv(f"{OUT_DIR}/risk_weight_robustness_summary.csv", index=False)

# ---- shapefile ----
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d, on="county_fips", how="left")

# ============================================================
# FIGURE
# ============================================================
# ---- standalone figure: A. LHS-mean risk map (viridis) ----
figA, axA = plt.subplots(figsize=(15, 9))
geo.plot(column="R_lhs_pct", cmap=RISK_CMAP, vmin=0, vmax=100, linewidth=0.025,
         edgecolor="white", ax=axA, legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6, "pad": 0.01, "label": "LHS-mean risk percentile"})
axA.set_axis_off()
axA.set_title("Storm-Outage Disruption Index", fontsize=15, fontweight="bold")
figA.tight_layout()
figA.savefig(f"{FIG_DIR}/fig_risk_lhs_mean.png", dpi=600, bbox_inches="tight")
plt.close(figA)

# ---- robustness figure: B full-width top row, then C/D and E/F ----
fig = plt.figure(figsize=(18, 20))
gs = fig.add_gridspec(3, 2, height_ratios=[1.25, 0.9, 0.9], hspace=0.22, wspace=0.20)

# B: robustness map, full width
ax = fig.add_subplot(gs[0, :])
geo.plot(column="P_ATL", cmap=ROBUST_CMAP, vmin=0, vmax=1, linewidth=0.025,
         edgecolor="white", ax=ax, legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.5, "pad": 0.01,
                      "label": f"P(top {int(TARGET_FRAC*100)}%) across weightings"})
ax.set_axis_off()
ax.set_title("A. Weight Robustness: probability of top-50% risk across 10,000 weightings",
             fontsize=15, fontweight="bold", pad=6)

# C: rank uncertainty histogram
ax = fig.add_subplot(gs[1, 0])
ax.hist(d["rank_width"], bins=50, color="#4575b4", edgecolor="white", linewidth=0.3)
ax.axvline(med_width_all, color="black", lw=1.5, label=f"Median = {med_width_all:.0f}")
ax.set_xlabel("Rank uncertainty across weightings (p95 - p5 rank)")
ax.set_ylabel("Number of counties")
ax.set_title("B. County Rank Uncertainty Under Reweighting", fontsize=13, fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# D: dimension correlation heatmap (blue = high)
ax = fig.add_subplot(gs[1, 1])
im = ax.imshow(corr.values, cmap=CORR_CMAP, vmin=-1, vmax=1)
ax.set_xticks(range(len(DIMS))); ax.set_yticks(range(len(DIMS)))
ax.set_xticklabels(DIMS, rotation=30, ha="right", fontsize=10)
ax.set_yticklabels(DIMS, fontsize=10)
for i in range(len(DIMS)):
    for j in range(len(DIMS)):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=10,
                fontweight="bold", color="white" if abs(v) > 0.55 else "black")
ax.set_title("C. Correlation Among Risk Dimensions", fontsize=13, fontweight="bold")
fig.colorbar(im, ax=ax, shrink=0.78, pad=0.03).set_label("Spearman correlation", fontsize=10)

# E: dimension-heavy scenarios
ax = fig.add_subplot(gs[2, 0])
plot = scen[scen["scenario"] != "equal"].copy()
xx = np.arange(len(plot)); bw = 0.36
ax.bar(xx - bw/2, plot["spearman_vs_equal"], bw, color="#4575b4", label="Spearman vs equal")
axr = ax.twinx()
axr.bar(xx + bw/2, plot["top50_overlap_pct"], bw, color="#d73027", label="Top-50% overlap")
ax.set_xticks(xx)
ax.set_xticklabels(plot["scenario"].str.replace("-heavy", "", regex=False), fontsize=10)
ax.set_ylabel("Spearman correlation", color="#4575b4"); ax.set_ylim(0, 1.02)
axr.set_ylabel("Top-50% overlap (%)", color="#d73027"); axr.set_ylim(0, 102)
ax.set_title("D. Dimension-Heavy Scenarios vs. Equal Weighting", fontsize=13, fontweight="bold")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = axr.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="lower left"); ax.grid(alpha=0.25, axis="y")

# F: robustness vs risk
ax = fig.add_subplot(gs[2, 1])
ax.scatter(d["R_lhs_pct"], d["P_ATL"], s=12, alpha=0.30, color="#b3a2d4", edgecolors="none")
bins = np.arange(0, 101, 5)
d["_bin"] = pd.cut(d["R_lhs_pct"], bins=bins, include_lowest=True)
gm = d.groupby("_bin", observed=False)["P_ATL"].mean()
centers = np.array([(iv.left + iv.right) / 2 for iv in gm.index])
ax.plot(centers, gm.values, "-o", color="#54278f", lw=2.4, markersize=4,
        label="Mean P(top 50%) per 5-pct bin")
ax.axhline(0.90, color="#d73027", ls="--", lw=1.2, label="High robustness (0.90)")
ax.axhline(0.50, color="gray", ls=":", lw=1)
ax.set_xlim(0, 100); ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("LHS-mean risk percentile")
ax.set_ylabel(f"P(top {int(TARGET_FRAC*100)}%) across weightings")
ax.set_title("E. Weight Robustness vs. Risk Score", fontsize=13, fontweight="bold")
ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3)

fig.suptitle("Weight Robustness of the Four-Dimension Storm-Outage Risk Framework",
             fontsize=18, fontweight="bold", y=0.995)
fig.subplots_adjust(top=0.95, bottom=0.05, left=0.055, right=0.95)
fig.savefig(f"{FIG_DIR}/fig_risk_weight_robustness.png", dpi=600, bbox_inches="tight")
plt.close(fig)
print(f"Wrote {FIG_DIR}/fig_risk_lhs_mean.png (risk, {RISK_CMAP})")
print(f"Wrote {FIG_DIR}/fig_risk_weight_robustness.png (robustness, {ROBUST_CMAP})")

Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_risk_lhs_mean.png (risk, cividis)
Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_risk_weight_robustness.png (robustness, plasma)


In [109]:
"""
Combined trends + seasonal figure (6 panels)
============================================
  A total outage customer-hours per year
  B number of outage events per year
  C customer-hours by hazard per year (stacked)
  D counties holding 50% of that year's burden
  E seasonal timing by hazard (month x hazard heatmap, row-scaled, wider panel,
    horizontal colorbar, white cell borders)
  F monthly outage burden by hazard (absolute stacked area, single axis)

Reads:  storm_linked_outage_events_2015_2025.csv, county_level_storm_records_2015_2025.csv
Writes: Results/Start_From_Begining/figures/fig_trends_and_seasonal.png
        Results/Start_From_Begining/trends_11yr.csv, seasonal_hazard_burden.csv
Requires: pandas, numpy, matplotlib (seaborn optional, for 'rocket'/'mako')
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except Exception:
    sns = None

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

LINKED_CSV = f"{Q1_3}/storm_linked_outage_events_2015_2025.csv"
STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
BUFFER = 12
HAZARDS = ["Thunderstorm Wind", "Flood", "Tornado"]
HAZ_COLOR = {"Thunderstorm Wind": "#2166ac", "Flood": "#238b45", "Tornado": "#d94801"}
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# heatmap colormap. try: "mako", "rocket", "magma", "YlGnBu", "cividis"
HEAT_CMAP = "crest"


def cmap_of(name):
    if name in ("rocket", "mako", "flare", "crest"):
        return sns.color_palette(name, as_cmap=True) if sns is not None else "magma"
    return name


STORM_MAP = {
    "Thunderstorm Wind": "Thunderstorm Wind", "Lightning": "Thunderstorm Wind",
    "Hail": "Thunderstorm Wind",
    "Hurricane": "Tropical Cyclone", "Hurricane (Typhoon)": "Tropical Cyclone",
    "Tropical Storm": "Tropical Cyclone", "Tropical Depression": "Tropical Cyclone",
    "Storm Surge/Tide": "Tropical Cyclone",
    "Flood": "Flood", "Flash Flood": "Flood", "Coastal Flood": "Flood",
    "Heavy Rain": "Flood", "Excessive Rainfall": "Flood",
    "Tornado": "Tornado", "Funnel Cloud": "Tornado", "Waterspout": "Tornado",
    "Winter Storm": "Winter Storm", "Winter Weather": "Winter Storm",
    "Blizzard": "Winter Storm", "Ice Storm": "Winter Storm",
    "Heavy Snow": "Winter Storm", "Sleet": "Winter Storm",
    "Lake-Effect Snow": "Winter Storm", "High Wind": "High Wind", "Strong Wind": "High Wind",
}


def truthy(s):
    return s.astype(str).str.lower().isin(["true", "1", "1.0", "yes", "t"])


def counties_to_50(v):
    v = np.sort(np.asarray(v, float))[::-1]; v = v[v > 0]
    if v.size == 0:
        return np.nan
    cum = np.cumsum(v) / v.sum()
    return int(np.searchsorted(cum, 0.5) + 1)


st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
st["hazard"] = st["event_type"].map(STORM_MAP).fillna("Other")
st["s_start"] = pd.to_datetime(st["start_time"], errors="coerce")
st["s_end"] = pd.to_datetime(st["end_time"], errors="coerce")
st = st.dropna(subset=["s_start", "s_end"])
storms_by_c = {c: g for c, g in st.groupby("county_fips")}

o = pd.read_csv(LINKED_CSV, dtype={"county_fips": str})
o["county_fips"] = o["county_fips"].str.zfill(5)
o["auc"] = pd.to_numeric(o["auc"], errors="coerce").fillna(0)
o["year"] = pd.to_numeric(o["year"], errors="coerce")
o["peak_dt"] = pd.to_datetime(o["peak_time"], errors="coerce").fillna(
    pd.to_datetime(o["event_start"], errors="coerce"))
o = o.dropna(subset=["peak_dt", "year"])
o["year"] = o["year"].astype(int)
o["month"] = o["peak_dt"].dt.month
o["linked12"] = truthy(o[f"storm_linked_{BUFFER}h"])
years = sorted(o["year"].unique())

per_year = o.groupby("year").agg(
    total_customer_hours=("auc", "sum"),
    n_outage_events=("county_fips", "size")).reindex(years)
per_year["counties_to_50pct"] = [
    counties_to_50(o[o["year"] == y].groupby("county_fips")["auc"].sum().values)
    for y in years]
per_year.to_csv(f"{OUT_DIR}/trends_11yr.csv")


def attribute(o_df, buffer_hr):
    buf = np.timedelta64(buffer_hr, "h")
    haz = np.array([None] * len(o_df), dtype=object)
    pos = {ix: i for i, ix in enumerate(o_df.index)}
    for c, og in o_df.groupby("county_fips"):
        sg = storms_by_c.get(c)
        if sg is None or len(sg) == 0:
            continue
        ss = sg["s_start"].values.astype("datetime64[ns]")
        se = sg["s_end"].values.astype("datetime64[ns]")
        sh = sg["hazard"].values
        peaks = og["peak_dt"].values.astype("datetime64[ns]")
        loc = [pos[ix] for ix in og.index]
        ch = max(1, min(4000, int(2_000_000 / max(1, len(sg)))))
        for i in range(0, len(peaks), ch):
            P = peaks[i:i+ch][:, None]
            dist = np.maximum(np.maximum(ss[None, :] - P, P - se[None, :]),
                              np.timedelta64(0, "ns"))
            within = dist <= buf
            df_ = dist.astype("timedelta64[s]").astype(float); df_[~within] = np.inf
            j = df_.argmin(axis=1); any_ = within.any(axis=1)
            sel = np.where(any_, sh[j], None)
            for r, val in enumerate(sel):
                haz[loc[i+r]] = val
    return pd.Series(haz, index=o_df.index)


ol = o[o["linked12"]].copy()
ol["hazard"] = attribute(ol, BUFFER)
olh = ol[ol["hazard"].isin(HAZARDS)]
hz_year = (olh.groupby(["year", "hazard"])["auc"].sum().unstack("hazard")
           .reindex(index=years, columns=HAZARDS).fillna(0.0))
mat = (olh.groupby(["hazard", "month"])["auc"].sum().unstack("month")
       .reindex(index=HAZARDS, columns=range(1, 13)).fillna(0.0))
mat_out = mat.copy(); mat_out.columns = MONTHS
mat_out.to_csv(f"{OUT_DIR}/seasonal_hazard_burden.csv")

# ---- figure: wider middle column so heatmap can breathe ----
fig = plt.figure(figsize=(20, 10.5))
gs = fig.add_gridspec(2, 3, width_ratios=[1, 1.35, 1], hspace=0.32, wspace=0.28)
axes = np.empty((2, 3), dtype=object)
for r in range(2):
    for c in range(3):
        axes[r, c] = fig.add_subplot(gs[r, c])
yr = np.array(years)


def line(ax, y, color, title, ylab):
    ax.plot(yr, y, "-o", color=color, lw=2.2)
    ax.set_title(title, fontsize=12, fontweight="bold"); ax.set_ylabel(ylab)
    ax.grid(alpha=0.3); ax.set_xticks(yr); ax.tick_params(axis="x", rotation=45)


line(axes[0, 0], per_year["total_customer_hours"].values, "#b2182b",
     "A. Total outage customer-hours", "customer-hours")
line(axes[0, 1], per_year["n_outage_events"].values, "#2166ac",
     "B. Number of outage events", "events")
axC = axes[0, 2]
bottom = np.zeros(len(yr))
for h in HAZARDS:
    axC.bar(yr, hz_year[h].values, bottom=bottom, color=HAZ_COLOR[h], label=h)
    bottom += hz_year[h].values
axC.set_title("C. Customer-hours by hazard", fontsize=12, fontweight="bold")
axC.set_ylabel("customer-hours"); axC.set_xticks(yr)
axC.tick_params(axis="x", rotation=45); axC.legend(fontsize=8)
line(axes[1, 0], per_year["counties_to_50pct"].values, "#d94801",
     "D. Counties holding 50% of burden", "# counties")

# E heatmap (wider panel, horizontal colorbar, white borders)
axE = axes[1, 1]
row_norm = mat.div(mat.max(axis=1).replace(0, np.nan), axis=0)
im = axE.imshow(row_norm.values, cmap=cmap_of(HEAT_CMAP), aspect="auto", vmin=0, vmax=1)
axE.set_xticks(range(12)); axE.set_xticklabels(MONTHS, fontsize=8)
axE.set_yticks(range(len(HAZARDS))); axE.set_yticklabels(HAZARDS, fontsize=9)
axE.set_xticks(np.arange(-.5, 12, 1), minor=True)
axE.set_yticks(np.arange(-.5, len(HAZARDS), 1), minor=True)
axE.grid(which="minor", color="white", linewidth=1.5)
axE.tick_params(which="both", length=0)
for sp in axE.spines.values():
    sp.set_visible(False)
for i in range(len(HAZARDS)):
    for j in range(12):
        v = row_norm.values[i, j]
        if not np.isnan(v):
            axE.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=7,
                     color="white" if v > 0.55 else "#dddddd")
axE.set_title("E. Seasonal timing by hazard", fontsize=12, fontweight="bold")
cb = fig.colorbar(im, ax=axE, orientation="horizontal", shrink=0.9, pad=0.20,
                  fraction=0.05)
cb.set_label("share of hazard's peak month", fontsize=8)
cb.ax.tick_params(labelsize=7)

# F monthly hazard burden (absolute stacked area)
axF = axes[1, 2]
xs = np.arange(12); bottom = np.zeros(12)
for h in HAZARDS:
    y = mat.loc[h].values
    axF.fill_between(xs, bottom, bottom + y, color=HAZ_COLOR[h], alpha=0.85, label=h)
    bottom += y
axF.set_xticks(range(12)); axF.set_xticklabels(MONTHS, rotation=45, fontsize=8)
axF.set_ylabel("customer-hours"); axF.margins(x=0)
axF.set_title("F. Monthly outage burden by hazard", fontsize=12, fontweight="bold")
axF.legend(fontsize=8, loc="upper right")

fig.suptitle("Storm-induced outage impact: yearly trends and seasonality, 2015-2025",
             fontsize=18, fontweight="bold", y=0.995)
fig.subplots_adjust(top=0.93, bottom=0.09, left=0.05, right=0.97)
fig.savefig(f"{FIG_DIR}/fig_trends_and_seasonal.png", dpi=600, bbox_inches="tight")
plt.close(fig)
print(f"Wrote {FIG_DIR}/fig_trends_and_seasonal.png (heatmap={HEAT_CMAP})")

Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_trends_and_seasonal.png (heatmap=crest)


In [110]:
"""
Concentration tier maps — normalized burden & number of outage events
=====================================================================
Two CONUS maps side by side, each classifying counties into concentration tiers
(pooled 2015-2025):
    Top 1%   (>=99th percentile)
    Top 1-5% (95-99)
    Top 5-10%(90-95)
    Top 10-20%(80-90)
    Bottom 80%(<80)

Left  : normalized customer-hour burden  (sum auc / customers served)
Right : number of outage events

Per-county values are recomputed from the raw table (the concentration_*.csv
files hold only period-level shares, not per-county values). Tier definitions
match those CSVs, and each tier's share of the national total is shown in the
legend.

Reads:  outage_event_table_2015_2025.csv, MCC.csv, shapefile
Writes: Results/Start_From_Begining/figures/fig_concentration_tier_maps.png
        Results/Start_From_Begining/concentration_tiers_by_county.csv
Requires: pandas, numpy, matplotlib, geopandas
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

OUTAGE_CSV = f"{Q1_3}/outage_event_table_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#d9d9d9"

TIERS = ["Top 1%", "Top 1-5%", "Top 5-10%", "Top 10-20%", "Bottom 80%"]
# purple ramp for burden, blue ramp for events (dark = most concentrated)
PURPLE = {"Top 1%": "#3f007d", "Top 1-5%": "#6a51a3", "Top 5-10%": "#9e9ac8",
          "Top 10-20%": "#cbc9e2", "Bottom 80%": "#f2f0f7"}
BLUE = {"Top 1%": "#084081", "Top 1-5%": "#2166ac", "Top 5-10%": "#4292c6",
        "Top 10-20%": "#9ecae1", "Bottom 80%": "#eef5fb"}


def load_mcc(path):
    m = pd.read_csv(path); m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    m = m[fn.notna() & m[cc].notna()]
    return dict(zip(fn[fn.notna()].astype("int64").astype(str).str.zfill(5), m[cc]))


def assign_tier(pct):
    return np.select(
        [pct >= 99, pct >= 95, pct >= 90, pct >= 80],
        ["Top 1%", "Top 1-5%", "Top 5-10%", "Top 10-20%"],
        default="Bottom 80%")


# ---- per-county metrics ----
ot = pd.read_csv(OUTAGE_CSV, dtype={"county_fips": str})
ot["county_fips"] = ot["county_fips"].str.zfill(5)
ot["auc"] = pd.to_numeric(ot["auc"], errors="coerce")
g = ot.groupby("county_fips").agg(abs_burden=("auc", "sum"),
                                  n_events=("county_fips", "size")).reset_index()
sv = g["county_fips"].map(load_mcc(MCC_CSV)).astype(float)
g["norm_burden"] = g["abs_burden"] / sv.where(sv > 0, np.nan)

# tiers (norm_burden only where defined; n_events for all)
gm = g.dropna(subset=["norm_burden"]).copy()
gm["norm_pct"] = gm["norm_burden"].rank(pct=True) * 100
gm["norm_tier"] = assign_tier(gm["norm_pct"])
g["events_pct"] = g["n_events"].rank(pct=True) * 100
g["events_tier"] = assign_tier(g["events_pct"])

out = g.merge(gm[["county_fips", "norm_pct", "norm_tier"]], on="county_fips", how="left")
out.to_csv(f"{OUT_DIR}/concentration_tiers_by_county.csv", index=False)


def tier_share(df, valcol, tiercol):
    """share (%) of total and n counties per tier."""
    tot = df[valcol].sum()
    res = {}
    for t in TIERS:
        sub = df[df[tiercol] == t]
        res[t] = (len(sub), sub[valcol].sum() / tot * 100 if tot else np.nan)
    return res


sh_norm = tier_share(gm, "norm_burden", "norm_tier")
sh_evt = tier_share(g, "n_events", "events_tier")
print("Normalized burden tiers:", {t: sh_norm[t] for t in TIERS})
print("Outage-event tiers:", {t: sh_evt[t] for t in TIERS})

# ---- maps ----
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")

fig, (axL, axR) = plt.subplots(1, 2, figsize=(22, 9))


def draw(ax, tiercol, colors, shares, title):
    geo = gdf.merge(out, on="county_fips", how="left")
    geo[geo[tiercol].isna()].plot(ax=ax, color=MISSING, linewidth=0.04,
                                  edgecolor="white")
    for t in TIERS[::-1]:          # light first, dark last (on top)
        sub = geo[geo[tiercol] == t]
        if len(sub):
            sub.plot(ax=ax, color=colors[t], linewidth=0.04, edgecolor="white")
    ax.set_axis_off()
    ax.set_title(title, fontsize=14, fontweight="bold", pad=6)
    handles = [Patch(color=colors[t],
               label=f"{t}  (n={shares[t][0]:,}, {shares[t][1]:.0f}% of total)")
               for t in TIERS]
    handles.append(Patch(color=MISSING, label="No data"))
    ax.legend(handles=handles, loc="lower left", fontsize=9, frameon=False)


draw(axL, "norm_tier", PURPLE, sh_norm,
     "A. Normalized customer-hour burden")
draw(axR, "events_tier", BLUE, sh_evt,
     "B. Number of outage events")

fig.suptitle("County concentration tiers, 2015-2025", fontsize=17,
             fontweight="bold", y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{FIG_DIR}/fig_concentration_tier_maps.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"\nWrote {FIG_DIR}/fig_concentration_tier_maps.png")
print(f"  {OUT_DIR}/concentration_tiers_by_county.csv")

Normalized burden tiers: {'Top 1%': (31, np.float64(80.16609500556575)), 'Top 1-5%': (121, np.float64(4.2240098861572415)), 'Top 5-10%': (152, np.float64(3.12719341397705)), 'Top 10-20%': (303, np.float64(3.9764645888915617)), 'Bottom 80%': (2426, np.float64(8.506237105408383))}
Outage-event tiers: {'Top 1%': (31, np.float64(3.9002796785915868)), 'Top 1-5%': (122, np.float64(9.629727107260862)), 'Top 5-10%': (153, np.float64(9.637654504981578)), 'Top 10-20%': (306, np.float64(15.917263335468446)), 'Bottom 80%': (2440, np.float64(60.915075373697526))}

Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_concentration_tier_maps.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/concentration_tiers_by_county.csv


In [113]:
"""
Concentration tier maps — normalized burden & number of outage events
=====================================================================
Two CONUS maps side by side, each classifying counties into concentration tiers
(pooled 2015-2025):
    Top 1%   (>=99th percentile)
    Top 1-5% (95-99)
    Top 5-10%(90-95)
    Top 10-20%(80-90)
    Bottom 80%(<80)

Left  : normalized customer-hour burden  (sum auc / customers served)
Right : number of outage events

Per-county values are recomputed from the raw table (the concentration_*.csv
files hold only period-level shares, not per-county values). Tier definitions
match those CSVs, and each tier's share of the national total is shown in the
legend.

Reads:  outage_event_table_2015_2025.csv, MCC.csv, shapefile
Writes: Results/Start_From_Begining/figures/fig_concentration_tier_maps.png
        Results/Start_From_Begining/concentration_tiers_by_county.csv
Requires: pandas, numpy, matplotlib, geopandas
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

OUTAGE_CSV = f"{Q1_3}/outage_event_table_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#d9d9d9"

TIERS = ["Top 1%", "Top 1-5%", "Top 5-10%", "Top 10-20%", "Bottom 80%"]
# purple ramp for burden, blue ramp for events (dark = most concentrated)
PURPLE = {"Top 1%": "#3f007d", "Top 1-5%": "#6a51a3", "Top 5-10%": "#9e9ac8",
          "Top 10-20%": "#cbc9e2", "Bottom 80%": "#f2f0f7"}
BLUE = {"Top 1%": "#084081", "Top 1-5%": "#2166ac", "Top 5-10%": "#4292c6",
        "Top 10-20%": "#9ecae1", "Bottom 80%": "#eef5fb"}


def load_mcc(path):
    m = pd.read_csv(path); m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    m = m[fn.notna() & m[cc].notna()]
    return dict(zip(fn[fn.notna()].astype("int64").astype(str).str.zfill(5), m[cc]))


def assign_tier(pct):
    return np.select(
        [pct >= 99, pct >= 95, pct >= 90, pct >= 80],
        ["Top 1%", "Top 1-5%", "Top 5-10%", "Top 10-20%"],
        default="Bottom 80%")


# ---- per-county metrics ----
ot = pd.read_csv(OUTAGE_CSV, dtype={"county_fips": str})
ot["county_fips"] = ot["county_fips"].str.zfill(5)
ot["auc"] = pd.to_numeric(ot["auc"], errors="coerce")
g = ot.groupby("county_fips").agg(abs_burden=("auc", "sum"),
                                  n_events=("county_fips", "size")).reset_index()
sv = g["county_fips"].map(load_mcc(MCC_CSV)).astype(float)
g["norm_burden"] = g["abs_burden"] / sv.where(sv > 0, np.nan)

# tiers (norm_burden only where defined; n_events for all)
gm = g.dropna(subset=["norm_burden"]).copy()
gm["norm_pct"] = gm["norm_burden"].rank(pct=True) * 100
gm["norm_tier"] = assign_tier(gm["norm_pct"])
g["events_pct"] = g["n_events"].rank(pct=True) * 100
g["events_tier"] = assign_tier(g["events_pct"])

out = g.merge(gm[["county_fips", "norm_pct", "norm_tier"]], on="county_fips", how="left")
out.to_csv(f"{OUT_DIR}/concentration_tiers_by_county.csv", index=False)


def tier_share(df, valcol, tiercol):
    """share (%) of total and n counties per tier."""
    tot = df[valcol].sum()
    res = {}
    for t in TIERS:
        sub = df[df[tiercol] == t]
        res[t] = (len(sub), sub[valcol].sum() / tot * 100 if tot else np.nan)
    return res


sh_norm = tier_share(gm, "norm_burden", "norm_tier")
sh_evt = tier_share(g, "n_events", "events_tier")
print("Normalized burden tiers:", {t: sh_norm[t] for t in TIERS})
print("Outage-event tiers:", {t: sh_evt[t] for t in TIERS})

# ---- maps ----
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")

fig, (axL, axR) = plt.subplots(1, 2, figsize=(22, 9))


SHORT = {"Top 1%": "Top 1%", "Top 1-5%": "1-5%", "Top 5-10%": "5-10%",
         "Top 10-20%": "10-20%"}
LEGEND_TIERS = ["Top 1%", "Top 1-5%", "Top 5-10%", "Top 10-20%"]


def draw(ax, tiercol, colors, shares, title):
    base = colors["Bottom 80%"]           # no-data uses the bottom-80 shade
    geo = gdf.merge(out, on="county_fips", how="left")
    geo[geo[tiercol].isna()].plot(ax=ax, color=base, linewidth=0.04,
                                  edgecolor="white")
    for t in TIERS[::-1]:                  # light first, dark last (on top)
        sub = geo[geo[tiercol] == t]
        if len(sub):
            sub.plot(ax=ax, color=colors[t], linewidth=0.04, edgecolor="white")
    ax.set_axis_off()
    ax.set_title(title, fontsize=14, fontweight="bold", pad=6)
    handles = [Patch(color=colors[t], label=SHORT[t]) for t in LEGEND_TIERS]
    ax.legend(handles=handles, loc="lower left", fontsize=10, frameon=False)


draw(axL, "norm_tier", PURPLE, sh_norm,
     "A. Normalized customer-hour burden")
draw(axR, "events_tier", BLUE, sh_evt,
     "B. Number of outage events")

fig.suptitle("County concentration tiers, 2015-2025", fontsize=17,
             fontweight="bold", y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{FIG_DIR}/fig_concentration_tier_maps.png", dpi=600, bbox_inches="tight")
plt.close(fig)
print(f"\nWrote {FIG_DIR}/fig_concentration_tier_maps.png")
print(f"  {OUT_DIR}/concentration_tiers_by_county.csv")

Normalized burden tiers: {'Top 1%': (31, np.float64(80.16609500556575)), 'Top 1-5%': (121, np.float64(4.2240098861572415)), 'Top 5-10%': (152, np.float64(3.12719341397705)), 'Top 10-20%': (303, np.float64(3.9764645888915617)), 'Bottom 80%': (2426, np.float64(8.506237105408383))}
Outage-event tiers: {'Top 1%': (31, np.float64(3.9002796785915868)), 'Top 1-5%': (122, np.float64(9.629727107260862)), 'Top 5-10%': (153, np.float64(9.637654504981578)), 'Top 10-20%': (306, np.float64(15.917263335468446)), 'Bottom 80%': (2440, np.float64(60.915075373697526))}

Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_concentration_tier_maps.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/concentration_tiers_by_county.csv


In [124]:
"""
Storm-Outage Disruption Index (SODI): four dimensions, LHS-weighted (top-50% target)
====================================================================
Dimensions (each an equal-average of its sub-elements, all as percentiles):
  Hazard   = mean( storm exposure count* , storm duration* )
  Severity = mean( normalized customer-hours* , P90 recovery* )
  Disproportionality = DR*  where DR = norm-burden pct - exposure pct
  Storm-Dependence   = storm-linkage share* (share of outages storm-linked at 12h)
(* = percentile 0-100)

SODI (observed disruption index) = R_lhs_mean, the mean weighted score across K LHS weightings on the
4-D simplex. (Baseline equal-weight risk is dropped: it is ~identical to the
LHS mean by construction.)

P_ATL = fraction of weightings in which a county is in the TOP TARGET_FRAC of
counties. With TARGET_FRAC = 0.50 this is "reliably above-median risk", which
fills the map instead of leaving most counties in a 'rare' bucket.

Six robustness tiers on P_ATL:
  >=0.90, 0.75-0.90, 0.50-0.75, 0.25-0.50, 0.10-0.25, <0.10

Outputs (Start_From_Begining) -- a CSV per stage:
  risk_elements_by_county.csv, correlation_subelements.csv,
  correlation_dimensions.csv, lhs_weight_scenarios.csv,
  risk_scores_by_county.csv, risk_class_counts.csv
Figures:
  fig_risk_correlation.png, fig_risk_dimension_maps.png,
  fig_risk_pvh_surface.png       (continuous P_ATL choropleth)
  fig_risk_pvh_tiers.png         (6-tier classified map)
  fig_risk_lhs_mean.png          (LHS-mean risk map)
Requires: pandas, numpy, matplotlib, geopandas, scipy
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, PowerNorm, BoundaryNorm
from matplotlib.patches import Patch
from scipy.stats.qmc import LatinHypercube
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

LINKED_CSV = f"{Q1_3}/storm_linked_outage_events_2015_2025.csv"
STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#e6e6e6"

BUFFER = 12
K = 10000
SEED = 42
TARGET_FRAC = 0.50          # "reliably above-median" target (was 0.10)
GAMMA = 2.5

# six robustness tiers on P_ATL
TIER_BINS = [-0.001, 0.10, 0.25, 0.50, 0.75, 0.90, 1.001]
TIER_LABELS = ["<0.10", "0.10-0.25", "0.25-0.50", "0.50-0.75", "0.75-0.90", ">=0.90"]
TIER_COLORS = ["#f7f7f7", "#fddbc7", "#f4a582", "#d6604d", "#b2182b", "#67001f"]


def load_mcc(path):
    m = pd.read_csv(path); m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    m = m[fn.notna() & m[cc].notna()]
    return dict(zip(fn[fn.notna()].astype("int64").astype(str).str.zfill(5), m[cc]))


def pctl(s):
    return s.rank(pct=True, method="average") * 100.0


def truthy(s):
    return s.astype(str).str.lower().isin(["true", "1", "1.0", "yes", "t"])


def p90(s):
    s = s.dropna()
    return float(np.percentile(s, 90)) if len(s) else np.nan


# ---- sub-elements per county ----
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
st["s_start"] = pd.to_datetime(st["start_time"], errors="coerce")
st["s_end"] = pd.to_datetime(st["end_time"], errors="coerce")
st["dur"] = ((st["s_end"] - st["s_start"]).dt.total_seconds() / 3600).clip(lower=0)
se = st.groupby("county_fips").agg(exposure=("event_type", "size"),
                                   duration=("dur", "sum")).reset_index()

o = pd.read_csv(LINKED_CSV, dtype={"county_fips": str})
o["county_fips"] = o["county_fips"].str.zfill(5)
o["auc"] = pd.to_numeric(o["auc"], errors="coerce")
o["recovery_hr"] = pd.to_numeric(o["recovery_hr"], errors="coerce")
o["linked"] = truthy(o[f"storm_linked_{BUFFER}h"])
link = o.groupby("county_fips")["linked"].mean().rename("linkage_share").reset_index()
ol = o[o["linked"]]
burden = ol.groupby("county_fips").agg(
    customer_hours=("auc", "sum"),
    p90_recovery=("recovery_hr", p90)).reset_index()

mcc = load_mcc(MCC_CSV)
d = se.merge(burden, on="county_fips", how="outer").merge(link, on="county_fips", how="outer")
sv = d["county_fips"].map(mcc).astype(float); sv = sv.where(sv > 0, np.nan)
d["norm_burden"] = d["customer_hours"] / sv
for c in ["exposure", "duration", "customer_hours", "p90_recovery",
          "linkage_share", "norm_burden"]:
    d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)

d["exposure_pct"] = pctl(d["exposure"])
d["duration_pct"] = pctl(d["duration"])
d["norm_burden_pct"] = pctl(d["norm_burden"])
d["p90_recovery_pct"] = pctl(d["p90_recovery"])
d["linkage_pct"] = pctl(d["linkage_share"])
d["DR"] = d["norm_burden_pct"] - d["exposure_pct"]
d["DR_pct"] = pctl(d["DR"])

d["Hazard"] = d[["exposure_pct", "duration_pct"]].mean(axis=1)
d["Severity"] = d[["norm_burden_pct", "p90_recovery_pct"]].mean(axis=1)
d["Disproportionality"] = d["DR_pct"]
d["Storm-Dependence"] = d["linkage_pct"]
DIMS = ["Hazard", "Severity", "Disproportionality", "Storm-Dependence"]
d.to_csv(f"{OUT_DIR}/risk_elements_by_county.csv", index=False)
print(f"counties: {len(d):,}")

# ---- correlation ----
SUBS = ["exposure_pct", "duration_pct", "norm_burden_pct", "p90_recovery_pct",
        "DR_pct", "linkage_pct"]
SUBLAB = ["Exposure", "Duration", "Norm burden", "P90 recovery",
          "DR (disproportionality)", "Storm-dependence"]
corr_sub = d[SUBS].corr(method="spearman")
corr_sub.to_csv(f"{OUT_DIR}/correlation_subelements.csv")
corr_dim = d[DIMS].corr(method="spearman")
corr_dim.to_csv(f"{OUT_DIR}/correlation_dimensions.csv")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(18, 8))
for ax, corr, labs, ttl in [(a1, corr_sub, SUBLAB, "Sub-elements"),
                            (a2, corr_dim, DIMS, "Dimensions")]:
    im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(labs))); ax.set_yticks(range(len(labs)))
    ax.set_xticklabels(labs, rotation=40, ha="right"); ax.set_yticklabels(labs)
    for i in range(len(labs)):
        for j in range(len(labs)):
            v = corr.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if abs(v) > 0.55 else "black")
    ax.set_title(f"Spearman correlation — {ttl}", fontsize=13, fontweight="bold")
    fig.colorbar(im, ax=ax, shrink=0.7)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_correlation.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# ---- LHS ----
X = d[DIMS].to_numpy()
u = LatinHypercube(d=4, seed=SEED).random(n=K)
e = -np.log(1.0 - u); W = e / e.sum(axis=1, keepdims=True)
pd.DataFrame(W, columns=[f"w_{c}" for c in DIMS]).assign(
    scenario=np.arange(1, K + 1)).to_csv(
    f"{OUT_DIR}/lhs_weight_scenarios.csv", index=False)
scores = W @ X.T
n = len(d)
order = np.argsort(-scores, axis=1)
ranks = np.empty_like(order); ranks[np.arange(K)[:, None], order] = np.arange(1, n + 1)
kv = max(1, int(np.ceil(TARGET_FRAC * n)))          # top-50% cutoff rank
d["P_ATL"] = (ranks <= kv).mean(axis=0)             # prob. above target line
d["R_lhs_mean"] = scores.mean(axis=0)
d["R_lhs_pct"] = pctl(d["R_lhs_mean"])
d["robustness_tier"] = pd.cut(d["P_ATL"], bins=TIER_BINS, labels=TIER_LABELS)

keep = ["county_fips"] + DIMS + ["R_lhs_mean", "R_lhs_pct", "P_ATL",
        "robustness_tier"]
d[keep].to_csv(f"{OUT_DIR}/risk_scores_by_county.csv", index=False)
cc = d["robustness_tier"].value_counts().reindex(TIER_LABELS).rename("n")
cc.to_frame().to_csv(f"{OUT_DIR}/risk_class_counts.csv")
print(f"target = top {int(TARGET_FRAC*100)}% (cutoff rank {kv})")
print("robustness tiers:\n", cc.to_string())

# ---- maps ----
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d, on="county_fips", how="left")
pn = PowerNorm(gamma=GAMMA, vmin=0, vmax=100)

# dimension maps
fig, axes = plt.subplots(2, 2, figsize=(18, 11))
for ax, dim in zip(axes.ravel(), DIMS):
    geo.plot(column=dim, cmap="magma_r", norm=pn, linewidth=0.03, edgecolor="white",
             ax=ax, legend=True, missing_kwds={"color": MISSING},
             legend_kwds={"shrink": 0.6, "label": f"{dim} (0-100)"})
    ax.set_axis_off(); ax.set_title(dim, fontsize=13, fontweight="bold")
fig.suptitle("Storm-Outage Disruption Index (SODI) dimensions", fontsize=16, fontweight="bold", y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{FIG_DIR}/fig_risk_dimension_maps.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# LHS-mean risk map (single)
fig, ax = plt.subplots(figsize=(15, 9))
geo.plot(column="R_lhs_pct", cmap="magma_r", norm=pn, linewidth=0.03,
         edgecolor="white", ax=ax, legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6, "label": "SODI percentile (LHS-mean)"})
ax.set_axis_off()
ax.set_title("Storm-Outage Disruption Index (SODI), observed 2015-2025",
             fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_lhs_mean.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# continuous P_ATL surface
fig, ax = plt.subplots(figsize=(15, 9))
geo.plot(column="P_ATL", cmap="magma_r", vmin=0, vmax=1, linewidth=0.03,
         edgecolor="white", ax=ax, legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6,
                      "label": f"P(top {int(TARGET_FRAC*100)}%) across weightings"})
ax.set_axis_off()
ax.set_title(f"SODI robustness surface: probability of top-{int(TARGET_FRAC*100)}%\n"
             "under 10,000 weightings", fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_pvh_surface.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# six-tier classified map
fig, ax = plt.subplots(figsize=(15, 9))
tier_cmap = ListedColormap(TIER_COLORS)
geo["_tier_code"] = pd.Categorical(geo["robustness_tier"],
                                   categories=TIER_LABELS).codes
geo.loc[geo["_tier_code"] < 0, "_tier_code"] = np.nan
geo[geo["_tier_code"].isna()].plot(ax=ax, color=MISSING, linewidth=0.03,
                                   edgecolor="white")
geo.dropna(subset=["_tier_code"]).plot(
    column="_tier_code", cmap=tier_cmap,
    norm=BoundaryNorm(np.arange(-0.5, len(TIER_LABELS) + 0.5), tier_cmap.N),
    linewidth=0.03, edgecolor="white", ax=ax)
ax.set_axis_off()
ax.set_title(f"SODI robustness tiers: P(top {int(TARGET_FRAC*100)}%) across weightings",
             fontsize=15, fontweight="bold")
handles = [Patch(color=TIER_COLORS[i],
           label=f"{TIER_LABELS[i]}  (n={int((d['robustness_tier']==TIER_LABELS[i]).sum())})")
           for i in range(len(TIER_LABELS))]
handles.append(Patch(color=MISSING, label="No score"))
ax.legend(handles=handles, loc="lower left", fontsize=9, frameon=False,
          title="P(above target)")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_pvh_tiers.png", dpi=250, bbox_inches="tight")
plt.close(fig)

print(f"\nWrote CSVs + figures to {OUT_DIR}")

counties: 3,204
target = top 50% (cutoff rank 1602)
robustness tiers:
 robustness_tier
<0.10        806
0.10-0.25    308
0.25-0.50    458
0.50-0.75    516
0.75-0.90    340
>=0.90       776


/local/tmp/ipykernel_4102714/2933721308.py:238: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  geo[geo["_tier_code"].isna()].plot(ax=ax, color=MISSING, linewidth=0.03,



Wrote CSVs + figures to /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining


In [121]:
"""
Weight-robustness of the Storm-Outage Disruption Index (SODI)
===================================================================
Layout (3x2):
  A: LHS-mean risk percentile map        B: robustness map P(top 50%)
  C: county rank-uncertainty histogram   D: dimension correlation (blue = high)
  E: dimension-heavy scenarios vs equal  F: robustness vs risk score

COLORS: map colormap is set once via MAP_CMAP. Swap it to taste, e.g.
  "YlOrRd" (light->dark red, current), "cividis", "viridis", "YlGnBu",
  "rocket"* (*needs seaborn). Correlation heatmap uses RdBu (blue = high +).

Reads:  risk_scores_by_county.csv, lhs_weight_scenarios.csv,
        correlation_dimensions.csv, shapefile
Outputs: risk_rank_uncertainty_by_county.csv, risk_named_scenarios.csv,
         risk_weight_robustness_summary.csv, fig_risk_weight_robustness.png
Requires: pandas, numpy, matplotlib, scipy, geopandas
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#e6e6e6"

DIMS = ["Hazard", "Severity", "Disproportionality", "Storm-Dependence"]
TARGET_FRAC = 0.50
TIER_LABELS = ["<0.10", "0.10-0.25", "0.25-0.50", "0.50-0.75", "0.75-0.90", ">=0.90"]

# ---- MAP COLORS (two distinct maps) ----
RISK_CMAP = "viridis"        # A: risk magnitude (standalone figure)
ROBUST_CMAP = "plasma"       # B: robustness probability (try: Purples, Blues, cividis)
CORR_CMAP = "RdBu"           # blue = high positive correlation

d = pd.read_csv(f"{OUT_DIR}/risk_scores_by_county.csv", dtype={"county_fips": str})
d["county_fips"] = d["county_fips"].str.zfill(5)
n = len(d); k = int(np.ceil(TARGET_FRAC * n))
X = d[DIMS].to_numpy()
W = pd.read_csv(f"{OUT_DIR}/lhs_weight_scenarios.csv")[[f"w_{c}" for c in DIMS]].to_numpy()
K = len(W)

# ---- ranks across weightings ----
scores = W @ X.T
order = np.argsort(-scores, axis=1)
ranks = np.empty_like(order); ranks[np.arange(K)[:, None], order] = np.arange(1, n + 1)
r5, r50, r95 = np.percentile(ranks, [5, 50, 95], axis=0)
d["rank_median"], d["rank_p5"], d["rank_p95"] = r50, r5, r95
d["rank_width"] = d["rank_p95"] - d["rank_p5"]
d[["county_fips", "rank_median", "rank_p5", "rank_p95", "rank_width",
   "R_lhs_pct", "P_ATL"]].sort_values("rank_median").to_csv(
    f"{OUT_DIR}/risk_rank_uncertainty_by_county.csv", index=False)
med_width_all = d["rank_width"].median()
med_width_top = d.loc[d["R_lhs_pct"] >= 100 * (1 - TARGET_FRAC), "rank_width"].median()

# ---- dimension correlation ----
corr = pd.read_csv(f"{OUT_DIR}/correlation_dimensions.csv", index_col=0).loc[DIMS, DIMS]
flag = [(DIMS[i], DIMS[j], round(corr.iloc[i, j], 3))
        for i in range(len(DIMS)) for j in range(i + 1, len(DIMS))
        if abs(corr.iloc[i, j]) >= 0.7]

# ---- dimension-heavy scenarios ----
SCEN = {"equal": [.25, .25, .25, .25]}
for i, dim in enumerate(DIMS):
    w = [.2, .2, .2, .2]; w[i] = .4; SCEN[f"{dim}-heavy"] = w
sc_scores = {name: X @ np.array(w) for name, w in SCEN.items()}
ref = sc_scores["equal"]
T_ref = set(d.iloc[np.argsort(-ref)[:k]]["county_fips"])
rows = []
for name, sc in sc_scores.items():
    r, _ = spearmanr(sc, ref, nan_policy="omit")
    T = set(d.iloc[np.argsort(-sc)[:k]]["county_fips"])
    rows.append({"scenario": name, "spearman_vs_equal": round(r, 4),
                 "top50_overlap_pct": round(len(T & T_ref) / k * 100, 1),
                 "n_entering": len(T - T_ref), "n_leaving": len(T_ref - T)})
scen = pd.DataFrame(rows)
scen.to_csv(f"{OUT_DIR}/risk_named_scenarios.csv", index=False)

# ---- summary ----
tiers = d["robustness_tier"].value_counts().reindex(TIER_LABELS).fillna(0).astype(int)
worst = scen.iloc[1:].sort_values("spearman_vs_equal").iloc[0]
srows = [{"metric": "Target tier", "value": f"top {int(TARGET_FRAC*100)}%"},
         {"metric": "Median rank uncertainty (all)", "value": f"{med_width_all:.0f}"},
         {"metric": "Median rank uncertainty (above-target)", "value": f"{med_width_top:.0f}"},
         {"metric": "Most disruptive dimension", "value": f"{worst['scenario']} ({worst['spearman_vs_equal']})"},
         {"metric": "Highly correlated pairs (|rho|>=0.7)",
          "value": "; ".join(f"{a}-{b}={v}" for a, b, v in flag) if flag else "none"}]
for t in TIER_LABELS:
    srows.append({"metric": f"P_ATL tier {t}", "value": str(int(tiers[t]))})
pd.DataFrame(srows).to_csv(f"{OUT_DIR}/risk_weight_robustness_summary.csv", index=False)

# ---- shapefile ----
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d, on="county_fips", how="left")

# ============================================================
# FIGURE
# ============================================================
# ---- standalone figure: A. LHS-mean risk map (viridis) ----
figA, axA = plt.subplots(figsize=(15, 9))
geo.plot(column="R_lhs_pct", cmap=RISK_CMAP, vmin=0, vmax=100, linewidth=0.025,
         edgecolor="white", ax=axA, legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6, "pad": 0.01, "label": "SODI percentile (LHS-mean)"})
axA.set_axis_off()
axA.set_title("LHS-Mean Storm-Outage Risk (percentile)", fontsize=15, fontweight="bold")
figA.tight_layout()
figA.savefig(f"{FIG_DIR}/fig_risk_lhs_mean.png", dpi=600, bbox_inches="tight")
plt.close(figA)

# ---- robustness figure: B full-width top row, then C/D and E/F ----
fig = plt.figure(figsize=(18, 20))
gs = fig.add_gridspec(3, 2, height_ratios=[1.25, 0.9, 0.9], hspace=0.22, wspace=0.20)

# B: robustness map, full width
ax = fig.add_subplot(gs[0, :])
geo.plot(column="P_ATL", cmap=ROBUST_CMAP, vmin=0, vmax=1, linewidth=0.025,
         edgecolor="white", ax=ax, legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.5, "pad": 0.01,
                      "label": f"P(top {int(TARGET_FRAC*100)}%) across weightings"})
ax.set_axis_off()
ax.set_title("A. SODI robustness: probability of top-50% across 10,000 weightings",
             fontsize=15, fontweight="bold", pad=6)

# C: rank uncertainty histogram
ax = fig.add_subplot(gs[1, 0])
ax.hist(d["rank_width"], bins=50, color="#4575b4", edgecolor="white", linewidth=0.3)
ax.axvline(med_width_all, color="black", lw=1.5, label=f"Median = {med_width_all:.0f}")
ax.set_xlabel("Rank uncertainty across weightings (p95 - p5 rank)")
ax.set_ylabel("Number of counties")
ax.set_title("B. County Rank Uncertainty Under Reweighting", fontsize=13, fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# D: dimension correlation heatmap (blue = high)
ax = fig.add_subplot(gs[1, 1])
im = ax.imshow(corr.values, cmap=CORR_CMAP, vmin=-1, vmax=1)
ax.set_xticks(range(len(DIMS))); ax.set_yticks(range(len(DIMS)))
ax.set_xticklabels(DIMS, rotation=30, ha="right", fontsize=10)
ax.set_yticklabels(DIMS, fontsize=10)
for i in range(len(DIMS)):
    for j in range(len(DIMS)):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=10,
                fontweight="bold", color="white" if abs(v) > 0.55 else "black")
ax.set_title("C. Correlation Among Storm-Outage Disruption Index Dimensions", fontsize=13, fontweight="bold")
fig.colorbar(im, ax=ax, shrink=0.78, pad=0.03).set_label("Spearman correlation", fontsize=10)

# E: dimension-heavy scenarios
ax = fig.add_subplot(gs[2, 0])
plot = scen[scen["scenario"] != "equal"].copy()
xx = np.arange(len(plot)); bw = 0.36
ax.bar(xx - bw/2, plot["spearman_vs_equal"], bw, color="#4575b4", label="Spearman vs equal")
axr = ax.twinx()
axr.bar(xx + bw/2, plot["top50_overlap_pct"], bw, color="#d73027", label="Top-50% overlap")
ax.set_xticks(xx)
ax.set_xticklabels(plot["scenario"].str.replace("-heavy", "", regex=False), fontsize=10)
ax.set_ylabel("Spearman correlation", color="#4575b4"); ax.set_ylim(0, 1.02)
axr.set_ylabel("Top-50% overlap (%)", color="#d73027"); axr.set_ylim(0, 102)
ax.set_title("D. Dimension-Heavy Scenarios vs. Equal Weighting", fontsize=13, fontweight="bold")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = axr.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="lower left"); ax.grid(alpha=0.25, axis="y")

# F: robustness vs risk
ax = fig.add_subplot(gs[2, 1])
ax.scatter(d["R_lhs_pct"], d["P_ATL"], s=12, alpha=0.30, color="#b3a2d4", edgecolors="none")
bins = np.arange(0, 101, 5)
d["_bin"] = pd.cut(d["R_lhs_pct"], bins=bins, include_lowest=True)
gm = d.groupby("_bin", observed=False)["P_ATL"].mean()
centers = np.array([(iv.left + iv.right) / 2 for iv in gm.index])
ax.plot(centers, gm.values, "-o", color="#54278f", lw=2.4, markersize=4,
        label="Mean P(top 50%) per 5-pct bin")
ax.axhline(0.90, color="#d73027", ls="--", lw=1.2, label="High robustness (0.90)")
ax.axhline(0.50, color="gray", ls=":", lw=1)
ax.set_xlim(0, 100); ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("Storm-Outage Disruption Index percentile")
ax.set_ylabel(f"P(top {int(TARGET_FRAC*100)}%) across weightings")
ax.set_title("E. Robustness vs. Storm-Outage Disruption Index score", fontsize=13, fontweight="bold")
ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3)

fig.suptitle("Weight Robustness of the Storm-Outage Disruption Index (SODI)",
             fontsize=18, fontweight="bold", y=0.995)
fig.subplots_adjust(top=0.95, bottom=0.05, left=0.055, right=0.95)
fig.savefig(f"{FIG_DIR}/fig_risk_weight_robustness.png", dpi=600, bbox_inches="tight")
plt.close(fig)
print(f"Wrote {FIG_DIR}/fig_risk_lhs_mean.png (risk, {RISK_CMAP})")
print(f"Wrote {FIG_DIR}/fig_risk_weight_robustness.png (robustness, {ROBUST_CMAP})")

Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_risk_lhs_mean.png (risk, viridis)
Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_risk_weight_robustness.png (robustness, plasma)


In [127]:
"""
Weight-robustness of the Storm-Outage Disruption Index (SODI)
===================================================================
Layout (3x2):
  A: LHS-mean risk percentile map        B: robustness map P(top 50%)
  C: county rank-uncertainty histogram   D: dimension correlation (blue = high)
  E: dimension-heavy scenarios vs equal  F: robustness vs risk score

COLORS: map colormap is set once via MAP_CMAP. Swap it to taste, e.g.
  "YlOrRd" (light->dark red, current), "cividis", "viridis", "YlGnBu",
  "rocket"* (*needs seaborn). Correlation heatmap uses RdBu (blue = high +).

Reads:  risk_scores_by_county.csv, lhs_weight_scenarios.csv,
        correlation_dimensions.csv, shapefile
Outputs: risk_rank_uncertainty_by_county.csv, risk_named_scenarios.csv,
         risk_weight_robustness_summary.csv, fig_risk_weight_robustness.png
Requires: pandas, numpy, matplotlib, scipy, geopandas
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#e6e6e6"

DIMS = ["Hazard", "Severity", "Disproportionality", "Storm-Dependence"]
TARGET_FRAC = 0.10
TIER_LABELS = ["<0.10", "0.10-0.25", "0.25-0.50", "0.50-0.75", "0.75-0.90", ">=0.90"]

# ---- MAP COLORS (two distinct maps) ----
RISK_CMAP = "viridis"        # A: risk magnitude (standalone figure)
ROBUST_CMAP = "plasma"       # B: robustness probability (try: Purples, Blues, cividis)
CORR_CMAP = "RdBu"           # blue = high positive correlation

d = pd.read_csv(f"{OUT_DIR}/risk_scores_by_county.csv", dtype={"county_fips": str})
d["county_fips"] = d["county_fips"].str.zfill(5)
n = len(d); k = int(np.ceil(TARGET_FRAC * n))
X = d[DIMS].to_numpy()
W = pd.read_csv(f"{OUT_DIR}/lhs_weight_scenarios.csv")[[f"w_{c}" for c in DIMS]].to_numpy()
K = len(W)

# ---- ranks across weightings ----
scores = W @ X.T
order = np.argsort(-scores, axis=1)
ranks = np.empty_like(order); ranks[np.arange(K)[:, None], order] = np.arange(1, n + 1)
r5, r50, r95 = np.percentile(ranks, [5, 50, 95], axis=0)
d["rank_median"], d["rank_p5"], d["rank_p95"] = r50, r5, r95
d["rank_width"] = d["rank_p95"] - d["rank_p5"]
d[["county_fips", "rank_median", "rank_p5", "rank_p95", "rank_width",
   "R_lhs_pct", "P_ATL"]].sort_values("rank_median").to_csv(
    f"{OUT_DIR}/risk_rank_uncertainty_by_county.csv", index=False)
med_width_all = d["rank_width"].median()
med_width_top = d.loc[d["R_lhs_pct"] >= 100 * (1 - TARGET_FRAC), "rank_width"].median()

# ---- dimension correlation ----
corr = pd.read_csv(f"{OUT_DIR}/correlation_dimensions.csv", index_col=0).loc[DIMS, DIMS]
flag = [(DIMS[i], DIMS[j], round(corr.iloc[i, j], 3))
        for i in range(len(DIMS)) for j in range(i + 1, len(DIMS))
        if abs(corr.iloc[i, j]) >= 0.7]

# ---- dimension-heavy scenarios ----
SCEN = {"equal": [.25, .25, .25, .25]}
for i, dim in enumerate(DIMS):
    w = [.2, .2, .2, .2]; w[i] = .4; SCEN[f"{dim}-heavy"] = w
sc_scores = {name: X @ np.array(w) for name, w in SCEN.items()}
ref = sc_scores["equal"]
T_ref = set(d.iloc[np.argsort(-ref)[:k]]["county_fips"])
rows = []
for name, sc in sc_scores.items():
    r, _ = spearmanr(sc, ref, nan_policy="omit")
    T = set(d.iloc[np.argsort(-sc)[:k]]["county_fips"])
    rows.append({"scenario": name, "spearman_vs_equal": round(r, 4),
                 "top50_overlap_pct": round(len(T & T_ref) / k * 100, 1),
                 "n_entering": len(T - T_ref), "n_leaving": len(T_ref - T)})
scen = pd.DataFrame(rows)
scen.to_csv(f"{OUT_DIR}/risk_named_scenarios.csv", index=False)

# ---- summary ----
tiers = d["robustness_tier"].value_counts().reindex(TIER_LABELS).fillna(0).astype(int)
worst = scen.iloc[1:].sort_values("spearman_vs_equal").iloc[0]
srows = [{"metric": "Target tier", "value": f"top {int(TARGET_FRAC*100)}%"},
         {"metric": "Median rank uncertainty (all)", "value": f"{med_width_all:.0f}"},
         {"metric": "Median rank uncertainty (above-target)", "value": f"{med_width_top:.0f}"},
         {"metric": "Most disruptive dimension", "value": f"{worst['scenario']} ({worst['spearman_vs_equal']})"},
         {"metric": "Highly correlated pairs (|rho|>=0.7)",
          "value": "; ".join(f"{a}-{b}={v}" for a, b, v in flag) if flag else "none"}]
for t in TIER_LABELS:
    srows.append({"metric": f"P_ATL tier {t}", "value": str(int(tiers[t]))})
pd.DataFrame(srows).to_csv(f"{OUT_DIR}/risk_weight_robustness_summary.csv", index=False)

# ---- shapefile ----
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d, on="county_fips", how="left")

# ============================================================
# FIGURE
# ============================================================
# ---- standalone figure: A. LHS-mean risk map (viridis) ----
figA, axA = plt.subplots(figsize=(15, 9))
geo.plot(column="R_lhs_pct", cmap=RISK_CMAP, vmin=0, vmax=100, linewidth=0.025,
         edgecolor="white", ax=axA, legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6, "pad": 0.01, "label": "Storm-Outage Disruption Index percentile"})
axA.set_axis_off()
axA.set_title("LHS-Mean Storm-Outage Risk (percentile)", fontsize=15, fontweight="bold")
figA.tight_layout()
figA.savefig(f"{FIG_DIR}/fig_risk_lhs_mean.png", dpi=600, bbox_inches="tight")
plt.close(figA)

# ---- robustness figure: B full-width top row, then C/D and E/F ----
fig = plt.figure(figsize=(18, 20))
gs = fig.add_gridspec(3, 2, height_ratios=[1.25, 0.9, 0.9], hspace=0.22, wspace=0.20)

# B: robustness map, full width
ax = fig.add_subplot(gs[0, :])
geo.plot(column="P_ATL", cmap=ROBUST_CMAP, vmin=0, vmax=1, linewidth=0.025,
         edgecolor="white", ax=ax, legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.5, "pad": 0.01,
                      "label": f"P(top {int(TARGET_FRAC*100)}%) across weightings"})
ax.set_axis_off()
ax.set_title("A. Storm-Outage Disruption Index robustness: probability of top-50% across 10,000 weightings",
             fontsize=15, fontweight="bold", pad=6)

# C: rank uncertainty histogram
ax = fig.add_subplot(gs[1, 0])
ax.hist(d["rank_width"], bins=50, color="#4575b4", edgecolor="white", linewidth=0.3)
ax.axvline(med_width_all, color="black", lw=1.5, label=f"Median = {med_width_all:.0f}")
ax.set_xlabel("Rank uncertainty across weightings (p95 - p5 rank)")
ax.set_ylabel("Number of counties")
ax.set_title("B. County Rank Uncertainty Under Reweighting", fontsize=13, fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# D: dimension correlation heatmap (blue = high)
ax = fig.add_subplot(gs[1, 1])
im = ax.imshow(corr.values, cmap=CORR_CMAP, vmin=-1, vmax=1)
ax.set_xticks(range(len(DIMS))); ax.set_yticks(range(len(DIMS)))
ax.set_xticklabels(DIMS, rotation=30, ha="right", fontsize=10)
ax.set_yticklabels(DIMS, fontsize=10)
for i in range(len(DIMS)):
    for j in range(len(DIMS)):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=10,
                fontweight="bold", color="white" if abs(v) > 0.55 else "black")
ax.set_title("C. Correlation Among Storm-Outage Disruption Index Dimensions", fontsize=13, fontweight="bold")
fig.colorbar(im, ax=ax, shrink=0.78, pad=0.03).set_label("Spearman correlation", fontsize=10)

# E: dimension-heavy scenarios
ax = fig.add_subplot(gs[2, 0])
plot = scen[scen["scenario"] != "equal"].copy()
xx = np.arange(len(plot)); bw = 0.36
ax.bar(xx - bw/2, plot["spearman_vs_equal"], bw, color="#4575b4", label="Spearman vs equal")
axr = ax.twinx()
axr.bar(xx + bw/2, plot["top50_overlap_pct"], bw, color="#d73027", label="Top-50% overlap")
ax.set_xticks(xx)
ax.set_xticklabels(plot["scenario"].str.replace("-heavy", "", regex=False), fontsize=10)
ax.set_ylabel("Spearman correlation", color="#4575b4"); ax.set_ylim(0, 1.02)
axr.set_ylabel("Top-50% overlap (%)", color="#d73027"); axr.set_ylim(0, 102)
ax.set_title("D. Dimension-Heavy Scenarios vs. Equal Weighting", fontsize=13, fontweight="bold")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = axr.get_legend_handles_labels()
leg = ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="upper left",
                framealpha=0.95, facecolor="white", edgecolor="#888")
leg.set_zorder(10)
ax.grid(alpha=0.25, axis="y")

# F: robustness vs risk
ax = fig.add_subplot(gs[2, 1])
ax.scatter(d["R_lhs_pct"], d["P_ATL"], s=12, alpha=0.30, color="#b3a2d4", edgecolors="none")
bins = np.arange(0, 101, 5)
d["_bin"] = pd.cut(d["R_lhs_pct"], bins=bins, include_lowest=True)
gm = d.groupby("_bin", observed=False)["P_ATL"].mean()
centers = np.array([(iv.left + iv.right) / 2 for iv in gm.index])
ax.plot(centers, gm.values, "-o", color="#54278f", lw=2.4, markersize=4,
        label="Mean P(top 50%) per 5-pct bin")
ax.axhline(0.90, color="#d73027", ls="--", lw=1.2, label="High robustness (0.90)")
ax.axhline(0.50, color="gray", ls=":", lw=1)
ax.set_xlim(0, 100); ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("Storm-Outage Disruption Index percentile (LHS-mean)")
ax.set_ylabel(f"P(top {int(TARGET_FRAC*100)}%) across weightings")
ax.set_title("E. Robustness vs. Storm-Outage Disruption Index score", fontsize=13, fontweight="bold")
ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3)

fig.suptitle("Weight Robustness of the Storm-Outage Disruption Index (Storm-Outage Disruption Index)",
             fontsize=18, fontweight="bold", y=0.995)
fig.subplots_adjust(top=0.95, bottom=0.05, left=0.055, right=0.95)
fig.savefig(f"{FIG_DIR}/fig_risk_weight_robustness.png", dpi=600, bbox_inches="tight")
plt.close(fig)
print(f"Wrote {FIG_DIR}/fig_risk_lhs_mean.png (risk, {RISK_CMAP})")
print(f"Wrote {FIG_DIR}/fig_risk_weight_robustness.png (robustness, {ROBUST_CMAP})")

Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_risk_lhs_mean.png (risk, viridis)
Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_risk_weight_robustness.png (robustness, plasma)


In [128]:
"""
Storm-Outage Disruption Index (SODI): four dimensions, LHS-weighted (top-50% target)
====================================================================
Dimensions (each an equal-average of its sub-elements, all as percentiles):
  Hazard   = mean( storm exposure count* , storm duration* )
  Severity = mean( normalized customer-hours* , P90 recovery* )
  Disproportionality = DR*  where DR = norm-burden pct - exposure pct
  Storm-Dependence   = storm-linkage share* (share of outages storm-linked at 12h)
(* = percentile 0-100)

SODI (observed disruption index) = R_lhs_mean, the mean weighted score across K LHS weightings on the
4-D simplex. (Baseline equal-weight risk is dropped: it is ~identical to the
LHS mean by construction.)

P_ATL = fraction of weightings in which a county is in the TOP TARGET_FRAC of
counties. With TARGET_FRAC = 0.50 this is "reliably above-median risk", which
fills the map instead of leaving most counties in a 'rare' bucket.

Six robustness tiers on P_ATL:
  >=0.90, 0.75-0.90, 0.50-0.75, 0.25-0.50, 0.10-0.25, <0.10

Outputs (Start_From_Begining) -- a CSV per stage:
  risk_elements_by_county.csv, correlation_subelements.csv,
  correlation_dimensions.csv, lhs_weight_scenarios.csv,
  risk_scores_by_county.csv, risk_class_counts.csv
Figures:
  fig_risk_correlation.png, fig_risk_dimension_maps.png,
  fig_risk_pvh_surface.png       (continuous P_ATL choropleth)
  fig_risk_pvh_tiers.png         (6-tier classified map)
  fig_risk_lhs_mean.png          (LHS-mean risk map)
Requires: pandas, numpy, matplotlib, geopandas, scipy
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, PowerNorm, BoundaryNorm
from matplotlib.patches import Patch
from scipy.stats.qmc import LatinHypercube
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

LINKED_CSV = f"{Q1_3}/storm_linked_outage_events_2015_2025.csv"
STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#e6e6e6"

BUFFER = 12
K = 10000
SEED = 42
TARGET_FRAC = 0.10          # top-10% target
GAMMA = 2.5

# six robustness tiers on P_ATL
TIER_BINS = [-0.001, 0.10, 0.25, 0.50, 0.75, 0.90, 1.001]
TIER_LABELS = ["<0.10", "0.10-0.25", "0.25-0.50", "0.50-0.75", "0.75-0.90", ">=0.90"]
TIER_COLORS = ["#f7f7f7", "#fddbc7", "#f4a582", "#d6604d", "#b2182b", "#67001f"]


def load_mcc(path):
    m = pd.read_csv(path); m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    m = m[fn.notna() & m[cc].notna()]
    return dict(zip(fn[fn.notna()].astype("int64").astype(str).str.zfill(5), m[cc]))


def pctl(s):
    return s.rank(pct=True, method="average") * 100.0


def truthy(s):
    return s.astype(str).str.lower().isin(["true", "1", "1.0", "yes", "t"])


def p90(s):
    s = s.dropna()
    return float(np.percentile(s, 90)) if len(s) else np.nan


# ---- sub-elements per county ----
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
st["s_start"] = pd.to_datetime(st["start_time"], errors="coerce")
st["s_end"] = pd.to_datetime(st["end_time"], errors="coerce")
st["dur"] = ((st["s_end"] - st["s_start"]).dt.total_seconds() / 3600).clip(lower=0)
se = st.groupby("county_fips").agg(exposure=("event_type", "size"),
                                   duration=("dur", "sum")).reset_index()

o = pd.read_csv(LINKED_CSV, dtype={"county_fips": str})
o["county_fips"] = o["county_fips"].str.zfill(5)
o["auc"] = pd.to_numeric(o["auc"], errors="coerce")
o["recovery_hr"] = pd.to_numeric(o["recovery_hr"], errors="coerce")
o["linked"] = truthy(o[f"storm_linked_{BUFFER}h"])
link = o.groupby("county_fips")["linked"].mean().rename("linkage_share").reset_index()
ol = o[o["linked"]]
burden = ol.groupby("county_fips").agg(
    customer_hours=("auc", "sum"),
    p90_recovery=("recovery_hr", p90)).reset_index()

mcc = load_mcc(MCC_CSV)
d = se.merge(burden, on="county_fips", how="outer").merge(link, on="county_fips", how="outer")
sv = d["county_fips"].map(mcc).astype(float); sv = sv.where(sv > 0, np.nan)
d["norm_burden"] = d["customer_hours"] / sv
for c in ["exposure", "duration", "customer_hours", "p90_recovery",
          "linkage_share", "norm_burden"]:
    d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)

d["exposure_pct"] = pctl(d["exposure"])
d["duration_pct"] = pctl(d["duration"])
d["norm_burden_pct"] = pctl(d["norm_burden"])
d["p90_recovery_pct"] = pctl(d["p90_recovery"])
d["linkage_pct"] = pctl(d["linkage_share"])
d["DR"] = d["norm_burden_pct"] - d["exposure_pct"]
d["DR_pct"] = pctl(d["DR"])

d["Hazard"] = d[["exposure_pct", "duration_pct"]].mean(axis=1)
d["Severity"] = d[["norm_burden_pct", "p90_recovery_pct"]].mean(axis=1)
d["Disproportionality"] = d["DR_pct"]
d["Storm-Dependence"] = d["linkage_pct"]
DIMS = ["Hazard", "Severity", "Disproportionality", "Storm-Dependence"]
d.to_csv(f"{OUT_DIR}/risk_elements_by_county.csv", index=False)
print(f"counties: {len(d):,}")

# ---- correlation ----
SUBS = ["exposure_pct", "duration_pct", "norm_burden_pct", "p90_recovery_pct",
        "DR_pct", "linkage_pct"]
SUBLAB = ["Exposure", "Duration", "Norm burden", "P90 recovery",
          "DR (disproportionality)", "Storm-dependence"]
corr_sub = d[SUBS].corr(method="spearman")
corr_sub.to_csv(f"{OUT_DIR}/correlation_subelements.csv")
corr_dim = d[DIMS].corr(method="spearman")
corr_dim.to_csv(f"{OUT_DIR}/correlation_dimensions.csv")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(18, 8))
for ax, corr, labs, ttl in [(a1, corr_sub, SUBLAB, "Sub-elements"),
                            (a2, corr_dim, DIMS, "Dimensions")]:
    im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(labs))); ax.set_yticks(range(len(labs)))
    ax.set_xticklabels(labs, rotation=40, ha="right"); ax.set_yticklabels(labs)
    for i in range(len(labs)):
        for j in range(len(labs)):
            v = corr.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if abs(v) > 0.55 else "black")
    ax.set_title(f"Spearman correlation — {ttl}", fontsize=13, fontweight="bold")
    fig.colorbar(im, ax=ax, shrink=0.7)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_correlation.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# ---- LHS ----
X = d[DIMS].to_numpy()
u = LatinHypercube(d=4, seed=SEED).random(n=K)
e = -np.log(1.0 - u); W = e / e.sum(axis=1, keepdims=True)
pd.DataFrame(W, columns=[f"w_{c}" for c in DIMS]).assign(
    scenario=np.arange(1, K + 1)).to_csv(
    f"{OUT_DIR}/lhs_weight_scenarios.csv", index=False)
scores = W @ X.T
n = len(d)
order = np.argsort(-scores, axis=1)
ranks = np.empty_like(order); ranks[np.arange(K)[:, None], order] = np.arange(1, n + 1)
kv = max(1, int(np.ceil(TARGET_FRAC * n)))          # top-50% cutoff rank
d["P_ATL"] = (ranks <= kv).mean(axis=0)             # prob. above target line
d["R_lhs_mean"] = scores.mean(axis=0)
d["R_lhs_pct"] = pctl(d["R_lhs_mean"])
d["robustness_tier"] = pd.cut(d["P_ATL"], bins=TIER_BINS, labels=TIER_LABELS)

keep = ["county_fips"] + DIMS + ["R_lhs_mean", "R_lhs_pct", "P_ATL",
        "robustness_tier"]
d[keep].to_csv(f"{OUT_DIR}/risk_scores_by_county.csv", index=False)
cc = d["robustness_tier"].value_counts().reindex(TIER_LABELS).rename("n")
cc.to_frame().to_csv(f"{OUT_DIR}/risk_class_counts.csv")
print(f"target = top {int(TARGET_FRAC*100)}% (cutoff rank {kv})")
print("robustness tiers:\n", cc.to_string())

# ---- maps ----
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d, on="county_fips", how="left")
pn = PowerNorm(gamma=GAMMA, vmin=0, vmax=100)

# dimension maps
fig, axes = plt.subplots(2, 2, figsize=(18, 11))
for ax, dim in zip(axes.ravel(), DIMS):
    geo.plot(column=dim, cmap="magma_r", norm=pn, linewidth=0.03, edgecolor="white",
             ax=ax, legend=True, missing_kwds={"color": MISSING},
             legend_kwds={"shrink": 0.6, "label": f"{dim} (0-100)"})
    ax.set_axis_off(); ax.set_title(dim, fontsize=13, fontweight="bold")
fig.suptitle("Storm-Outage Disruption Index (SODI) dimensions", fontsize=16, fontweight="bold", y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{FIG_DIR}/fig_risk_dimension_maps.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# LHS-mean risk map (single)
fig, ax = plt.subplots(figsize=(15, 9))
geo.plot(column="R_lhs_pct", cmap="magma_r", norm=pn, linewidth=0.03,
         edgecolor="white", ax=ax, legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6, "label": "SODI percentile (LHS-mean)"})
ax.set_axis_off()
ax.set_title("Storm-Outage Disruption Index (SODI), observed 2015-2025",
             fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_lhs_mean.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# continuous P_ATL surface
fig, ax = plt.subplots(figsize=(15, 9))
geo.plot(column="P_ATL", cmap="magma_r", vmin=0, vmax=1, linewidth=0.03,
         edgecolor="white", ax=ax, legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6,
                      "label": f"P(top {int(TARGET_FRAC*100)}%) across weightings"})
ax.set_axis_off()
ax.set_title(f"SODI robustness surface: probability of top-{int(TARGET_FRAC*100)}%\n"
             "under 10,000 weightings", fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_pvh_surface.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# six-tier classified map
fig, ax = plt.subplots(figsize=(15, 9))
tier_cmap = ListedColormap(TIER_COLORS)
geo["_tier_code"] = pd.Categorical(geo["robustness_tier"],
                                   categories=TIER_LABELS).codes
geo.loc[geo["_tier_code"] < 0, "_tier_code"] = np.nan
geo[geo["_tier_code"].isna()].plot(ax=ax, color=MISSING, linewidth=0.03,
                                   edgecolor="white")
geo.dropna(subset=["_tier_code"]).plot(
    column="_tier_code", cmap=tier_cmap,
    norm=BoundaryNorm(np.arange(-0.5, len(TIER_LABELS) + 0.5), tier_cmap.N),
    linewidth=0.03, edgecolor="white", ax=ax)
ax.set_axis_off()
ax.set_title(f"SODI robustness tiers: P(top {int(TARGET_FRAC*100)}%) across weightings",
             fontsize=15, fontweight="bold")
handles = [Patch(color=TIER_COLORS[i],
           label=f"{TIER_LABELS[i]}  (n={int((d['robustness_tier']==TIER_LABELS[i]).sum())})")
           for i in range(len(TIER_LABELS))]
handles.append(Patch(color=MISSING, label="No score"))
ax.legend(handles=handles, loc="lower left", fontsize=9, frameon=False,
          title="P(above target)")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_pvh_tiers.png", dpi=600, bbox_inches="tight")
plt.close(fig)

print(f"\nWrote CSVs + figures to {OUT_DIR}")

counties: 3,204
target = top 10% (cutoff rank 321)
robustness tiers:
 robustness_tier
<0.10        2393
0.10-0.25     300
0.25-0.50     285
0.50-0.75     171
0.75-0.90      45
>=0.90         10


/local/tmp/ipykernel_4102714/1120345541.py:238: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  geo[geo["_tier_code"].isna()].plot(ax=ax, color=MISSING, linewidth=0.03,



Wrote CSVs + figures to /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining


In [131]:
"""
Create revised SODI robustness figure
=====================================

Panel A: existing fig_risk_pvh_tiers.png
Panel B: county rank uncertainty
Panel C: dimension correlation
Panel D: dimension-heavy scenarios
Panel E: robustness vs SODI

Output:
    fig_risk_weight_robustness.png
"""

import os
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
from scipy.stats import spearmanr


# ============================================================
# PATHS
# ============================================================

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"

os.makedirs(FIG_DIR, exist_ok=True)

TIER_FIG = f"{FIG_DIR}/fig_risk_pvh_tiers.png"
OUTPUT_FIG = f"{FIG_DIR}/fig_risk_weight_robustness.png"


# ============================================================
# SETTINGS
# ============================================================

DIMS = [
    "Hazard",
    "Severity",
    "Disproportionality",
    "Storm-Dependence"
]

TARGET_FRAC = 0.10
CORR_CMAP = "RdBu"


# ============================================================
# LOAD DATA
# ============================================================

d = pd.read_csv(
    f"{OUT_DIR}/risk_scores_by_county.csv",
    dtype={"county_fips": str}
)

d["county_fips"] = d["county_fips"].str.zfill(5)

W = pd.read_csv(
    f"{OUT_DIR}/lhs_weight_scenarios.csv"
)[[f"w_{c}" for c in DIMS]].to_numpy()

corr = pd.read_csv(
    f"{OUT_DIR}/correlation_dimensions.csv",
    index_col=0
).loc[DIMS, DIMS]


# ============================================================
# RECREATE RANK UNCERTAINTY
# This fixes the KeyError: 'rank_width'
# ============================================================

X = d[DIMS].to_numpy()

n = len(d)
K = len(W)

scores = W @ X.T

order = np.argsort(-scores, axis=1)

ranks = np.empty_like(order)

ranks[
    np.arange(K)[:, None],
    order
] = np.arange(1, n + 1)

r5, r50, r95 = np.percentile(
    ranks,
    [5, 50, 95],
    axis=0
)

d["rank_median"] = r50
d["rank_p5"] = r5
d["rank_p95"] = r95

d["rank_width"] = (
    d["rank_p95"] -
    d["rank_p5"]
)

med_width_all = d["rank_width"].median()

med_width_top = d.loc[
    d["R_lhs_pct"] >= 100 * (1 - TARGET_FRAC),
    "rank_width"
].median()


# ============================================================
# DIMENSION-HEAVY SCENARIOS
# ============================================================

k = int(np.ceil(TARGET_FRAC * n))

SCEN = {
    "equal": [0.25, 0.25, 0.25, 0.25]
}

for i, dim in enumerate(DIMS):

    w = [0.2, 0.2, 0.2, 0.2]

    w[i] = 0.4

    SCEN[f"{dim}-heavy"] = w


sc_scores = {
    name: X @ np.array(w)
    for name, w in SCEN.items()
}

ref = sc_scores["equal"]

T_ref = set(
    d.iloc[
        np.argsort(-ref)[:k]
    ]["county_fips"]
)

rows = []

for name, sc in sc_scores.items():

    r, _ = spearmanr(
        sc,
        ref,
        nan_policy="omit"
    )

    T = set(
        d.iloc[
            np.argsort(-sc)[:k]
        ]["county_fips"]
    )

    rows.append({
        "scenario": name,
        "spearman_vs_equal": r,
        "top50_overlap_pct": len(T & T_ref) / k * 100
    })

scen = pd.DataFrame(rows)


# ============================================================
# CREATE FIGURE
# ============================================================

fig = plt.figure(
    figsize=(18, 20)
)

gs = fig.add_gridspec(
    3,
    2,
    height_ratios=[1.25, 0.9, 0.9],
    hspace=0.22,
    wspace=0.20
)


# ============================================================
# PANEL A
# EXISTING SIX-TIER ROBUSTNESS FIGURE
# ============================================================

ax = fig.add_subplot(
    gs[0, :]
)

tier_img = plt.imread(
    TIER_FIG
)

ax.imshow(
    tier_img
)

ax.set_axis_off()


# Add "A." separately
ax.text(
    0.01,
    0.98,
    "A.",
    transform=ax.transAxes,
    fontsize=16,
    fontweight="bold",
    ha="left",
    va="top"
)


# ============================================================
# PANEL B
# RANK UNCERTAINTY
# ============================================================

ax = fig.add_subplot(
    gs[1, 0]
)

ax.hist(
    d["rank_width"],
    bins=50,
    color="#4575b4",
    edgecolor="white",
    linewidth=0.3
)

ax.axvline(
    med_width_all,
    color="black",
    lw=1.5,
    label=f"Median = {med_width_all:.0f}"
)

ax.set_xlabel(
    "Rank uncertainty across weightings (p95 - p5 rank)"
)

ax.set_ylabel(
    "Number of counties"
)

ax.set_title(
    "B. County Rank Uncertainty Under Reweighting",
    fontsize=13,
    fontweight="bold"
)

ax.legend(
    fontsize=9
)

ax.grid(
    alpha=0.3
)


# ============================================================
# PANEL C
# DIMENSION CORRELATION
# ============================================================

ax = fig.add_subplot(
    gs[1, 1]
)

im = ax.imshow(
    corr.values,
    cmap=CORR_CMAP,
    vmin=-1,
    vmax=1
)

ax.set_xticks(
    range(len(DIMS))
)

ax.set_yticks(
    range(len(DIMS))
)

ax.set_xticklabels(
    DIMS,
    rotation=30,
    ha="right",
    fontsize=10
)

ax.set_yticklabels(
    DIMS,
    fontsize=10
)

for i in range(len(DIMS)):

    for j in range(len(DIMS)):

        v = corr.iloc[i, j]

        ax.text(
            j,
            i,
            f"{v:.2f}",
            ha="center",
            va="center",
            fontsize=10,
            fontweight="bold",
            color="white" if abs(v) > 0.55 else "black"
        )


ax.set_title(
    "C. Correlation Among Storm-Outage Disruption Index Dimensions",
    fontsize=13,
    fontweight="bold"
)

cbar = fig.colorbar(
    im,
    ax=ax,
    shrink=0.78,
    pad=0.03
)

cbar.set_label(
    "Spearman correlation",
    fontsize=10
)


# ============================================================
# PANEL D
# DIMENSION-HEAVY SCENARIOS
# ============================================================

ax = fig.add_subplot(
    gs[2, 0]
)

plot = scen[
    scen["scenario"] != "equal"
].copy()

xx = np.arange(
    len(plot)
)

bw = 0.36


ax.bar(
    xx - bw / 2,
    plot["spearman_vs_equal"],
    bw,
    color="#4575b4",
    label="Spearman vs equal"
)

axr = ax.twinx()

axr.bar(
    xx + bw / 2,
    plot["top50_overlap_pct"],
    bw,
    color="#d73027",
    label=f"Top-{int(TARGET_FRAC*100)}% overlap"
)


ax.set_xticks(
    xx
)

ax.set_xticklabels(
    plot["scenario"].str.replace(
        "-heavy",
        "",
        regex=False
    ),
    fontsize=10
)

ax.set_ylabel(
    "Spearman correlation",
    color="#4575b4"
)

ax.set_ylim(
    0,
    1.02
)

axr.set_ylabel(
    f"Top-{int(TARGET_FRAC*100)}% overlap (%)",
    color="#d73027"
)

axr.set_ylim(
    0,
    102
)


ax.set_title(
    "D. Dimension-Heavy Scenarios vs. Equal Weighting",
    fontsize=13,
    fontweight="bold"
)


h1, l1 = ax.get_legend_handles_labels()
h2, l2 = axr.get_legend_handles_labels()

leg = ax.legend(
    h1 + h2,
    l1 + l2,
    fontsize=8,
    loc="upper left",
    framealpha=0.95,
    facecolor="white",
    edgecolor="#888"
)

leg.set_zorder(10)

ax.grid(
    alpha=0.25,
    axis="y"
)


# ============================================================
# PANEL E
# ROBUSTNESS VS SODI
# ============================================================

ax = fig.add_subplot(
    gs[2, 1]
)

ax.scatter(
    d["R_lhs_pct"],
    d["P_ATL"],
    s=12,
    alpha=0.30,
    color="#b3a2d4",
    edgecolors="none"
)


bins = np.arange(
    0,
    101,
    5
)

d["_bin"] = pd.cut(
    d["R_lhs_pct"],
    bins=bins,
    include_lowest=True
)

gm = d.groupby(
    "_bin",
    observed=False
)["P_ATL"].mean()


centers = np.array([
    (iv.left + iv.right) / 2
    for iv in gm.index
])


ax.plot(
    centers,
    gm.values,
    "-o",
    color="#54278f",
    lw=2.4,
    markersize=4,
    label=f"Mean P(top {int(TARGET_FRAC*100)}%) per 5-pct bin"
)


ax.axhline(
    0.90,
    color="#d73027",
    ls="--",
    lw=1.2,
    label="High robustness (0.90)"
)

ax.axhline(
    0.50,
    color="gray",
    ls=":",
    lw=1
)


ax.set_xlim(
    0,
    100
)

ax.set_ylim(
    -0.02,
    1.02
)


ax.set_xlabel(
    "Storm-Outage Disruption Index percentile (LHS-mean)"
)

ax.set_ylabel(
    f"P(top {int(TARGET_FRAC*100)}%) across weightings"
)


ax.set_title(
    "E. Robustness vs. Storm-Outage Disruption Index Score",
    fontsize=13,
    fontweight="bold"
)


ax.legend(
    fontsize=8,
    loc="upper left"
)

ax.grid(
    alpha=0.3
)


# ============================================================
# FINAL LAYOUT
# ============================================================

fig.suptitle(
    "Weight Robustness of the Storm-Outage Disruption Index (SODI)",
    fontsize=18,
    fontweight="bold",
    y=0.995
)

fig.subplots_adjust(
    top=0.95,
    bottom=0.05,
    left=0.055,
    right=0.95
)


fig.savefig(
    OUTPUT_FIG,
    dpi=600,
    bbox_inches="tight"
)

plt.close(fig)


print(
    f"Saved: {OUTPUT_FIG}"
)

Saved: /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_risk_weight_robustness.png


In [3]:
"""
Figure 2 — County-level relationships between storm exposure and power outage
characteristics across CONUS, 2015-2025.

  (a) Storm-exposure percentile vs normalized customer-hour-burden percentile,
      four exposure-burden groups split at the median (50th percentile),
      dashed diagonal = equal exposure and burden ranks.
  (b) Bivariate map: storm frequency vs outage-event frequency (quartiles).
  (c) Bivariate map: cumulative storm duration vs outage duration (quartiles).

Panel (a) reproduces the calculation of the existing quadrant-scatter cell
(including the MCC cap for counties whose P90 peak exceeds MCC), so the
quadrant counts should match the numbers reported in the Results.

Reads : county_level_storm_records_2015_2025.csv, outage_event_table_2015_2025.csv,
        MCC.csv, cb_2022_us_county_5m.shp
Writes: Results/Start_From_Begining/figures/fig2_exposure_outage_relationships.png
        Results/Start_From_Begining/fig2_county_values.csv
Requires: pandas, numpy, matplotlib, geopandas
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
import geopandas as gpd

# ----------------------------------------------------------------------------
# Paths and settings
# ----------------------------------------------------------------------------
BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
OUTAGE_CSV = f"{Q1_3}/outage_event_table_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
os.makedirs(FIG_DIR, exist_ok=True)

NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING_COLOR = "#ececec"
SPLIT = 50.0
DPI = 600

# Overall figure title (set SHOW_TITLE = False if the caption is enough)
SHOW_TITLE = True
FIG_TITLE = ("County-level relationships between storm exposure and power outage "
             "characteristics across CONUS, 2015\u20132025")

# Outage duration for panel (c):
#   "duration" = sum of event durations (end - start, or a duration column)
#   "recovery" = sum of recovery_hr (what the earlier bivariate cell used)
DURATION_MODE = "duration"

# Quadrant colours (same as the existing scatter)
COL = {"HH": "#8c4a63", "HL": "#6f9bd1", "LH": "#e8695f", "LL": "#d9d9d9"}
LEG = {"HH": "High exposure – high burden", "HL": "High exposure – low burden",
       "LH": "Low exposure – high burden", "LL": "Low exposure – low burden"}

# Bivariate palette (same corners as the existing maps)
C00, Cx0 = to_rgb("#e8e8e8"), to_rgb("#5ac8c8")
C0y, Cxy = to_rgb("#be64ac"), to_rgb("#3b4994")


def bivariate_color(ix, iy, n=4):
    fx, fy = ix / (n - 1), iy / (n - 1)
    bottom = [(1 - fx) * C00[k] + fx * Cx0[k] for k in range(3)]
    top = [(1 - fx) * C0y[k] + fx * Cxy[k] for k in range(3)]
    return tuple((1 - fy) * bottom[k] + fy * top[k] for k in range(3))


PALETTE = {(ix, iy): bivariate_color(ix, iy) for ix in range(4) for iy in range(4)}


# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------
def load_mcc(path):
    m = pd.read_csv(path, encoding="utf-8-sig")
    m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    ok = fn.notna() & m[cc].notna()
    return dict(zip(fn[ok].astype("int64").astype(str).str.zfill(5), m.loc[ok, cc]))


def pctl(s):
    return s.fillna(0).rank(pct=True, method="average") * 100.0


def p90(s):
    s = s.dropna()
    return float(np.percentile(s, 90)) if len(s) else np.nan


def quartile(series):
    r = series.rank(method="average", pct=True)
    q = np.ceil(r * 4).clip(1, 4) - 1
    return q.where(series.notna(), np.nan)


def find_col(df, names):
    lower = {c.lower(): c for c in df.columns}
    return next((lower[n] for n in names if n in lower), None)


def event_duration_hours(ot):
    """Event duration from a duration column, or end - start."""
    dcol = find_col(ot, ["duration_hr", "duration_h", "duration_hours", "duration"])
    if dcol:
        return pd.to_numeric(ot[dcol], errors="coerce"), f"column '{dcol}'"
    scol = find_col(ot, ["start_time", "event_start", "outage_start", "start", "begin_time"])
    ecol = find_col(ot, ["end_time", "event_end", "outage_end", "end"])
    if scol and ecol:
        dur = (pd.to_datetime(ot[ecol], errors="coerce")
               - pd.to_datetime(ot[scol], errors="coerce")).dt.total_seconds() / 3600
        return dur, f"'{ecol}' - '{scol}'"
    return None, None


# ----------------------------------------------------------------------------
# County metrics
# ----------------------------------------------------------------------------
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
st["dur_hr"] = ((pd.to_datetime(st["end_time"], errors="coerce")
                 - pd.to_datetime(st["start_time"], errors="coerce"))
                .dt.total_seconds() / 3600).clip(lower=0)
storm = st.groupby("county_fips").agg(n_storms=("event_type", "size"),
                                      storm_hours=("dur_hr", "sum")).reset_index()

ot = pd.read_csv(OUTAGE_CSV, dtype={"county_fips": str})
ot["county_fips"] = ot["county_fips"].str.zfill(5)
for c in ["auc", "peak_customers_out", "recovery_hr"]:
    ot[c] = pd.to_numeric(ot[c], errors="coerce")

if DURATION_MODE == "duration":
    ot["dur_used"], source = event_duration_hours(ot)
    if source is None:
        print("WARNING: no duration or start/end columns found; using recovery_hr.")
        ot["dur_used"], source = ot["recovery_hr"], "recovery_hr (fallback)"
else:
    ot["dur_used"], source = ot["recovery_hr"], "recovery_hr"
print(f"Outage duration for panel (c) taken from: {source}")

outage = ot.groupby("county_fips").agg(
    n_outage_events=("county_fips", "size"),
    customer_hours=("auc", "sum"),
    p90_peak=("peak_customers_out", p90),
    outage_hours=("dur_used", "sum")).reset_index()

df = storm.merge(outage, on="county_fips", how="outer")
df = df[~df["county_fips"].str[:2].isin(NON_CONUS)].reset_index(drop=True)

# ----------------------------------------------------------------------------
# Panel (a): exposure vs normalized burden (same logic as existing scatter)
# ----------------------------------------------------------------------------
mcc = load_mcc(MCC_CSV)
served = df["county_fips"].map(mcc).astype(float)
served = served.where(served > 0, np.nan)
q = df[served.notna()].copy()
sv = served[q.index]
q["n_storms"] = q["n_storms"].fillna(0)
flag = (sv < q["p90_peak"]).fillna(False)
norm = (q["customer_hours"] / sv).replace([np.inf, -np.inf], np.nan)
norm = norm.where(~flag, norm[~flag].max())
q["exposure_pct"] = pctl(q["n_storms"])
q["burden_pct"] = pctl(norm)
q.loc[flag.values, "burden_pct"] = 100.0
hx, hy = q["exposure_pct"] > SPLIT, q["burden_pct"] > SPLIT
q["quad"] = np.where(hx & hy, "HH", np.where(hx & ~hy, "HL",
                     np.where(~hx & hy, "LH", "LL")))
print(f"Counties in panel (a): {len(q):,} (MCC-capped: {int(flag.sum()):,})")
print(q["quad"].value_counts().reindex(["HH", "HL", "LH", "LL"]).to_string())

# ----------------------------------------------------------------------------
# Panels (b) and (c): bivariate quartile classes
# ----------------------------------------------------------------------------
def bivar(d, xcol, ycol):
    qx, qy = quartile(d[xcol]), quartile(d[ycol])
    ok = qx.notna() & qy.notna()
    cls = pd.Series(np.nan, index=d.index, dtype=object)
    cls[ok] = qx[ok].astype(int).astype(str) + qy[ok].astype(int).astype(str)
    return cls


df["class_b"] = bivar(df, "n_storms", "n_outage_events")
df["class_c"] = bivar(df, "storm_hours", "outage_hours")

gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(df[["county_fips", "class_b", "class_c"]], on="county_fips", how="left")


# ----------------------------------------------------------------------------
# Drawing
# ----------------------------------------------------------------------------
def draw_bivariate(ax, col, xlabel, ylabel):
    for ix in range(4):
        for iy in range(4):
            sub = geo[geo[col] == f"{ix}{iy}"]
            if len(sub):
                sub.plot(ax=ax, color=PALETTE[(ix, iy)], linewidth=0.05, edgecolor="white")
    miss = geo[geo[col].isna()]
    if len(miss):
        miss.plot(ax=ax, color=MISSING_COLOR, linewidth=0.05, edgecolor="white")
    ax.set_axis_off()

    lax = ax.inset_axes([0.88, 0.06, 0.15, 0.30])
    for ix in range(4):
        for iy in range(4):
            lax.add_patch(plt.Rectangle((ix, iy), 1, 1, facecolor=PALETTE[(ix, iy)],
                                        edgecolor="white", linewidth=0.4))
    lax.set_xlim(0, 4); lax.set_ylim(0, 4); lax.set_aspect("equal")
    ticks = [0.5, 1.5, 2.5, 3.5]
    lax.set_xticks(ticks); lax.set_yticks(ticks)
    lax.set_xticklabels(["Q1", "Q2", "Q3", "Q4"], fontsize=7)
    lax.set_yticklabels(["Q1", "Q2", "Q3", "Q4"], fontsize=7)
    lax.set_xlabel(f"{xlabel} \u2192", fontsize=8, fontweight="bold", labelpad=2)
    lax.set_ylabel(f"{ylabel} \u2192", fontsize=8, fontweight="bold", labelpad=2)
    lax.tick_params(length=0, pad=1)
    for sp in lax.spines.values():
        sp.set_linewidth(0.6); sp.set_edgecolor("#666666")


def panel_title(ax, text, pad=8):
    ax.set_title(text, fontsize=14, fontweight="bold", loc="center", pad=pad)


# Label for panel (c) follows the duration source actually used
y_label_c = "Recovery time" if "recovery" in source else "Outage duration"
title_c = ("(c) Storm duration vs. outage recovery time" if "recovery" in source
           else "(c) Storm duration vs. outage duration")

plt.rcParams.update({"font.family": "DejaVu Sans", "figure.facecolor": "white"})
fig = plt.figure(figsize=(20, 15.5))
# Top row: scatter centred between two empty columns; bottom row: two maps
gs = fig.add_gridspec(2, 4, height_ratios=[1.35, 1], width_ratios=[1, 1, 1, 1],
                      hspace=0.08, wspace=0.02, left=0.02, right=0.98,
                      top=0.93 if SHOW_TITLE else 0.97, bottom=0.02)

# (a) scatter, top centre
axA = fig.add_subplot(gs[0, 1:3])
for k in ["LL", "HL", "LH", "HH"]:
    s = q[q["quad"] == k]
    axA.scatter(s["exposure_pct"], s["burden_pct"], s=12, alpha=0.75, color=COL[k],
                edgecolors="none", label=f"{LEG[k]} (n = {len(s):,})")
axA.axvline(SPLIT, color="gray", lw=0.8, alpha=0.7)
axA.axhline(SPLIT, color="gray", lw=0.8, alpha=0.7)
axA.plot([0, 100], [0, 100], "k--", lw=1, alpha=0.6, label="Equal exposure and burden ranks")
axA.set_xlim(-1, 101); axA.set_ylim(-1, 101); axA.set_aspect("equal")
axA.set_xlabel("Storm exposure percentile", fontsize=12)
axA.set_ylabel("Normalized customer-hour burden percentile", fontsize=12)
axA.tick_params(labelsize=10)
axA.legend(loc="lower right", fontsize=9, framealpha=0.95)
panel_title(axA, "(a) Storm exposure vs. normalized outage burden")
# Make panel (a) slightly smaller while keeping it centered
pos = axA.get_position()
scale = 0.88   # 0.90 = slightly smaller, 0.85 = noticeably smaller

new_w = pos.width * scale
new_h = pos.height * scale

axA.set_position([
    pos.x0 + (pos.width - new_w) / 2,
    pos.y0 + (pos.height - new_h) / 2,
    new_w,
    new_h
])
# (b) and (c) maps, side by side
axB = fig.add_subplot(gs[1, 0:2])
draw_bivariate(axB, "class_b", "Storm frequency", "Outage frequency")
panel_title(axB, "(b) Storm frequency vs. outage frequency", pad=4)

axC = fig.add_subplot(gs[1, 2:4])
draw_bivariate(axC, "class_c", "Storm duration", y_label_c)
panel_title(axC, title_c, pad=4)

if SHOW_TITLE:
    fig.suptitle(FIG_TITLE, fontsize=18, fontweight="bold", y=0.985)

out_png = f"{FIG_DIR}/fig2_exposure_outage_relationships.png"
fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
plt.close(fig)

# ----------------------------------------------------------------------------
# County values behind the figure
# ----------------------------------------------------------------------------
out = df.merge(q[["county_fips", "exposure_pct", "burden_pct", "quad"]],
               on="county_fips", how="left")
out.to_csv(f"{OUT_DIR}/fig2_county_values.csv", index=False)
print(f"\nWrote:\n  {out_png}\n  {OUT_DIR}/fig2_county_values.csv")

Outage duration for panel (c) taken from: 'event_end' - 'event_start'
Counties in panel (a): 3,098 (MCC-capped: 64)
quad
HH    721
HL    827
LH    828
LL    722

Wrote:
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig2_exposure_outage_relationships.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/fig2_county_values.csv


In [6]:
"""
Single-panel disproportionality map, 2015-2025 (CONUS counties).

PANEL = "count"  -> Supplementary Figure 1:
                    outage-event frequency percentile - storm-exposure percentile

PANEL = "burden" -> Figure 5:
                    normalized customer-hour burden percentile - storm-exposure percentile
                    (same MCC cap and percentile rules as Figure 2a)

Storm exposure = number of storm records per county. Percentiles treat
missing values as 0, as in Figure 2a, so colours match the Figure 2 groups.

Reads : county_level_storm_records_2015_2025.csv,
        outage_event_table_2015_2025.csv,
        MCC.csv (burden only),
        cb_2022_us_county_5m.shp

Writes: Results/Start_From_Begining/figures/<output name>.png
        Results/Start_From_Begining/<output name>_values.csv

Requires: pandas, numpy, matplotlib, geopandas
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import geopandas as gpd


# ----------------------------------------------------------------------------
# Settings
# ----------------------------------------------------------------------------

PANEL = "count"       # "count" or "burden"
LIMIT = 80            # colour scale runs from -LIMIT to +LIMIT
DPI = 600

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"

STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
OUTAGE_CSV = f"{Q1_3}/outage_event_table_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"

os.makedirs(FIG_DIR, exist_ok=True)

NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING_COLOR = "#e6e6e6"


SETUP = {
    "count": dict(
        out="supp_fig1_outage_frequency_disproportionality",
        cbar="Outage frequency percentile − storm exposure percentile",
        title=(
            "Disproportionality between outage occurrence and storm exposure "
        )
    ),

    "burden": dict(
        out="fig5_burden_disproportionality",
        cbar="Normalized burden percentile − storm exposure percentile",
        title=(
            "Disproportionality between normalized outage burden and storm exposure "
        )
    ),
}[PANEL]


# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------

def pctl(s):
    return s.fillna(0).rank(pct=True, method="average") * 100.0


def p90(s):
    s = s.dropna()
    return float(np.percentile(s, 90)) if len(s) else np.nan


def load_mcc(path):
    m = pd.read_csv(path, encoding="utf-8-sig")
    m.columns = [c.strip() for c in m.columns]

    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]

    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")

    ok = fn.notna() & m[cc].notna()

    return dict(
        zip(
            fn[ok].astype("int64").astype(str).str.zfill(5),
            m.loc[ok, cc]
        )
    )


# ----------------------------------------------------------------------------
# County values
# ----------------------------------------------------------------------------

# Storm exposure
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)

expo = (
    st.groupby("county_fips")
      .size()
      .rename("n_storms")
      .reset_index()
)


# Outage metrics
ot = pd.read_csv(OUTAGE_CSV, dtype={"county_fips": str})
ot["county_fips"] = ot["county_fips"].str.zfill(5)

for c in ["auc", "peak_customers_out"]:
    ot[c] = pd.to_numeric(ot[c], errors="coerce")


outs = (
    ot.groupby("county_fips")
      .agg(
          n_outage_events=("county_fips", "size"),
          customer_hours=("auc", "sum"),
          p90_peak=("peak_customers_out", p90)
      )
      .reset_index()
)


# Merge storm and outage metrics
d = expo.merge(outs, on="county_fips", how="outer")

# Remove non-CONUS areas
d = d[
    ~d["county_fips"].str[:2].isin(NON_CONUS)
].reset_index(drop=True)


# ----------------------------------------------------------------------------
# Calculate disproportionality
# ----------------------------------------------------------------------------

if PANEL == "count":

    d["exposure_pct"] = pctl(d["n_storms"])
    d["impact_pct"] = pctl(d["n_outage_events"])


else:

    # Same normalization / MCC-cap rules as Figure 2a
    mcc = load_mcc(MCC_CSV)

    served = d["county_fips"].map(mcc).astype(float)
    served = served.where(served > 0, np.nan)

    # Keep only counties with valid MCC
    d = d[served.notna()].copy()
    sv = served[d.index]

    # Flag counties where P90 outage peak exceeds MCC
    flag = (sv < d["p90_peak"]).fillna(False)

    # Normalized customer-hour burden
    norm_burden = (
        d["customer_hours"] / sv
    ).replace([np.inf, -np.inf], np.nan)

    # Apply MCC cap
    norm_burden = norm_burden.where(
        ~flag,
        norm_burden[~flag].max()
    )

    d["exposure_pct"] = pctl(d["n_storms"])
    d["impact_pct"] = pctl(norm_burden)

    # Capped counties receive maximum burden percentile
    d.loc[flag.values, "impact_pct"] = 100.0

    print(f"MCC-capped counties: {int(flag.sum()):,}")


# Disproportionality:
# positive = outage impact percentile > storm exposure percentile
# negative = storm exposure percentile > outage impact percentile
d["gap"] = d["impact_pct"] - d["exposure_pct"]


print(f"Counties: {len(d):,}")
print(
    f"Gap > +20: {(d['gap'] > 20).sum():,}   "
    f"Gap < -20: {(d['gap'] < -20).sum():,}"
)


# ----------------------------------------------------------------------------
# Geographic data
# ----------------------------------------------------------------------------

gdf = gpd.read_file(SHP)

gdf["county_fips"] = (
    gdf["GEOID"]
    .astype(str)
    .str.zfill(5)
)

gdf = gdf[
    ~gdf["county_fips"].str[:2].isin(NON_CONUS)
]

# CONUS projection
gdf = gdf.to_crs("EPSG:5070")

geo = gdf.merge(
    d[["county_fips", "gap"]],
    on="county_fips",
    how="left"
)


# ----------------------------------------------------------------------------
# Plot
# ----------------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.facecolor": "white"
})


fig, ax = plt.subplots(figsize=(13, 7.5))


# Figure title
fig.suptitle(
    SETUP["title"],
    fontsize=15,
    fontweight="bold",
    y=0.96
)


# Diverging colour scale centered at zero
norm = TwoSlopeNorm(
    vmin=-LIMIT,
    vcenter=0,
    vmax=LIMIT
)


geo.plot(
    column="gap",
    cmap="RdBu_r",
    norm=norm,
    linewidth=0.05,
    edgecolor="white",
    ax=ax,
    missing_kwds={
        "color": MISSING_COLOR,
        "edgecolor": "white",
        "linewidth": 0.05
    }
)


ax.set_axis_off()


# ----------------------------------------------------------------------------
# Colorbar
# ----------------------------------------------------------------------------

sm = plt.cm.ScalarMappable(
    cmap="RdBu_r",
    norm=norm
)

sm.set_array([])


cb = fig.colorbar(
    sm,
    ax=ax,
    orientation="horizontal",
    fraction=0.045,
    pad=0.02,
    shrink=0.55,
    extend="both"
)

cb.set_label(
    SETUP["cbar"],
    fontsize=11
)

cb.ax.tick_params(
    labelsize=9
)


# Leave enough space for title
fig.subplots_adjust(top=0.90)


# ----------------------------------------------------------------------------
# Save outputs
# ----------------------------------------------------------------------------

out_png = f"{FIG_DIR}/{SETUP['out']}.png"

fig.savefig(
    out_png,
    dpi=DPI,
    bbox_inches="tight"
)

plt.close(fig)


out_csv = f"{OUT_DIR}/{SETUP['out']}_values.csv"

d.to_csv(
    out_csv,
    index=False
)


print(
    f"\nWrote:\n"
    f"  {out_png}\n"
    f"  {out_csv}"
)

Counties: 3,204
Gap > +20: 877   Gap < -20: 859

Wrote:
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/supp_fig1_outage_frequency_disproportionality.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/supp_fig1_outage_frequency_disproportionality_values.csv


In [8]:
"""
Figure 9 — County-level Storm-Outage Disruption Score (SODS), CONUS, 2015-2025.

Re-draws the LHS-mean score map (previously "SODI", fig_risk_lhs_mean.png) with
SODS labels. No values are recomputed: the script reads the county scores already
written by the index cell (risk_scores_by_county.csv, column R_lhs_pct), so the map
is identical to the earlier version apart from the labels.

Reads : Results/Start_From_Begining/risk_scores_by_county.csv
        RQ2/shapefiles/cb_2022_us_county_5m.shp
Writes: Results/Start_From_Begining/figures/fig9_sods_map.png
Requires: pandas, matplotlib, geopandas
"""
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
SCORES_CSV = f"{OUT_DIR}/risk_scores_by_county.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
os.makedirs(FIG_DIR, exist_ok=True)

NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#e6e6e6"
GAMMA = 2.5          # same colour stretch as the earlier map
DPI = 600

SHOW_TITLE = True   # the caption describes the figure; set True for a title
TITLE = "Storm-Outage Disruption Score (SODS), 2015\u20132025"

# ---- scores ----
d = pd.read_csv(SCORES_CSV, dtype={"county_fips": str})
d["county_fips"] = d["county_fips"].str.zfill(5)
if "R_lhs_pct" not in d.columns:
    raise ValueError(f"'R_lhs_pct' not found in {SCORES_CSV}; columns: {list(d.columns)}")
d = d.rename(columns={"R_lhs_pct": "SODS_pct", "R_lhs_mean": "SODS_mean"})
print(f"Counties with a SODS score: {d['SODS_pct'].notna().sum():,}")

# ---- map ----
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d[["county_fips", "SODS_pct"]], on="county_fips", how="left")

plt.rcParams.update({"font.family": "DejaVu Sans", "figure.facecolor": "white"})
fig, ax = plt.subplots(figsize=(15, 9))
norm = PowerNorm(gamma=GAMMA, vmin=0, vmax=100)
geo.plot(column="SODS_pct", cmap="magma_r", norm=norm, linewidth=0.03,
         edgecolor="white", ax=ax, missing_kwds={"color": MISSING})
ax.set_axis_off()

sm = plt.cm.ScalarMappable(cmap="magma_r", norm=norm)
cb = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.01)
cb.set_label("SODS percentile", fontsize=12)
cb.set_ticks([0, 50, 70, 80, 90, 100])   # spaced for the gamma stretch
cb.ax.tick_params(labelsize=10)

if SHOW_TITLE:
    ax.set_title(TITLE, fontsize=15, fontweight="bold")

out_png = f"{FIG_DIR}/fig9_sods_map.png"
fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
plt.close(fig)

d.to_csv(f"{OUT_DIR}/sods_scores_by_county.csv", index=False)
print(f"Wrote:\n  {out_png}\n  {OUT_DIR}/sods_scores_by_county.csv")

Counties with a SODS score: 3,204
Wrote:
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig9_sods_map.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/sods_scores_by_county.csv


In [9]:
"""
Figure 10 — Weight robustness of the Storm-Outage Disruption Score (SODS), 2015-2025.

  (a) County robustness tiers: probability of ranking in the top TARGET_FRAC
  (b) County rank uncertainty (P95 - P5 rank across weightings)
  (c) Spearman correlations among the four SODS dimensions
  (d) Dimension-heavy scenarios vs. equal weighting (Spearman and top-k overlap)
  (e) Robustness vs. SODS percentile

Same calculations as the earlier robustness cell, with SODS labels. Panel (a) is
now drawn directly from the county scores instead of pasting the old tier image,
so it no longer carries the old "SODI" title.

Reads : Results/Start_From_Begining/risk_scores_by_county.csv
        Results/Start_From_Begining/lhs_weight_scenarios.csv
        Results/Start_From_Begining/correlation_dimensions.csv
        RQ2/shapefiles/cb_2022_us_county_5m.shp
Writes: Results/Start_From_Begining/figures/fig10_sods_robustness.png
        Results/Start_From_Begining/sods_dimension_heavy_scenarios.csv
Requires: pandas, numpy, matplotlib, scipy, geopandas
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from scipy.stats import spearmanr
import geopandas as gpd

# ----------------------------------------------------------------------------
# Settings
# ----------------------------------------------------------------------------
BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
os.makedirs(FIG_DIR, exist_ok=True)

DIMS = ["Hazard", "Severity", "Disproportionality", "Storm-Dependence"]
TARGET_FRAC = 0.10        # must match the value used when P_ATL was computed
DPI = 600
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#e6e6e6"

TIER_BINS = [-0.001, 0.10, 0.25, 0.50, 0.75, 0.90, 1.001]
TIER_LABELS = ["<0.10", "0.10\u20130.25", "0.25\u20130.50", "0.50\u20130.75",
               "0.75\u20130.90", "\u22650.90"]
TIER_COLORS = ["#f7f7f7", "#fddbc7", "#f4a582", "#d6604d", "#b2182b", "#67001f"]
PCT = int(TARGET_FRAC * 100)

# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
d = pd.read_csv(f"{OUT_DIR}/risk_scores_by_county.csv", dtype={"county_fips": str})
d["county_fips"] = d["county_fips"].str.zfill(5)
W = pd.read_csv(f"{OUT_DIR}/lhs_weight_scenarios.csv")[[f"w_{c}" for c in DIMS]].to_numpy()
corr = pd.read_csv(f"{OUT_DIR}/correlation_dimensions.csv", index_col=0).loc[DIMS, DIMS]

# Rank uncertainty across the LHS weightings
X = d[DIMS].to_numpy()
n, K = len(d), len(W)
scores = W @ X.T
order = np.argsort(-scores, axis=1)
ranks = np.empty_like(order)
ranks[np.arange(K)[:, None], order] = np.arange(1, n + 1)
r5, r95 = np.percentile(ranks, [5, 95], axis=0)
d["rank_width"] = r95 - r5
med_width = d["rank_width"].median()

# Robustness tiers from P_ATL
d["tier"] = pd.cut(d["P_ATL"], bins=TIER_BINS, labels=TIER_LABELS)

# Dimension-heavy scenarios vs equal weighting
k = int(np.ceil(TARGET_FRAC * n))
scen_w = {"Equal": [0.25] * 4}
for i, dim in enumerate(DIMS):
    w = [0.2] * 4
    w[i] = 0.4
    scen_w[dim] = w
sc = {name: X @ np.array(w) for name, w in scen_w.items()}
top_ref = set(np.argsort(-sc["Equal"])[:k])
rows = []
for name, s in sc.items():
    r, _ = spearmanr(s, sc["Equal"], nan_policy="omit")
    rows.append({"scenario": name, "spearman_vs_equal": r,
                 f"top{PCT}_overlap_pct": 100 * len(set(np.argsort(-s)[:k]) & top_ref) / k})
scen = pd.DataFrame(rows)
scen.to_csv(f"{OUT_DIR}/sods_dimension_heavy_scenarios.csv", index=False)

print(f"Counties: {n:,}   weightings: {K:,}   top-{PCT}% cutoff: {k}")
print(f"Median rank uncertainty (P95-P5): {med_width:.0f}")
print(d["tier"].value_counts().reindex(TIER_LABELS).to_string())
print(scen.round(3).to_string(index=False))

# ----------------------------------------------------------------------------
# Figure
# ----------------------------------------------------------------------------
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.facecolor": "white"})
fig = plt.figure(figsize=(18, 20))
gs = fig.add_gridspec(3, 2, height_ratios=[1.25, 0.9, 0.9], hspace=0.25, wspace=0.25,
                      left=0.06, right=0.95, top=0.97, bottom=0.05)


def title(ax, text):
    ax.set_title(text, fontsize=13, fontweight="bold", loc="left")


# (a) robustness tiers map
ax = fig.add_subplot(gs[0, :])
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d[["county_fips", "tier"]], on="county_fips", how="left")
geo["code"] = pd.Categorical(geo["tier"], categories=TIER_LABELS).codes.astype(float)
geo.loc[geo["code"] < 0, "code"] = np.nan
cmap = ListedColormap(TIER_COLORS)
if geo["code"].isna().any():
    geo[geo["code"].isna()].plot(ax=ax, color=MISSING, linewidth=0.03, edgecolor="white")
geo.dropna(subset=["code"]).plot(column="code", cmap=cmap,
                                 norm=BoundaryNorm(np.arange(-0.5, 6.5), cmap.N),
                                 linewidth=0.03, edgecolor="#bbbbbb", ax=ax)
ax.set_axis_off()
handles = [Patch(facecolor=TIER_COLORS[i], edgecolor="#999999",
                 label=f"{TIER_LABELS[i]}  (n = {int((d['tier'] == TIER_LABELS[i]).sum()):,})")
           for i in range(len(TIER_LABELS))]
handles.append(Patch(facecolor=MISSING, edgecolor="#999999", label="No score"))
ax.legend(handles=handles, loc="lower left", fontsize=10, frameon=False,
          title=f"P(top {PCT}%)", title_fontsize=11)
title(ax, f"(a) Robustness tiers: probability of ranking in the top {PCT}% of SODS")

# (b) rank uncertainty
ax = fig.add_subplot(gs[1, 0])
ax.hist(d["rank_width"], bins=50, color="#4575b4", edgecolor="white", linewidth=0.3)
ax.axvline(med_width, color="black", lw=1.5, label=f"Median = {med_width:,.0f}")
ax.set_xlabel("Rank uncertainty across weightings (P95 \u2212 P5 rank)")
ax.set_ylabel("Number of counties")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
title(ax, "(b) County rank uncertainty under reweighting")

# (c) dimension correlations
ax = fig.add_subplot(gs[1, 1])
im = ax.imshow(corr.values, cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(DIMS, rotation=30, ha="right", fontsize=10)
ax.set_yticklabels(DIMS, fontsize=10)
for i in range(4):
    for j in range(4):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=10, fontweight="bold",
                color="white" if abs(v) > 0.55 else "black")
cb = fig.colorbar(im, ax=ax, shrink=0.78, pad=0.03)
cb.set_label("Spearman correlation", fontsize=10)
title(ax, "(c) Correlation among SODS dimensions")

# (d) dimension-heavy scenarios
ax = fig.add_subplot(gs[2, 0])
p = scen[scen["scenario"] != "Equal"].reset_index(drop=True)
xx, bw = np.arange(len(p)), 0.36
ax.bar(xx - bw / 2, p["spearman_vs_equal"], bw, color="#4575b4", label="Spearman vs. equal")
axr = ax.twinx()
axr.bar(xx + bw / 2, p[f"top{PCT}_overlap_pct"], bw, color="#d73027",
        label=f"Top-{PCT}% overlap")
ax.set_xticks(xx)
ax.set_xticklabels([f"{s}-heavy" for s in p["scenario"]], fontsize=10, rotation=15)
ax.set_ylabel("Spearman correlation", color="#4575b4")
axr.set_ylabel(f"Top-{PCT}% overlap (%)", color="#d73027")
ax.set_ylim(0, 1.2); axr.set_ylim(0, 120)          # headroom for the legend
ax.set_yticks(np.arange(0, 1.01, 0.2)); axr.set_yticks(np.arange(0, 101, 20))
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = axr.get_legend_handles_labels()
axr.legend(h1 + h2, l1 + l2, fontsize=9, loc="upper center", ncol=2, framealpha=0.95)
ax.grid(alpha=0.25, axis="y")
title(ax, "(d) Dimension-heavy scenarios vs. equal weighting")

# (e) robustness vs SODS percentile
ax = fig.add_subplot(gs[2, 1])
ax.scatter(d["R_lhs_pct"], d["P_ATL"], s=12, alpha=0.30, color="#b3a2d4", edgecolors="none")
bins = np.arange(0, 101, 5)
gm = d.groupby(pd.cut(d["R_lhs_pct"], bins=bins, include_lowest=True),
               observed=False)["P_ATL"].mean()
centers = [(iv.left + iv.right) / 2 for iv in gm.index]
ax.plot(centers, gm.values, "-o", color="#54278f", lw=2.4, markersize=4,
        label=f"Mean P(top {PCT}%) per 5-percentile bin")
ax.axhline(0.90, color="#d73027", ls="--", lw=1.2, label="High robustness (0.90)")
ax.axhline(0.50, color="gray", ls=":", lw=1)
ax.set_xlim(0, 100); ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("SODS percentile")
ax.set_ylabel(f"P(top {PCT}%) across weightings")
ax.legend(fontsize=9, loc="upper left")
ax.grid(alpha=0.3)
title(ax, "(e) Robustness vs. SODS percentile")

out_png = f"{FIG_DIR}/fig10_sods_robustness.png"
fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
plt.close(fig)
print(f"\nWrote:\n  {out_png}\n  {OUT_DIR}/sods_dimension_heavy_scenarios.csv")

Counties: 3,204   weightings: 10,000   top-10% cutoff: 321
Median rank uncertainty (P95-P5): 1284
tier
<0.10        2393
0.10–0.25     300
0.25–0.50     285
0.50–0.75     171
0.75–0.90      45
≥0.90          10
          scenario  spearman_vs_equal  top10_overlap_pct
             Equal              1.000            100.000
            Hazard              0.959             80.374
          Severity              0.985             83.489
Disproportionality              0.939             71.340
  Storm-Dependence              0.957             79.128

Wrote:
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig10_sods_robustness.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/sods_dimension_heavy_scenarios.csv


In [12]:
"""
Create revised SODS robustness figure
=====================================

Panel A: existing fig_risk_pvh_tiers.png
Panel B: county rank uncertainty
Panel C: dimension correlation
Panel D: dimension-heavy scenarios
Panel E: robustness vs SODS

Output:
    fig_risk_weight_robustness.png
"""

import os
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
from scipy.stats import spearmanr


# ============================================================
# PATHS
# ============================================================

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"

os.makedirs(FIG_DIR, exist_ok=True)

TIER_FIG = f"{FIG_DIR}/fig_risk_pvh_tiers.png"
OUTPUT_FIG = f"{FIG_DIR}/fig_risk_weight_robustness.png"


# ============================================================
# SETTINGS
# ============================================================

DIMS = [
    "Hazard",
    "Severity",
    "Disproportionality",
    "Storm-Dependence"
]

TARGET_FRAC = 0.10
CORR_CMAP = "RdBu"


# ============================================================
# LOAD DATA
# ============================================================

d = pd.read_csv(
    f"{OUT_DIR}/risk_scores_by_county.csv",
    dtype={"county_fips": str}
)

d["county_fips"] = d["county_fips"].str.zfill(5)

W = pd.read_csv(
    f"{OUT_DIR}/lhs_weight_scenarios.csv"
)[[f"w_{c}" for c in DIMS]].to_numpy()

corr = pd.read_csv(
    f"{OUT_DIR}/correlation_dimensions.csv",
    index_col=0
).loc[DIMS, DIMS]


# ============================================================
# RECREATE RANK UNCERTAINTY
# This fixes the KeyError: 'rank_width'
# ============================================================

X = d[DIMS].to_numpy()

n = len(d)
K = len(W)

scores = W @ X.T

order = np.argsort(-scores, axis=1)

ranks = np.empty_like(order)

ranks[
    np.arange(K)[:, None],
    order
] = np.arange(1, n + 1)

r5, r50, r95 = np.percentile(
    ranks,
    [5, 50, 95],
    axis=0
)

d["rank_median"] = r50
d["rank_p5"] = r5
d["rank_p95"] = r95

d["rank_width"] = (
    d["rank_p95"] -
    d["rank_p5"]
)

med_width_all = d["rank_width"].median()

med_width_top = d.loc[
    d["R_lhs_pct"] >= 100 * (1 - TARGET_FRAC),
    "rank_width"
].median()


# ============================================================
# DIMENSION-HEAVY SCENARIOS
# ============================================================

k = int(np.ceil(TARGET_FRAC * n))

SCEN = {
    "equal": [0.25, 0.25, 0.25, 0.25]
}

for i, dim in enumerate(DIMS):

    w = [0.2, 0.2, 0.2, 0.2]

    w[i] = 0.4

    SCEN[f"{dim}-heavy"] = w


sc_scores = {
    name: X @ np.array(w)
    for name, w in SCEN.items()
}

ref = sc_scores["equal"]

T_ref = set(
    d.iloc[
        np.argsort(-ref)[:k]
    ]["county_fips"]
)

rows = []

for name, sc in sc_scores.items():

    r, _ = spearmanr(
        sc,
        ref,
        nan_policy="omit"
    )

    T = set(
        d.iloc[
            np.argsort(-sc)[:k]
        ]["county_fips"]
    )

    rows.append({
        "scenario": name,
        "spearman_vs_equal": r,
        "top50_overlap_pct": len(T & T_ref) / k * 100
    })

scen = pd.DataFrame(rows)


# ============================================================
# CREATE FIGURE
# ============================================================

fig = plt.figure(
    figsize=(18, 20)
)

gs = fig.add_gridspec(
    3,
    2,
    height_ratios=[1.25, 0.9, 0.9],
    hspace=0.22,
    wspace=0.20
)


# ============================================================
# PANEL A
# EXISTING SIX-TIER ROBUSTNESS FIGURE
# ============================================================

ax = fig.add_subplot(
    gs[0, :]
)

tier_img = plt.imread(
    TIER_FIG
)

ax.imshow(
    tier_img
)

ax.set_axis_off()


# Add "A." separately
ax.text(
    0.01,
    0.98,
    "A.",
    transform=ax.transAxes,
    fontsize=16,
    fontweight="bold",
    ha="left",
    va="top"
)


# ============================================================
# PANEL B
# RANK UNCERTAINTY
# ============================================================

ax = fig.add_subplot(
    gs[1, 0]
)

ax.hist(
    d["rank_width"],
    bins=50,
    color="#4575b4",
    edgecolor="white",
    linewidth=0.3
)

ax.axvline(
    med_width_all,
    color="black",
    lw=1.5,
    label=f"Median = {med_width_all:.0f}"
)

ax.set_xlabel(
    "Rank uncertainty across weightings (p95 - p5 rank)"
)

ax.set_ylabel(
    "Number of counties"
)

ax.set_title(
    "B. County Rank Uncertainty Under Reweighting",
    fontsize=13,
    fontweight="bold"
)

ax.legend(
    fontsize=9
)

ax.grid(
    alpha=0.3
)


# ============================================================
# PANEL C
# DIMENSION CORRELATION
# ============================================================

ax = fig.add_subplot(
    gs[1, 1]
)

im = ax.imshow(
    corr.values,
    cmap=CORR_CMAP,
    vmin=-1,
    vmax=1
)

ax.set_xticks(
    range(len(DIMS))
)

ax.set_yticks(
    range(len(DIMS))
)

ax.set_xticklabels(
    DIMS,
    rotation=30,
    ha="right",
    fontsize=10
)

ax.set_yticklabels(
    DIMS,
    fontsize=10
)

for i in range(len(DIMS)):

    for j in range(len(DIMS)):

        v = corr.iloc[i, j]

        ax.text(
            j,
            i,
            f"{v:.2f}",
            ha="center",
            va="center",
            fontsize=10,
            fontweight="bold",
            color="white" if abs(v) > 0.55 else "black"
        )


ax.set_title(
    "C. Correlation Among Storm-Outage Disruption Index Dimensions",
    fontsize=13,
    fontweight="bold"
)

cbar = fig.colorbar(
    im,
    ax=ax,
    shrink=0.78,
    pad=0.03
)

cbar.set_label(
    "Spearman correlation",
    fontsize=10
)


# ============================================================
# PANEL D
# DIMENSION-HEAVY SCENARIOS
# ============================================================

ax = fig.add_subplot(
    gs[2, 0]
)

plot = scen[
    scen["scenario"] != "equal"
].copy()

xx = np.arange(
    len(plot)
)

bw = 0.36


ax.bar(
    xx - bw / 2,
    plot["spearman_vs_equal"],
    bw,
    color="#4575b4",
    label="Spearman vs equal"
)

axr = ax.twinx()

axr.bar(
    xx + bw / 2,
    plot["top50_overlap_pct"],
    bw,
    color="#d73027",
    label=f"Top-{int(TARGET_FRAC*100)}% overlap"
)


ax.set_xticks(
    xx
)

ax.set_xticklabels(
    plot["scenario"].str.replace(
        "-heavy",
        "",
        regex=False
    ),
    fontsize=10
)

ax.set_ylabel(
    "Spearman correlation",
    color="#4575b4"
)

ax.set_ylim(
    0,
    1.02
)

axr.set_ylabel(
    f"Top-{int(TARGET_FRAC*100)}% overlap (%)",
    color="#d73027"
)

axr.set_ylim(
    0,
    102
)


ax.set_title(
    "D. Dimension-Heavy Scenarios vs. Equal Weighting",
    fontsize=13,
    fontweight="bold"
)


h1, l1 = ax.get_legend_handles_labels()
h2, l2 = axr.get_legend_handles_labels()

leg = ax.legend(
    h1 + h2,
    l1 + l2,
    fontsize=8,
    loc="upper left",
    framealpha=0.95,
    facecolor="white",
    edgecolor="#888"
)

leg.set_zorder(10)

ax.grid(
    alpha=0.25,
    axis="y"
)


# ============================================================
# PANEL E
# ROBUSTNESS VS SODS
# ============================================================

ax = fig.add_subplot(
    gs[2, 1]
)

ax.scatter(
    d["R_lhs_pct"],
    d["P_ATL"],
    s=12,
    alpha=0.30,
    color="#b3a2d4",
    edgecolors="none"
)


bins = np.arange(
    0,
    101,
    5
)

d["_bin"] = pd.cut(
    d["R_lhs_pct"],
    bins=bins,
    include_lowest=True
)

gm = d.groupby(
    "_bin",
    observed=False
)["P_ATL"].mean()


centers = np.array([
    (iv.left + iv.right) / 2
    for iv in gm.index
])


ax.plot(
    centers,
    gm.values,
    "-o",
    color="#54278f",
    lw=2.4,
    markersize=4,
    label=f"Mean P(top {int(TARGET_FRAC*100)}%) per 5-pct bin"
)


ax.axhline(
    0.90,
    color="#d73027",
    ls="--",
    lw=1.2,
    label="High robustness (0.90)"
)

ax.axhline(
    0.50,
    color="gray",
    ls=":",
    lw=1
)


ax.set_xlim(
    0,
    100
)

ax.set_ylim(
    -0.02,
    1.02
)


ax.set_xlabel(
    "SODS percentile (LHS-mean)"
)

ax.set_ylabel(
    f"P(top {int(TARGET_FRAC*100)}%) across weightings"
)


ax.set_title(
    "E. Robustness vs. Storm-Outage Disruption Index Score",
    fontsize=13,
    fontweight="bold"
)


ax.legend(
    fontsize=8,
    loc="upper left"
)

ax.grid(
    alpha=0.3
)


# ============================================================
# FINAL LAYOUT
# ============================================================

fig.suptitle(
    "Weight Robustness of the Storm-Outage Disruption Index (SODS)",
    fontsize=18,
    fontweight="bold",
    y=0.995
)

fig.subplots_adjust(
    top=0.95,
    bottom=0.05,
    left=0.055,
    right=0.95
)


fig.savefig(
    OUTPUT_FIG,
    dpi=600,
    bbox_inches="tight"
)

plt.close(fig)


print(
    f"Saved: {OUTPUT_FIG}"
)

Saved: /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig_risk_weight_robustness.png


In [13]:
"""
Storm-Outage Disruption Index (SODS): four dimensions, LHS-weighted (top-50% target)
====================================================================
Dimensions (each an equal-average of its sub-elements, all as percentiles):
  Hazard   = mean( storm exposure count* , storm duration* )
  Severity = mean( normalized customer-hours* , P90 recovery* )
  Disproportionality = DR*  where DR = norm-burden pct - exposure pct
  Storm-Dependence   = storm-linkage share* (share of outages storm-linked at 12h)
(* = percentile 0-100)

SODS (observed disruption index) = R_lhs_mean, the mean weighted score across K LHS weightings on the
4-D simplex. (Baseline equal-weight risk is dropped: it is ~identical to the
LHS mean by construction.)

P_ATL = fraction of weightings in which a county is in the TOP TARGET_FRAC of
counties. With TARGET_FRAC = 0.50 this is "reliably above-median risk", which
fills the map instead of leaving most counties in a 'rare' bucket.

Six robustness tiers on P_ATL:
  >=0.90, 0.75-0.90, 0.50-0.75, 0.25-0.50, 0.10-0.25, <0.10

Outputs (Start_From_Begining) -- a CSV per stage:
  risk_elements_by_county.csv, correlation_subelements.csv,
  correlation_dimensions.csv, lhs_weight_scenarios.csv,
  risk_scores_by_county.csv, risk_class_counts.csv
Figures:
  fig_risk_correlation.png, fig_risk_dimension_maps.png,
  fig_risk_pvh_surface.png       (continuous P_ATL choropleth)
  fig_risk_pvh_tiers.png         (6-tier classified map)
  fig_risk_lhs_mean.png          (LHS-mean risk map)
Requires: pandas, numpy, matplotlib, geopandas, scipy
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, PowerNorm, BoundaryNorm
from matplotlib.patches import Patch
from scipy.stats.qmc import LatinHypercube
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

LINKED_CSV = f"{Q1_3}/storm_linked_outage_events_2015_2025.csv"
STORM_CSV = f"{Q1_3}/county_level_storm_records_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#e6e6e6"

BUFFER = 12
K = 10000
SEED = 42
TARGET_FRAC = 0.10          # top-10% target
GAMMA = 2.5

# six robustness tiers on P_ATL
TIER_BINS = [-0.001, 0.10, 0.25, 0.50, 0.75, 0.90, 1.001]
TIER_LABELS = ["<0.10", "0.10-0.25", "0.25-0.50", "0.50-0.75", "0.75-0.90", ">=0.90"]
TIER_COLORS = ["#f7f7f7", "#fddbc7", "#f4a582", "#d6604d", "#b2182b", "#67001f"]


def load_mcc(path):
    m = pd.read_csv(path); m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    m = m[fn.notna() & m[cc].notna()]
    return dict(zip(fn[fn.notna()].astype("int64").astype(str).str.zfill(5), m[cc]))


def pctl(s):
    return s.rank(pct=True, method="average") * 100.0


def truthy(s):
    return s.astype(str).str.lower().isin(["true", "1", "1.0", "yes", "t"])


def p90(s):
    s = s.dropna()
    return float(np.percentile(s, 90)) if len(s) else np.nan


# ---- sub-elements per county ----
st = pd.read_csv(STORM_CSV, dtype={"county_fips": str})
st["county_fips"] = st["county_fips"].str.zfill(5)
st["s_start"] = pd.to_datetime(st["start_time"], errors="coerce")
st["s_end"] = pd.to_datetime(st["end_time"], errors="coerce")
st["dur"] = ((st["s_end"] - st["s_start"]).dt.total_seconds() / 3600).clip(lower=0)
se = st.groupby("county_fips").agg(exposure=("event_type", "size"),
                                   duration=("dur", "sum")).reset_index()

o = pd.read_csv(LINKED_CSV, dtype={"county_fips": str})
o["county_fips"] = o["county_fips"].str.zfill(5)
o["auc"] = pd.to_numeric(o["auc"], errors="coerce")
o["recovery_hr"] = pd.to_numeric(o["recovery_hr"], errors="coerce")
o["linked"] = truthy(o[f"storm_linked_{BUFFER}h"])
link = o.groupby("county_fips")["linked"].mean().rename("linkage_share").reset_index()
ol = o[o["linked"]]
burden = ol.groupby("county_fips").agg(
    customer_hours=("auc", "sum"),
    p90_recovery=("recovery_hr", p90)).reset_index()

mcc = load_mcc(MCC_CSV)
d = se.merge(burden, on="county_fips", how="outer").merge(link, on="county_fips", how="outer")
sv = d["county_fips"].map(mcc).astype(float); sv = sv.where(sv > 0, np.nan)
d["norm_burden"] = d["customer_hours"] / sv
for c in ["exposure", "duration", "customer_hours", "p90_recovery",
          "linkage_share", "norm_burden"]:
    d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)

d["exposure_pct"] = pctl(d["exposure"])
d["duration_pct"] = pctl(d["duration"])
d["norm_burden_pct"] = pctl(d["norm_burden"])
d["p90_recovery_pct"] = pctl(d["p90_recovery"])
d["linkage_pct"] = pctl(d["linkage_share"])
d["DR"] = d["norm_burden_pct"] - d["exposure_pct"]
d["DR_pct"] = pctl(d["DR"])

d["Hazard"] = d[["exposure_pct", "duration_pct"]].mean(axis=1)
d["Severity"] = d[["norm_burden_pct", "p90_recovery_pct"]].mean(axis=1)
d["Disproportionality"] = d["DR_pct"]
d["Storm-Dependence"] = d["linkage_pct"]
DIMS = ["Hazard", "Severity", "Disproportionality", "Storm-Dependence"]
d.to_csv(f"{OUT_DIR}/risk_elements_by_county.csv", index=False)
print(f"counties: {len(d):,}")

# ---- correlation ----
SUBS = ["exposure_pct", "duration_pct", "norm_burden_pct", "p90_recovery_pct",
        "DR_pct", "linkage_pct"]
SUBLAB = ["Exposure", "Duration", "Norm burden", "P90 recovery",
          "DR (disproportionality)", "Storm-dependence"]
corr_sub = d[SUBS].corr(method="spearman")
corr_sub.to_csv(f"{OUT_DIR}/correlation_subelements.csv")
corr_dim = d[DIMS].corr(method="spearman")
corr_dim.to_csv(f"{OUT_DIR}/correlation_dimensions.csv")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(18, 8))
for ax, corr, labs, ttl in [(a1, corr_sub, SUBLAB, "Sub-elements"),
                            (a2, corr_dim, DIMS, "Dimensions")]:
    im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(labs))); ax.set_yticks(range(len(labs)))
    ax.set_xticklabels(labs, rotation=40, ha="right"); ax.set_yticklabels(labs)
    for i in range(len(labs)):
        for j in range(len(labs)):
            v = corr.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if abs(v) > 0.55 else "black")
    ax.set_title(f"Spearman correlation — {ttl}", fontsize=13, fontweight="bold")
    fig.colorbar(im, ax=ax, shrink=0.7)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_correlation.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# ---- LHS ----
X = d[DIMS].to_numpy()
u = LatinHypercube(d=4, seed=SEED).random(n=K)
e = -np.log(1.0 - u); W = e / e.sum(axis=1, keepdims=True)
pd.DataFrame(W, columns=[f"w_{c}" for c in DIMS]).assign(
    scenario=np.arange(1, K + 1)).to_csv(
    f"{OUT_DIR}/lhs_weight_scenarios.csv", index=False)
scores = W @ X.T
n = len(d)
order = np.argsort(-scores, axis=1)
ranks = np.empty_like(order); ranks[np.arange(K)[:, None], order] = np.arange(1, n + 1)
kv = max(1, int(np.ceil(TARGET_FRAC * n)))          # top-50% cutoff rank
d["P_ATL"] = (ranks <= kv).mean(axis=0)             # prob. above target line
d["R_lhs_mean"] = scores.mean(axis=0)
d["R_lhs_pct"] = pctl(d["R_lhs_mean"])
d["robustness_tier"] = pd.cut(d["P_ATL"], bins=TIER_BINS, labels=TIER_LABELS)

keep = ["county_fips"] + DIMS + ["R_lhs_mean", "R_lhs_pct", "P_ATL",
        "robustness_tier"]
d[keep].to_csv(f"{OUT_DIR}/risk_scores_by_county.csv", index=False)
cc = d["robustness_tier"].value_counts().reindex(TIER_LABELS).rename("n")
cc.to_frame().to_csv(f"{OUT_DIR}/risk_class_counts.csv")
print(f"target = top {int(TARGET_FRAC*100)}% (cutoff rank {kv})")
print("robustness tiers:\n", cc.to_string())

# ---- maps ----
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d, on="county_fips", how="left")
pn = PowerNorm(gamma=GAMMA, vmin=0, vmax=100)

# dimension maps
fig, axes = plt.subplots(2, 2, figsize=(18, 11))
for ax, dim in zip(axes.ravel(), DIMS):
    geo.plot(column=dim, cmap="magma_r", norm=pn, linewidth=0.03, edgecolor="white",
             ax=ax, legend=True, missing_kwds={"color": MISSING},
             legend_kwds={"shrink": 0.6, "label": f"{dim} (0-100)"})
    ax.set_axis_off(); ax.set_title(dim, fontsize=13, fontweight="bold")
fig.suptitle("Storm-Outage Disruption Index (SODS) dimensions", fontsize=16, fontweight="bold", y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{FIG_DIR}/fig_risk_dimension_maps.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# LHS-mean risk map (single)
fig, ax = plt.subplots(figsize=(15, 9))
geo.plot(column="R_lhs_pct", cmap="magma_r", norm=pn, linewidth=0.03,
         edgecolor="white", ax=ax, legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6, "label": "SODS percentile (LHS-mean)"})
ax.set_axis_off()
ax.set_title("Storm-Outage Disruption Index (SODS), observed 2015-2025",
             fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_lhs_mean.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# continuous P_ATL surface
fig, ax = plt.subplots(figsize=(15, 9))
geo.plot(column="P_ATL", cmap="magma_r", vmin=0, vmax=1, linewidth=0.03,
         edgecolor="white", ax=ax, legend=True, missing_kwds={"color": MISSING},
         legend_kwds={"shrink": 0.6,
                      "label": f"P(top {int(TARGET_FRAC*100)}%) across weightings"})
ax.set_axis_off()
ax.set_title(f"SODS robustness surface: probability of top-{int(TARGET_FRAC*100)}%\n"
             "under 10,000 weightings", fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_pvh_surface.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# six-tier classified map
fig, ax = plt.subplots(figsize=(15, 9))
tier_cmap = ListedColormap(TIER_COLORS)
geo["_tier_code"] = pd.Categorical(geo["robustness_tier"],
                                   categories=TIER_LABELS).codes
geo.loc[geo["_tier_code"] < 0, "_tier_code"] = np.nan
geo[geo["_tier_code"].isna()].plot(ax=ax, color=MISSING, linewidth=0.03,
                                   edgecolor="white")
geo.dropna(subset=["_tier_code"]).plot(
    column="_tier_code", cmap=tier_cmap,
    norm=BoundaryNorm(np.arange(-0.5, len(TIER_LABELS) + 0.5), tier_cmap.N),
    linewidth=0.03, edgecolor="white", ax=ax)
ax.set_axis_off()
ax.set_title(f"SODS robustness tiers: P(top {int(TARGET_FRAC*100)}%) across weightings",
             fontsize=15, fontweight="bold")
handles = [Patch(color=TIER_COLORS[i],
           label=f"{TIER_LABELS[i]}  (n={int((d['robustness_tier']==TIER_LABELS[i]).sum())})")
           for i in range(len(TIER_LABELS))]
handles.append(Patch(color=MISSING, label="No score"))
ax.legend(handles=handles, loc="lower left", fontsize=9, frameon=False,
          title="P(above target)")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_risk_pvh_tiers.png", dpi=600, bbox_inches="tight")
plt.close(fig)

print(f"\nWrote CSVs + figures to {OUT_DIR}")

counties: 3,204
target = top 10% (cutoff rank 321)
robustness tiers:
 robustness_tier
<0.10        2393
0.10-0.25     300
0.25-0.50     285
0.50-0.75     171
0.75-0.90      45
>=0.90         10


/local/tmp/ipykernel_2667540/3715477259.py:238: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  geo[geo["_tier_code"].isna()].plot(ax=ax, color=MISSING, linewidth=0.03,



Wrote CSVs + figures to /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining


In [14]:
"""
Figure 10 — Weight robustness of the Storm-Outage Disruption Score (SODS), 2015-2025.

  (a) County robustness tiers: probability of ranking in the top TARGET_FRAC
  (b) County rank uncertainty (P95 - P5 rank across weightings)
  (c) Spearman correlations among the four SODS dimensions
  (d) Dimension-heavy scenarios vs. equal weighting (Spearman and top-k overlap)
  (e) Robustness vs. SODS percentile

Same calculations as the earlier robustness cell, with SODS labels. Panel (a) is
now drawn directly from the county scores instead of pasting the old tier image,
so it no longer carries the old "SODI" title.

Reads : Results/Start_From_Begining/risk_scores_by_county.csv
        Results/Start_From_Begining/lhs_weight_scenarios.csv
        Results/Start_From_Begining/correlation_dimensions.csv
        RQ2/shapefiles/cb_2022_us_county_5m.shp
Writes: Results/Start_From_Begining/figures/fig10_sods_robustness.png
        Results/Start_From_Begining/sods_dimension_heavy_scenarios.csv
Requires: pandas, numpy, matplotlib, scipy, geopandas
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from scipy.stats import spearmanr
import geopandas as gpd

# ----------------------------------------------------------------------------
# Settings
# ----------------------------------------------------------------------------
BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
os.makedirs(FIG_DIR, exist_ok=True)

DIMS = ["Hazard", "Severity", "Disproportionality", "Storm-Dependence"]
TARGET_FRAC = 0.10        # must match the value used when P_ATL was computed
DPI = 600
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#e6e6e6"

TIER_BINS = [-0.001, 0.10, 0.25, 0.50, 0.75, 0.90, 1.001]
TIER_LABELS = ["<0.10", "0.10\u20130.25", "0.25\u20130.50", "0.50\u20130.75",
               "0.75\u20130.90", "\u22650.90"]
TIER_COLORS = ["#f7f7f7", "#fddbc7", "#f4a582", "#d6604d", "#b2182b", "#67001f"]
PCT = int(TARGET_FRAC * 100)

SHOW_TITLE = True
FIG_TITLE = "Weight Robustness of the Storm-Outage Disruption Score (SODS), 2015\u20132025"

# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
d = pd.read_csv(f"{OUT_DIR}/risk_scores_by_county.csv", dtype={"county_fips": str})
d["county_fips"] = d["county_fips"].str.zfill(5)
W = pd.read_csv(f"{OUT_DIR}/lhs_weight_scenarios.csv")[[f"w_{c}" for c in DIMS]].to_numpy()
corr = pd.read_csv(f"{OUT_DIR}/correlation_dimensions.csv", index_col=0).loc[DIMS, DIMS]

# Rank uncertainty across the LHS weightings
X = d[DIMS].to_numpy()
n, K = len(d), len(W)
scores = W @ X.T
order = np.argsort(-scores, axis=1)
ranks = np.empty_like(order)
ranks[np.arange(K)[:, None], order] = np.arange(1, n + 1)
r5, r95 = np.percentile(ranks, [5, 95], axis=0)
d["rank_width"] = r95 - r5
med_width = d["rank_width"].median()

# Robustness tiers from P_ATL
d["tier"] = pd.cut(d["P_ATL"], bins=TIER_BINS, labels=TIER_LABELS)

# Dimension-heavy scenarios vs equal weighting
k = int(np.ceil(TARGET_FRAC * n))
scen_w = {"Equal": [0.25] * 4}
for i, dim in enumerate(DIMS):
    w = [0.2] * 4
    w[i] = 0.4
    scen_w[dim] = w
sc = {name: X @ np.array(w) for name, w in scen_w.items()}
top_ref = set(np.argsort(-sc["Equal"])[:k])
rows = []
for name, s in sc.items():
    r, _ = spearmanr(s, sc["Equal"], nan_policy="omit")
    rows.append({"scenario": name, "spearman_vs_equal": r,
                 f"top{PCT}_overlap_pct": 100 * len(set(np.argsort(-s)[:k]) & top_ref) / k})
scen = pd.DataFrame(rows)
scen.to_csv(f"{OUT_DIR}/sods_dimension_heavy_scenarios.csv", index=False)

print(f"Counties: {n:,}   weightings: {K:,}   top-{PCT}% cutoff: {k}")
print(f"Median rank uncertainty (P95-P5): {med_width:.0f}")
print(d["tier"].value_counts().reindex(TIER_LABELS).to_string())
print(scen.round(3).to_string(index=False))

# ----------------------------------------------------------------------------
# Figure
# ----------------------------------------------------------------------------
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.facecolor": "white"})
fig = plt.figure(figsize=(18, 20))
gs = fig.add_gridspec(3, 2, height_ratios=[1.25, 0.9, 0.9], hspace=0.25, wspace=0.25,
                      left=0.06, right=0.95, top=0.94 if SHOW_TITLE else 0.97, bottom=0.05)


def title(ax, text):
    ax.set_title(text, fontsize=13, fontweight="bold", loc="center")


# (a) robustness tiers map
ax = fig.add_subplot(gs[0, :])
gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d[["county_fips", "tier"]], on="county_fips", how="left")
geo["code"] = pd.Categorical(geo["tier"], categories=TIER_LABELS).codes.astype(float)
geo.loc[geo["code"] < 0, "code"] = np.nan
cmap = ListedColormap(TIER_COLORS)
if geo["code"].isna().any():
    geo[geo["code"].isna()].plot(ax=ax, color=MISSING, linewidth=0.03, edgecolor="white")
geo.dropna(subset=["code"]).plot(column="code", cmap=cmap,
                                 norm=BoundaryNorm(np.arange(-0.5, 6.5), cmap.N),
                                 linewidth=0.03, edgecolor="#bbbbbb", ax=ax)
ax.set_axis_off()
handles = [Patch(facecolor=TIER_COLORS[i], edgecolor="#999999",
                 label=f"{TIER_LABELS[i]}  (n = {int((d['tier'] == TIER_LABELS[i]).sum()):,})")
           for i in range(len(TIER_LABELS))]
handles.append(Patch(facecolor=MISSING, edgecolor="#999999", label="No score"))
ax.legend(handles=handles, loc="lower left", fontsize=10, frameon=False,
          title=f"P(top {PCT}%)", title_fontsize=11)
title(ax, f"(a) Robustness tiers: probability of ranking in the top {PCT}% of SODS")

# (b) rank uncertainty
ax = fig.add_subplot(gs[1, 0])
ax.hist(d["rank_width"], bins=50, color="#4575b4", edgecolor="white", linewidth=0.3)
ax.axvline(med_width, color="black", lw=1.5, label=f"Median = {med_width:,.0f}")
ax.set_xlabel("Rank uncertainty across weightings (P95 \u2212 P5 rank)")
ax.set_ylabel("Number of counties")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
title(ax, "(b) County rank uncertainty under reweighting")

# (c) dimension correlations
ax = fig.add_subplot(gs[1, 1])
im = ax.imshow(corr.values, cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(DIMS, rotation=30, ha="right", fontsize=10)
ax.set_yticklabels(DIMS, fontsize=10)
for i in range(4):
    for j in range(4):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=10, fontweight="bold",
                color="white" if abs(v) > 0.55 else "black")
cb = fig.colorbar(im, ax=ax, shrink=0.78, pad=0.03)
cb.set_label("Spearman correlation", fontsize=10)
title(ax, "(c) Correlation among SODS dimensions")

# (d) dimension-heavy scenarios
ax = fig.add_subplot(gs[2, 0])
p = scen[scen["scenario"] != "Equal"].reset_index(drop=True)
xx, bw = np.arange(len(p)), 0.36
ax.bar(xx - bw / 2, p["spearman_vs_equal"], bw, color="#4575b4", label="Spearman vs. equal")
axr = ax.twinx()
axr.bar(xx + bw / 2, p[f"top{PCT}_overlap_pct"], bw, color="#d73027",
        label=f"Top-{PCT}% overlap")
ax.set_xticks(xx)
ax.set_xticklabels([f"{s}-heavy" for s in p["scenario"]], fontsize=10, rotation=15)
ax.set_ylabel("Spearman correlation", color="#4575b4")
axr.set_ylabel(f"Top-{PCT}% overlap (%)", color="#d73027")
ax.set_ylim(0, 1.2); axr.set_ylim(0, 120)          # headroom for the legend
ax.set_yticks(np.arange(0, 1.01, 0.2)); axr.set_yticks(np.arange(0, 101, 20))
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = axr.get_legend_handles_labels()
axr.legend(h1 + h2, l1 + l2, fontsize=9, loc="upper center", ncol=2, framealpha=0.95)
ax.grid(alpha=0.25, axis="y")
title(ax, "(d) Dimension-heavy scenarios vs. equal weighting")

# (e) robustness vs SODS percentile
ax = fig.add_subplot(gs[2, 1])
ax.scatter(d["R_lhs_pct"], d["P_ATL"], s=12, alpha=0.30, color="#b3a2d4", edgecolors="none")
bins = np.arange(0, 101, 5)
gm = d.groupby(pd.cut(d["R_lhs_pct"], bins=bins, include_lowest=True),
               observed=False)["P_ATL"].mean()
centers = [(iv.left + iv.right) / 2 for iv in gm.index]
ax.plot(centers, gm.values, "-o", color="#54278f", lw=2.4, markersize=4,
        label=f"Mean P(top {PCT}%) per 5-percentile bin")
ax.axhline(0.90, color="#d73027", ls="--", lw=1.2, label="High robustness (0.90)")
ax.axhline(0.50, color="gray", ls=":", lw=1)
ax.set_xlim(0, 100); ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("SODS percentile")
ax.set_ylabel(f"P(top {PCT}%) across weightings")
ax.legend(fontsize=9, loc="upper left")
ax.grid(alpha=0.3)
title(ax, "(e) Robustness vs. SODS percentile")

if SHOW_TITLE:
    fig.suptitle(FIG_TITLE, fontsize=18, fontweight="bold", y=0.985)

out_png = f"{FIG_DIR}/fig10_sods_robustness.png"
fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
plt.close(fig)
print(f"\nWrote:\n  {out_png}\n  {OUT_DIR}/sods_dimension_heavy_scenarios.csv")

Counties: 3,204   weightings: 10,000   top-10% cutoff: 321
Median rank uncertainty (P95-P5): 1284
tier
<0.10        2393
0.10–0.25     300
0.25–0.50     285
0.50–0.75     171
0.75–0.90      45
≥0.90          10
          scenario  spearman_vs_equal  top10_overlap_pct
             Equal              1.000            100.000
            Hazard              0.959             80.374
          Severity              0.985             83.489
Disproportionality              0.939             71.340
  Storm-Dependence              0.957             79.128

Wrote:
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig10_sods_robustness.png
  /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/sods_dimension_heavy_scenarios.csv


In [1]:
"""
Figure 3 — Concentration of power outage burden across U.S. counties, 2015-2025.

Same data and calculations as the notebook's concentration cell, with:
  * larger fonts throughout (titles, axes, ticks, legends, colorbars)
  * conventional Lorenz curves: counties ranked from LOWEST to HIGHEST burden,
    so the curves lie below the line of equality (set LORENZ_ORDER below)
  * lowercase panel labels (a)-(d), matching the other figures
Gini values are identical to the earlier version (Gini does not depend on order).

Reads : Results/Q1_3/outage_event_table_2015_2025.csv, MCC.csv, county shapefile
Writes: Results/Start_From_Begining/figures/fig3_outage_concentration.png
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
Q1_3 = f"{BASE}/Results/Q1_3"
MCC_CSV = f"{BASE}/MCC.csv"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
OUTAGE_CSV = f"{Q1_3}/outage_event_table_2015_2025.csv"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
os.makedirs(FIG_DIR, exist_ok=True)

LORENZ_ORDER = "ascending"   # "ascending" = conventional Lorenz; "descending" = old version
SHOW_TITLE = True
FIG_TITLE = "Outage Burden Concentration, 2015\u20132025"

NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING_COLOR = "#d9d9d9"
ABS_COLOR, NORM_COLOR = "#238b45", "#6a51a3"
GAMMA = 2.5
GREEN_CMAP = LinearSegmentedColormap.from_list("burden_green",
    ["#ffffff", "#f7fcfd", "#e5f5f9", "#ccece6", "#99d8c9",
     "#66c2a4", "#41ae76", "#238b45", "#006d2c", "#00441b"], N=256)
PURPLE_CMAP = LinearSegmentedColormap.from_list("burden_purple",
    ["#ffffff", "#fcfbfd", "#efedf5", "#dadaeb", "#bcbddc",
     "#9e9ac8", "#807dba", "#6a51a3", "#54278f", "#3f007d"], N=256)

# Font sizes
FS_SUPTITLE, FS_TITLE, FS_LABEL, FS_TICK, FS_LEGEND, FS_CBAR = 24, 17, 15, 13, 13, 14


def load_mcc(path):
    m = pd.read_csv(path, encoding="utf-8-sig")
    m.columns = [c.strip() for c in m.columns]
    fc = "County_FIPS" if "County_FIPS" in m.columns else m.columns[0]
    cc = "Customers" if "Customers" in m.columns else m.columns[1]
    m[cc] = pd.to_numeric(m[cc], errors="coerce")
    fn = pd.to_numeric(m[fc], errors="coerce")
    ok = fn.notna() & m[cc].notna()
    return dict(zip(fn[ok].astype("int64").astype(str).str.zfill(5), m.loc[ok, cc]))


def lorenz_xy(values, order=LORENZ_ORDER):
    x = pd.to_numeric(pd.Series(values), errors="coerce").dropna().values
    x = np.sort(x[x > 0])
    if order == "descending":
        x = x[::-1]
    if x.size == 0:
        return None, None
    cum_b = np.cumsum(x) / x.sum() * 100.0
    cum_c = np.arange(1, x.size + 1) / x.size * 100.0
    return np.concatenate([[0], cum_c]), np.concatenate([[0], cum_b])


def gini(values):
    v = np.sort(pd.to_numeric(pd.Series(values), errors="coerce").dropna().values)
    v = v[v >= 0]
    if v.size == 0 or v.sum() == 0:
        return np.nan
    n = v.size
    cum = np.cumsum(v)
    return (n + 1 - 2 * (cum / cum[-1]).sum()) / n


ot = pd.read_csv(OUTAGE_CSV, dtype={"county_fips": str})
ot["county_fips"] = ot["county_fips"].str.zfill(5)
ot["year"] = pd.to_numeric(ot["year"], errors="coerce")
ot["auc"] = pd.to_numeric(ot["auc"], errors="coerce")
mcc = load_mcc(MCC_CSV)


def county_metrics(sub):
    g = sub.groupby("county_fips").agg(abs_burden=("auc", "sum")).reset_index()
    sv = g["county_fips"].map(mcc).astype(float)
    g["norm_burden"] = g["abs_burden"] / sv.where(sv > 0, np.nan)
    return g


pooled = county_metrics(ot)
years = sorted(int(y) for y in ot["year"].dropna().unique())
per_year = {yr: county_metrics(ot[ot["year"] == yr]) for yr in years}

pooled["abs_pct"] = pooled["abs_burden"].rank(pct=True) * 100
pooled["norm_pct"] = pooled["norm_burden"].rank(pct=True) * 100

gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(pooled, on="county_fips", how="left")

g_abs, g_norm = gini(pooled["abs_burden"]), gini(pooled["norm_burden"])
print(f"Pooled Gini: absolute = {g_abs:.2f}, normalized = {g_norm:.2f}")

# ----------------------------------------------------------------------------
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.facecolor": "white",
                     "xtick.labelsize": FS_TICK, "ytick.labelsize": FS_TICK})
TITLE_KW = dict(fontsize=FS_TITLE, fontweight="bold", pad=10)
fig = plt.figure(figsize=(18, 14.5))

# (a) Lorenz curves
ax1 = fig.add_subplot(2, 2, 1)
ax1.plot([0, 100], [0, 100], "k--", lw=1.2, alpha=0.7, label="Line of equality")
for col, color, name, g in [("abs_burden", ABS_COLOR, "Absolute burden", g_abs),
                            ("norm_burden", NORM_COLOR, "Normalized burden", g_norm)]:
    cx, cy = lorenz_xy(pooled[col])
    if cx is not None:
        ax1.plot(cx, cy, lw=2.6, color=color, label=f"{name} (Gini = {g:.2f})")
rank_txt = "lowest" if LORENZ_ORDER == "ascending" else "highest"
ax1.set_xlabel(f"Cumulative share of counties (%), ranked from {rank_txt} burden",
               fontsize=FS_LABEL)
ax1.set_ylabel("Cumulative share of total burden (%)", fontsize=FS_LABEL)
ax1.set_title("(a) Lorenz curves of outage burden", **TITLE_KW)
ax1.set_xlim(0, 100); ax1.set_ylim(0, 100); ax1.set_aspect("equal")
ax1.legend(loc="upper left" if LORENZ_ORDER == "ascending" else "lower right",
           fontsize=FS_LEGEND, framealpha=0.95)
ax1.grid(alpha=0.3)

# (b) Gini per year
ax2 = fig.add_subplot(2, 2, 2)
ax2.plot(years, [gini(per_year[y]["abs_burden"]) for y in years], "-o", lw=2,
         markersize=7, color=ABS_COLOR, label="Absolute burden")
ax2.plot(years, [gini(per_year[y]["norm_burden"]) for y in years], "-o", lw=2,
         markersize=7, color=NORM_COLOR, label="Normalized burden")
ax2.set_xlabel("Year", fontsize=FS_LABEL)
ax2.set_ylabel("Gini coefficient", fontsize=FS_LABEL)
ax2.set_title("(b) Annual Gini coefficients", **TITLE_KW)
ax2.set_xticks(years[::2] if len(years) > 6 else years)
ax2.set_ylim(0.7, 1.0); ax2.grid(alpha=0.3)
ax2.legend(fontsize=FS_LEGEND, loc="upper left", framealpha=0.95)

# (c, d) maps
norm = PowerNorm(gamma=GAMMA, vmin=0, vmax=100)
for pos, col, cmap, lab, title in [
        (3, "abs_pct", GREEN_CMAP, "Absolute burden (percentile)",
         "(c) Absolute customer-hour burden"),
        (4, "norm_pct", PURPLE_CMAP, "Normalized burden (percentile)",
         "(d) Normalized customer-hour burden")]:
    ax = fig.add_subplot(2, 2, pos)
    geo.plot(column=col, cmap=cmap, norm=norm, linewidth=0.04, edgecolor="white",
             ax=ax, missing_kwds={"color": MISSING_COLOR})
    ax.set_axis_off()
    ax.set_title(title, **TITLE_KW)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cb = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
    cb.set_label(lab, fontsize=FS_CBAR)
    cb.set_ticks([0, 50, 70, 80, 90, 100])        # spaced for the gamma stretch
    cb.ax.tick_params(labelsize=FS_TICK)

if SHOW_TITLE:
    fig.suptitle(FIG_TITLE, fontsize=FS_SUPTITLE, fontweight="bold", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.97 if SHOW_TITLE else 1])
out = f"{FIG_DIR}/fig3_outage_concentration.png"
fig.savefig(out, dpi=600)
plt.close(fig)
print(f"Wrote {out}")

Pooled Gini: absolute = 0.75, normalized = 0.91
Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/fig3_outage_concentration.png


In [4]:
"""
Supplementary Figure 3 — The four SODS dimensions across CONUS counties, 2015-2025.

Same layout as the original notebook figure (2 x 2 maps, one colorbar per map,
centred titles, overall title), with the first dimension displayed as "Storm".
Reads the existing scores file, so nothing needs to be recomputed.

Reads : Results/Start_From_Begining/risk_scores_by_county.csv
        RQ2/shapefiles/cb_2022_us_county_5m.shp
Writes: Results/Start_From_Begining/figures/supp_fig3_sods_dimensions.png
"""
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
import geopandas as gpd

BASE = "/icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill"
OUT_DIR = f"{BASE}/Results/Start_From_Begining"
FIG_DIR = f"{OUT_DIR}/figures"
SHP = f"{BASE}/RQ2/shapefiles/cb_2022_us_county_5m.shp"
os.makedirs(FIG_DIR, exist_ok=True)

DIMS = ["Storm", "Severity", "Disproportionality", "Storm-Dependence"]
LABELS = ["(a) Storm", "(b) Severity", "(c) Disproportionality", "(d) Storm-Dependence"]
SHOW_TITLE = True
FIG_TITLE = "Storm-Outage Disruption Score (SODS) dimensions"
NON_CONUS = {"02", "15", "72", "66", "78", "60", "69"}
MISSING = "#e6e6e6"
GAMMA = 2.5
DPI = 600

d = pd.read_csv(f"{OUT_DIR}/risk_scores_by_county.csv", dtype={"county_fips": str})
d["county_fips"] = d["county_fips"].str.zfill(5)
d = d.rename(columns={"Hazard": "Storm"})   # display name only; data unchanged

gdf = gpd.read_file(SHP)
gdf["county_fips"] = gdf["GEOID"].astype(str).str.zfill(5)
gdf = gdf[~gdf["county_fips"].str[:2].isin(NON_CONUS)].to_crs("EPSG:5070")
geo = gdf.merge(d, on="county_fips", how="left")
pn = PowerNorm(gamma=GAMMA, vmin=0, vmax=100)

fig, axes = plt.subplots(2, 2, figsize=(18, 11))
for ax, dim, lab in zip(axes.ravel(), DIMS, LABELS):
    geo.plot(column=dim, cmap="magma_r", norm=pn, linewidth=0.03, edgecolor="white",
             ax=ax, legend=True, missing_kwds={"color": MISSING},
             legend_kwds={"shrink": 0.6, "label": f"{dim} (0-100)",
                          "ticks": [0, 40, 60, 80, 100]})
    ax.set_axis_off()
    ax.set_title(lab, fontsize=15, fontweight="bold", loc="center")

if SHOW_TITLE:
    fig.suptitle(FIG_TITLE, fontsize=17, fontweight="bold", y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96 if SHOW_TITLE else 1])
out = f"{FIG_DIR}/supp_fig3_sods_dimensions.png"
fig.savefig(out, dpi=DPI, bbox_inches="tight")
plt.close(fig)
print(f"Wrote {out}")

Wrote /icebox/data/shares/mh1/fgorginpaveh/Project3/Power_Outage_ZeroFill/Results/Start_From_Begining/figures/supp_fig3_sods_dimensions.png
